# UTS PjBL Kecerdasan Buatan & Sistem Cerdas — TIM 12
## Bot Game AI Connect-Four Berbasis Depth-Limited Minimax & Evaluation Function

- **Institusi:** Institut Digital Ekonomi (IDE) LPKIA Bandung — S1 Teknik Informatika / Sistem Informasi
- **Dosen Pengampu:** Wahyu Nurjaya WK, S.T., M.Kom.
- **Ruang Lingkup & Metode AI:** *Adversarial Search* (*Depth-Limited Minimax* + *Heuristic Evaluation Function* & *Alpha-Beta Pruning*)
- **Demo Web UI (GitHub Pages):** [https://zysrnh.github.io/ConnectFour_UTS/](https://zysrnh.github.io/ConnectFour_UTS/)
- **Repository GitHub:** [https://github.com/zysrnh/ConnectFour_UTS](https://github.com/zysrnh/ConnectFour_UTS)

### Cara Menjalankan di Google Colab
1. Klik **`▶ Jalankan semua`** (menu **Runtime → Run all**) di toolbar atas agar seluruh modul Python, pengujian taktis, benchmark, dan UI interaktif tereksekusi berurutan.
2. Scroll ke sel paling bawah (**Bagian 6B**) untuk memainkan **Arena Retro Connect-Four (Main Menu → Arena Beranimasi → Tabel Analisis Akhir)** langsung di Colab menggunakan **Python Kernel**!

### Identitas TIM 12
| Peran | Nama Mahasiswa | NIM | Tanggung Jawab Utama |
| :--- | :--- | :--- | :--- |
| **Mahasiswa A** | `[Isi Nama Mahasiswa A]` | `[NIM A]` | *AI System Architect / Modeler* (State Space, Eval Function, Minimax) |
| **Mahasiswa B** | `[Isi Nama Mahasiswa B]` | `[NIM B]` | *Lead Developer / Tester* (Arena Interaktif, Benchmark, Web UI) |

---
## Bagian 1: Pemodelan AI & Relevansi dengan Materi Sesi 1–4

Implementasi proyek ini memadukan konsep inti yang telah dipelajari dari **Sesi 1 hingga Sesi 4**:

1. **Arsitektur Sistem Cerdas & Intelligent Agent (Sesi 1):**
   - **Input:** Status papan Connect-Four `6 × 7` saat ini dan aksi kolom pemain.
   - **Knowledge Base:** Aturan kemenangan `4-in-a-row` dan bobot fitur *Evaluation Function* (`EVAL_WEIGHTS`).
   - **Inference Engine:** Algoritma *Depth-Limited Minimax* (serta opsi optimasi *Alpha-Beta Pruning*).
   - **Output:** Kolom terbaik `[0..6]`, skor evaluasi papan, dan jumlah node yang dievaluasi.
   - **Formulasi Agen:** `Percept` berupa posisi papan `6 × 7`; `Action` menjatuhkan bidak ke kolom valid; `Performance Measure` diukur dari keberhasilan menang (`WIN_SCORE`), pemblokiran ancaman lawan, dan efisiensi node.
2. **State Space & Search Tree (Sesi 2):**
   - Ruang keadaan dimodelkan sebagai *Game Tree* dengan *branching factor* `b <= 7` dan kedalaman maksimal permainan `m = 42`.
   - Penelusuran dilakukan secara rekursif berbasis *Depth-First Search* (DFS).
3. **Heuristic Function `h(n)` (Sesi 3):**
   - Ketika batas kedalaman pencarian tercapai pada posisi non-terminal (`depth >= max_depth`), fungsi `evaluate_board()` bertindak sebagai fungsi heuristik untuk menaksir keunggulan papan bagi Bot.
4. **Adversarial Search: Minimax & Alpha-Beta Pruning (Sesi 4):**
   - Permainan bersifat *zero-sum*: **Bot AI (`2` / `X`) sebagai MAX** dan **Manusia (`1` / `O`) sebagai MIN**.
   - **Kenapa pakai Depth-Limited Minimax?** Seperti dibahas pada Pre-Test Sesi 4, Minimax murni memiliki kompleksitas `O(b^m)`. Pada Connect-Four (`b <= 7, m = 42`), pencarian penuh sampai terminal state tidak efisien. Pembatasan kedalaman (`max_depth`) + *Evaluation Function* membuat keputusan cepat dan akurat.
   - **Optimasi Alpha-Beta Pruning:** Memangkas cabang yang tidak memengaruhi hasil akhir saat `alpha >= beta`, didukung *center-first column ordering* `[3, 2, 4, 1, 5, 0, 6]`.

In [ ]:
# Setup Environment & Konstanta Global (Sinkron 1:1 dengan js/ai.js)
import math
import random
import sys
import time
import pandas as pd

try:
    import matplotlib.pyplot as plt
    HAS_MATPLOTLIB = True
except ImportError:
    HAS_MATPLOTLIB = False

ROWS = 6
COLS = 7
WINDOW_LENGTH = 4

BOARD_SIZES = {
    "6x7": (6, 7),
    "7x8": (7, 8),
    "7x9": (7, 9),
    "8x10": (8, 10),
}

BLOCKED = -1  # Petak batu rintangan (mode Obstacle)
EMPTY = 0
HUMAN = 1   # Pemain 1 (Coral / 'O')
AI = 2      # Pemain 2 (Kuning / 'X')
P3 = 3      # Pemain 3 (Hijau / 'G')
P4 = 4      # Pemain 4 (Ungu / 'U')

WIN_SCORE = 100000


def get_column_order(cols=COLS):
    """Kembalikan urutan prioritas kolom dari tengah ke pinggir."""
    order = []
    if cols % 2 == 1:
        mid = cols // 2
        order.append(mid)
        for d in range(1, mid + 1):
            if mid - d >= 0:
                order.append(mid - d)
            if mid + d < cols:
                order.append(mid + d)
    else:
        left = cols // 2 - 1
        right = cols // 2
        for d in range(cols // 2):
            if left - d >= 0:
                order.append(left - d)
            if right + d < cols:
                order.append(right + d)
    return order


def get_next_player(piece, player_count=2):
    """Kembalikan giliran pemain berikutnya secara siklik (1..player_count)."""
    total = 4 if player_count >= 4 else 2
    return 1 if piece >= total else piece + 1


COLUMN_ORDER = get_column_order(COLS)  # Prioritas kolom tengah ke pinggir (standar 6x7)

EVAL_WEIGHTS = {
    "WIN": 100000,
    "AI_THREE": 12,
    "AI_TWO": 3,
    "OPP_THREE": -80,
    "OPP_TWO": -2,
    "CENTER_PER_DISC": 4,
}

DIFFICULTY_PRESETS = {
    "Easy": {"max_depth": 0, "use_pruning": False, "label": "Easy (Random Move)"},
    "Medium": {"max_depth": 3, "use_pruning": False, "label": "Medium (Depth-Limited Minimax, depth=3)"},
    "Hard": {"max_depth": 5, "use_pruning": True, "label": "Hard (Depth-Limited + Alpha-Beta, depth=5)"},
}

print("=== CEK ENVIRONMENT PRAKTIKUM AI TIM 12 ===")
print("Versi Python       :", sys.version.split()[0])
print("Matplotlib tersedia:", HAS_MATPLOTLIB)
print("Konfigurasi Papan  :", f"{ROWS} baris x {COLS} kolom (opsional: 7x8 Luas, 7x9 Ekstra)")

---
## Bagian 2: Representasi Papan & Aturan Permainan Connect-Four
Papan direpresentasikan sebagai matriks 2D berukuran standar `6 × 7` (juga mendukung varian lebih luas `7 × 8` Luas & `7 × 9` Ekstra; `row 0` = baris paling atas, `row rows-1` = baris paling bawah). Setiap bidak yang dijatuhkan ke kolom akan jatuh ke baris kosong terbawah akibat gravitasi.

In [ ]:
def create_board(rows=ROWS, cols=COLS):
    """Buat papan kosong berukuran rows x cols (baris 0 = atas, baris terakhir = bawah)."""
    return [[EMPTY for _ in range(cols)] for _ in range(rows)]


def create_special_setup(board_mode="classic", rows=ROWS, cols=COLS):
    """Buat papan awal sesuai mode ('classic', 'obstacle') dan ukuran rows x cols."""
    board = create_board(rows, cols)
    obstacles = []
    center_cols = {cols // 2} if cols % 2 == 1 else {cols // 2 - 1, cols // 2}
    side_cols = [c for c in range(cols) if c not in center_cols]

    if board_mode == "obstacle":
        col_pick_count = 4 if cols >= 8 else 3
        chosen = random.sample(side_cols, col_pick_count)
        spots = [
            (rows - 1, chosen[0]),
            (rows - 1, chosen[1]),
            (rows - 2, chosen[1]),
            (rows - 1, chosen[2]),
        ]
        if cols >= 8 and len(chosen) >= 4:
            spots.append((rows - 1, chosen[3]))
        if cols >= 10 and len(chosen) >= 4:
            spots.append((rows - 2, chosen[3]))
        for r, c in spots:
            board[r][c] = BLOCKED
            obstacles.append([r, c])

    return {"board": board, "obstacles": obstacles}


def clone_board(board):
    """Salin papan 2D secara mendalam."""
    return [row[:] for row in board]


def print_board(board):
    """Tampilkan papan Connect-Four ke konsol sesuai dimensi papan."""
    rows = len(board)
    cols = len(board[0])
    symbol_map = {BLOCKED: "#", EMPTY: ".", HUMAN: "O", AI: "X", P3: "G", P4: "U"}
    print("  " + " ".join(str(c) for c in range(cols)))
    print(" +" + "--" * cols + "+")
    for r in range(rows):
        row_str = " ".join(symbol_map[board[r][c]] for c in range(cols))
        print(f"{r}| {row_str}|")
    print(" +" + "--" * cols + "+")


def is_valid_location(board, col):
    """Cek apakah kolom valid dan baris paling atas masih kosong."""
    cols = len(board[0])
    return 0 <= col < cols and board[0][col] == EMPTY


def get_valid_locations(board):
    """Kembalikan daftar kolom valid sesuai urutan tengah-ke-pinggir."""
    cols = len(board[0])
    return [col for col in get_column_order(cols) if is_valid_location(board, col)]


def get_next_open_row(board, col):
    """Cari indeks baris terbawah yang masih kosong pada kolom tertentu."""
    for r in range(len(board) - 1, -1, -1):
        if board[r][col] == EMPTY:
            return r
    return -1


def drop_piece(board, row, col, piece):
    """Letakkan bidak pada koordinat (row, col)."""
    board[row][col] = piece


def get_winner_info(board):
    """Cek pemenang (pemain 1..4) atau seri beserta daftar koordinat kemenangan."""
    rows = len(board)
    cols = len(board[0])

    def is_player(p):
        return 1 <= p <= 4

    # Cek horizontal
    for r in range(rows):
        for c in range(cols - WINDOW_LENGTH + 1):
            p = board[r][c]
            if is_player(p) and p == board[r][c + 1] == board[r][c + 2] == board[r][c + 3]:
                return p, [(r, c), (r, c + 1), (r, c + 2), (r, c + 3)]

    # Cek vertikal
    for c in range(cols):
        for r in range(rows - WINDOW_LENGTH + 1):
            p = board[r][c]
            if is_player(p) and p == board[r + 1][c] == board[r + 2][c] == board[r + 3][c]:
                return p, [(r, c), (r + 1, c), (r + 2, c), (r + 3, c)]

    # Cek diagonal turun ke kanan (\)
    for r in range(rows - WINDOW_LENGTH + 1):
        for c in range(cols - WINDOW_LENGTH + 1):
            p = board[r][c]
            if is_player(p) and p == board[r + 1][c + 1] == board[r + 2][c + 2] == board[r + 3][c + 3]:
                return p, [(r, c), (r + 1, c + 1), (r + 2, c + 2), (r + 3, c + 3)]

    # Cek diagonal naik ke kanan (/)
    for r in range(WINDOW_LENGTH - 1, rows):
        for c in range(cols - WINDOW_LENGTH + 1):
            p = board[r][c]
            if is_player(p) and p == board[r - 1][c + 1] == board[r - 2][c + 2] == board[r - 3][c + 3]:
                return p, [(r, c), (r - 1, c + 1), (r - 2, c + 2), (r - 3, c + 3)]

    # Cek seri jika semua kolom teratas penuh
    if not get_valid_locations(board):
        return "DRAW", []

    return None, []


def check_winner(board):
    """Wrapper ringkas mengembalikan 1 (HUMAN), 2 (AI), 'DRAW', atau None."""
    winner, _ = get_winner_info(board)
    return winner


# Uji cepat papan & cek kemenangan
sample_board = create_board()
for col_idx in [3, 3, 2, 2, 4, 1]:
    r_open = get_next_open_row(sample_board, col_idx)
    drop_piece(sample_board, r_open, col_idx, AI if col_idx != 1 else HUMAN)

print_board(sample_board)
print("Status Game Contoh:", check_winner(sample_board))

---
## Bagian 3: Fungsi Evaluasi Papan (*Heuristic Evaluation Function*)
Karena pencarian dibatasi oleh `max_depth`, Bot AI membutuhkan **Evaluation Function** untuk menilai posisi papan non-terminal:

- Memeriksa seluruh **69 jendela 4-petak** (horizontal, vertikal, diagonal naik & turun):
  - `3 bidak AI + 1 kosong`: **+12**
  - `2 bidak AI + 2 kosong`: **+3**
  - `3 bidak Lawan + 1 kosong`: **-80** (penalti besar agar AI memprioritaskan blokir ancaman langsung)
  - `2 bidak Lawan + 2 kosong`: **-2**
- Memberikan **bonus kontrol kolom tengah (`kolom 3`)** sebesar **+4** per selisih bidak di kolom tengah.

In [ ]:
def classify_window(window, ai_piece):
    """Klasifikasikan satu jendela 4-petak untuk 2-Player maupun 4-Player."""
    ai_count = 0
    empty_count = 0
    sole_opp = 0
    sole_opp_count = 0

    for cell in window:
        if cell == BLOCKED:
            return "blocked", 0
        if cell == EMPTY:
            empty_count += 1
        elif cell == ai_piece:
            ai_count += 1
        else:
            if sole_opp == 0:
                sole_opp = cell
                sole_opp_count = 1
            elif sole_opp == cell:
                sole_opp_count += 1
            else:
                return "mixed", 0

    if ai_count > 0 and sole_opp_count > 0:
        return "mixed", 0
    if ai_count == 4:
        return "ai_win", EVAL_WEIGHTS["WIN"]
    if sole_opp_count == 4:
        return "opp_win", -EVAL_WEIGHTS["WIN"]
    if ai_count == 3 and empty_count == 1:
        return "ai3", EVAL_WEIGHTS["AI_THREE"]
    if ai_count == 2 and empty_count == 2:
        return "ai2", EVAL_WEIGHTS["AI_TWO"]
    if sole_opp_count == 3 and empty_count == 1:
        return "opp3", EVAL_WEIGHTS["OPP_THREE"]
    if sole_opp_count == 2 and empty_count == 2:
        return "opp2", EVAL_WEIGHTS["OPP_TWO"]
    return "neutral", 0


def evaluate_window(window, ai_piece):
    """Hitung skor heuristik satu jendela berisi 4 petak dari sudut pandang ai_piece."""
    _, score = classify_window(window, ai_piece)
    return score


def analyze_board(board, ai_piece=AI, player_count=2):
    """Hitung total skor evaluasi papan (2P maupun 4P) beserta rincian fitur heuristiknya."""
    rows = len(board)
    cols = len(board[0])
    total_players = 4 if player_count >= 4 else 2
    center_cols = [cols // 2] if cols % 2 == 1 else [cols // 2 - 1, cols // 2]
    center_weight = EVAL_WEIGHTS["CENTER_PER_DISC"] if cols % 2 == 1 else EVAL_WEIGHTS["CENTER_PER_DISC"] // 2

    center_ai = 0
    opp_center_counts = {1: 0, 2: 0, 3: 0, 4: 0}
    for cc in center_cols:
        for r in range(rows):
            cell = board[r][cc]
            if cell == ai_piece:
                center_ai += 1
            elif 1 <= cell <= total_players:
                opp_center_counts[cell] += 1

    center_opp = max((opp_center_counts[p] for p in range(1, total_players + 1) if p != ai_piece), default=0)

    ai_three = 0
    ai_two = 0
    opp_three = 0
    opp_two = 0
    window_score = 0

    def process_window(w):
        nonlocal ai_three, ai_two, opp_three, opp_two, window_score
        kind, sc = classify_window(w, ai_piece)
        if kind == "ai3":
            ai_three += 1
        elif kind == "ai2":
            ai_two += 1
        elif kind == "opp3":
            opp_three += 1
        elif kind == "opp2":
            opp_two += 1
        window_score += sc

    # Horizontal
    for r in range(rows):
        for c in range(cols - WINDOW_LENGTH + 1):
            process_window([board[r][c], board[r][c + 1], board[r][c + 2], board[r][c + 3]])

    # Vertikal
    for c in range(cols):
        for r in range(rows - WINDOW_LENGTH + 1):
            process_window([board[r][c], board[r + 1][c], board[r + 2][c], board[r + 3][c]])

    # Diagonal turun (\)
    for r in range(rows - WINDOW_LENGTH + 1):
        for c in range(cols - WINDOW_LENGTH + 1):
            process_window([board[r][c], board[r + 1][c + 1], board[r + 2][c + 2], board[r + 3][c + 3]])

    # Diagonal naik (/)
    for r in range(WINDOW_LENGTH - 1, rows):
        for c in range(cols - WINDOW_LENGTH + 1):
            process_window([board[r][c], board[r - 1][c + 1], board[r - 2][c + 2], board[r - 3][c + 3]])

    center_score = (center_ai - center_opp) * center_weight
    return {
        "total_score": window_score + center_score,
        "center_ai": center_ai,
        "center_opp": center_opp,
        "center_score": center_score,
        "ai_three": ai_three,
        "ai_two": ai_two,
        "opp_three": opp_three,
        "opp_two": opp_two,
        "window_score": window_score,
    }


def evaluate_board(board, ai_piece=AI, player_count=2):
    """Fungsi Evaluasi Papan (Evaluation Function) untuk non-terminal state."""
    return analyze_board(board, ai_piece, player_count)["total_score"]


# Uji fungsi evaluasi pada papan contoh
eval_demo = analyze_board(sample_board, AI)
print("Rincian Evaluasi Papan Contoh (Sudut Pandang AI):")
for k, v in eval_demo.items():
    print(f"  - {k:14s}: {v}")

---
## Bagian 4: Implementasi Depth-Limited Minimax & Alpha-Beta Pruning (2P & 4P Paranoid)
Di bawah ini terdapat dua versi algoritma pencarian:
1. `minimax(...)`: **Depth-Limited Minimax Murni** (menjelajah seluruh cabang hingga batas `max_depth`).
2. `minimax_alpha_beta(...)`: **Depth-Limited Minimax + Alpha-Beta Pruning** (memangkas cabang saat `alpha >= beta`, mendukung mode **2-Player** klasik & **4-Player Paranoid Minimax**).

Keduanya menerapkan **penalti kedalaman** (`WIN_SCORE - depth` saat menang, `depth - WIN_SCORE` saat kalah) seperti pola pada Sesi 4 agar Bot AI memilih kemenangan tercepat dan menunda kekalahan selama mungkin.

In [ ]:
def minimax(board, depth, max_depth, turn_or_max, ai_piece, stats, player_count=2):
    """Depth-Limited Minimax Murni (mendukung 2P & 4P Paranoid Minimax)."""
    stats["nodes"] += 1
    total_players = 4 if player_count >= 4 else 2
    if isinstance(turn_or_max, bool):
        turn_piece = ai_piece if turn_or_max else get_next_player(ai_piece, total_players)
    else:
        turn_piece = turn_or_max

    winner = check_winner(board)
    if winner == ai_piece:
        return WIN_SCORE - depth
    if winner == "DRAW":
        return 0
    if winner is not None:
        return depth - WIN_SCORE

    if depth >= max_depth:
        return evaluate_board(board, ai_piece, total_players)

    valid_cols = get_valid_locations(board)
    next_piece = get_next_player(turn_piece, total_players)

    if turn_piece == ai_piece:
        best_score = -math.inf
        for col in valid_cols:
            row = get_next_open_row(board, col)
            board[row][col] = turn_piece
            score = minimax(board, depth + 1, max_depth, next_piece, ai_piece, stats, total_players)
            board[row][col] = EMPTY
            if score > best_score:
                best_score = score
        return best_score

    best_score = math.inf
    for col in valid_cols:
        row = get_next_open_row(board, col)
        board[row][col] = turn_piece
        score = minimax(board, depth + 1, max_depth, next_piece, ai_piece, stats, total_players)
        board[row][col] = EMPTY
        if score < best_score:
            best_score = score
    return best_score


def minimax_alpha_beta(board, depth, max_depth, alpha, beta, turn_or_max, ai_piece, stats, player_count=2):
    """Depth-Limited Minimax dengan optimasi Alpha-Beta Pruning (2P & 4P Paranoid)."""
    stats["nodes"] += 1
    total_players = 4 if player_count >= 4 else 2
    if isinstance(turn_or_max, bool):
        turn_piece = ai_piece if turn_or_max else get_next_player(ai_piece, total_players)
    else:
        turn_piece = turn_or_max

    winner = check_winner(board)
    if winner == ai_piece:
        return WIN_SCORE - depth
    if winner == "DRAW":
        return 0
    if winner is not None:
        return depth - WIN_SCORE

    if depth >= max_depth:
        return evaluate_board(board, ai_piece, total_players)

    valid_cols = get_valid_locations(board)
    next_piece = get_next_player(turn_piece, total_players)

    if turn_piece == ai_piece:
        max_eval = -math.inf
        for col in valid_cols:
            row = get_next_open_row(board, col)
            board[row][col] = turn_piece
            eval_score = minimax_alpha_beta(
                board, depth + 1, max_depth, alpha, beta, next_piece, ai_piece, stats, total_players
            )
            board[row][col] = EMPTY
            if eval_score > max_eval:
                max_eval = eval_score
            if max_eval > alpha:
                alpha = max_eval
            if alpha >= beta:
                break  # Pemangkasan cabang MIN
        return max_eval

    min_eval = math.inf
    for col in valid_cols:
        row = get_next_open_row(board, col)
        board[row][col] = turn_piece
        eval_score = minimax_alpha_beta(
            board, depth + 1, max_depth, alpha, beta, next_piece, ai_piece, stats, total_players
        )
        board[row][col] = EMPTY
        if eval_score < min_eval:
            min_eval = eval_score
        if min_eval < beta:
            beta = min_eval
        if alpha >= beta:
            break  # Pemangkasan cabang MAX
    return min_eval


def find_best_move(board, max_depth=4, ai_piece=AI, use_pruning=True, player_count=2):
    """Cari kolom terbaik beserta skor evaluasi tiap kolom dan jumlah node."""
    working_board = clone_board(board)
    valid_cols = get_valid_locations(working_board)
    column_scores = {c: None for c in range(len(working_board[0]))}

    if not valid_cols:
        return {
            "best_col": -1,
            "best_score": 0,
            "column_scores": column_scores,
            "nodes": 0,
            "elapsed_ms": 0.0,
            "max_depth": max_depth,
            "use_pruning": use_pruning,
        }

    stats = {"nodes": 0}
    t0 = time.perf_counter()

    best_score = -math.inf
    best_col = valid_cols[0]
    alpha = -math.inf
    beta = math.inf
    next_turn = get_next_player(ai_piece, player_count)

    for col in valid_cols:
        row = get_next_open_row(working_board, col)
        working_board[row][col] = ai_piece

        if use_pruning:
            score = minimax_alpha_beta(
                working_board, 1, max_depth, alpha, beta, next_turn, ai_piece, stats, player_count
            )
        else:
            score = minimax(
                working_board, 1, max_depth, next_turn, ai_piece, stats, player_count
            )

        working_board[row][col] = EMPTY
        column_scores[col] = score

        if score > best_score:
            best_score = score
            best_col = col
        if use_pruning and best_score > alpha:
            alpha = best_score

    t1 = time.perf_counter()
    return {
        "best_col": best_col,
        "best_score": best_score,
        "column_scores": column_scores,
        "nodes": stats["nodes"],
        "elapsed_ms": round((t1 - t0) * 1000, 2),
        "max_depth": max_depth,
        "use_pruning": use_pruning,
    }


def choose_move(board, difficulty="Hard", ai_piece=AI, use_pruning=None, custom_depth=None, player_count=2):
    """Pilih langkah Bot AI sesuai tingkat kesulitan ('Easy', 'Medium', 'Hard')."""
    preset = DIFFICULTY_PRESETS.get(difficulty, DIFFICULTY_PRESETS["Hard"])

    if difficulty == "Easy" and custom_depth is None:
        valid_cols = get_valid_locations(board)
        column_scores = {c: None for c in range(len(board[0]))}
        if not valid_cols:
            return {
                "best_col": -1, "best_score": 0, "column_scores": column_scores,
                "nodes": 0, "elapsed_ms": 0.0, "max_depth": 0,
                "use_pruning": False, "is_random": True,
            }
        working_board = clone_board(board)
        t0 = time.perf_counter()
        for col in valid_cols:
            row = get_next_open_row(working_board, col)
            working_board[row][col] = ai_piece
            column_scores[col] = evaluate_board(working_board, ai_piece, player_count)
            working_board[row][col] = EMPTY
        chosen = random.choice(valid_cols)
        t1 = time.perf_counter()
        return {
            "best_col": chosen,
            "best_score": column_scores[chosen],
            "column_scores": column_scores,
            "nodes": len(valid_cols),
            "elapsed_ms": round((t1 - t0) * 1000, 2),
            "max_depth": 0,
            "use_pruning": False,
            "is_random": True,
        }

    default_depth = 4 if ((player_count >= 4 or len(board[0]) >= 10) and difficulty == "Hard") else preset["max_depth"]
    max_depth = custom_depth if custom_depth is not None else default_depth
    pruning = use_pruning if use_pruning is not None else preset["use_pruning"]
    res = find_best_move(board, max_depth=max_depth, ai_piece=ai_piece, use_pruning=pruning, player_count=player_count)
    res["is_random"] = False
    return res


def compare_algorithms(board, max_depth=4, ai_piece=AI, player_count=2):
    """Bandingkan Minimax Murni vs Alpha-Beta Pruning pada state papan yang sama."""
    pure = find_best_move(board, max_depth=max_depth, ai_piece=ai_piece, use_pruning=False, player_count=player_count)
    ab = find_best_move(board, max_depth=max_depth, ai_piece=ai_piece, use_pruning=True, player_count=player_count)
    reduction = round(((pure["nodes"] - ab["nodes"]) / pure["nodes"]) * 100, 1) if pure["nodes"] > 0 else 0.0
    return {
        "max_depth": max_depth,
        "pure": pure,
        "alphabeta": ab,
        "reduction_pct": reduction,
        "same_decision": pure["best_col"] == ab["best_col"],
    }

---
## Bagian 5: Pengujian Terstruktur & Benchmark Efisiensi Node
Sel-sel di bawah ini berjalan otomatis (tanpa `input()`) untuk menguji:
1. Ketepatan taktis Bot AI (menang 1 langkah, memblokir lawan, memilih kolom tengah di awal, Obstacle, papan `7×9`, & mode 4-Player Paranoid Minimax).
2. Perbandingan jumlah *node* yang dievaluasi dan waktu komputasi **Minimax Murni vs Alpha-Beta Pruning** pada berbagai kedalaman (`depth = 1..4`).
3. Simulasi pertandingan antar tingkat kesulitan (5 game per pasangan, mengikuti standar uji coba Sesi 4).

In [ ]:
# 1. Pengujian Taktis Unit Test
def run_tactical_tests():
    # Kasus A: Papan kosong -> harus pilih kolom tengah (3)
    b_empty = create_board()
    res_empty = find_best_move(b_empty, max_depth=3, ai_piece=AI, use_pruning=True)
    assert res_empty["best_col"] == 3, f"Harusnya kolom 3, dapat {res_empty['best_col']}"

    # Kasus B: AI punya 3 beruntun (kolom 0,1,2 di baris 5) -> wajib selesaikan menang di kolom 3
    b_win = create_board()
    for c in [0, 1, 2]:
        drop_piece(b_win, 5, c, AI)
    for c in [0, 1]:
        drop_piece(b_win, 4, c, HUMAN)
    res_win = find_best_move(b_win, max_depth=3, ai_piece=AI, use_pruning=True)
    assert res_win["best_col"] == 3 and res_win["best_score"] == WIN_SCORE - 1

    # Kasus C: Manusia punya 3 beruntun (kolom 4,5,6 di baris 5) -> Bot wajib blokir di kolom 3
    b_block = create_board()
    for c in [4, 5, 6]:
        drop_piece(b_block, 5, c, HUMAN)
    drop_piece(b_block, 5, 0, AI)
    drop_piece(b_block, 4, 0, AI)
    res_block = find_best_move(b_block, max_depth=3, ai_piece=AI, use_pruning=True)
    assert res_block["best_col"] == 3, f"Harusnya blokir kolom 3, dapat {res_block['best_col']}"

    # Kasus D: Keputusan Minimax Murni dan Alpha-Beta harus 100% identik
    comp = compare_algorithms(b_block, max_depth=4, ai_piece=AI)
    assert comp["same_decision"], "Keputusan Minimax vs Alpha-Beta berbeda!"

    # Kasus E: Mode Obstacle (BLOCKED = -1) menonaktifkan 4-in-a-row melewati batu
    b_obs = create_board()
    b_obs[5][2] = BLOCKED
    drop_piece(b_obs, 5, 0, AI)
    drop_piece(b_obs, 5, 1, AI)
    drop_piece(b_obs, 5, 3, AI)
    assert check_winner(b_obs) is None

    # Kasus F: Mode 4-Player (P1..P4) mendeteksi menang P3 & P4 serta memblokir ancaman lawan di 4P
    b_4p = create_board(7, 9)
    for c in [5, 6, 7]:
        drop_piece(b_4p, 6, c, P3)
    res_4p = find_best_move(b_4p, max_depth=3, ai_piece=P4, use_pruning=True, player_count=4)
    assert res_4p["best_col"] in (4, 8), f"P4 di mode 4-Player harus blokir ancaman P3 di kol 4/8, dapat {res_4p['best_col']}"
    comp_4p = compare_algorithms(b_4p, max_depth=3, ai_piece=P4, player_count=4)
    assert comp_4p["same_decision"], "Keputusan Minimax vs Alpha-Beta di mode 4P harus sama!"

    # Kasus G: Ukuran papan lebih luas (7x8, 7x9, & 8x10) mendukung pusat kolom & menang di pinggir kanan
    b_wide = create_board(8, 10)
    res_wide = find_best_move(b_wide, max_depth=2, ai_piece=AI, use_pruning=True)
    assert res_wide["best_col"] == 4, f"Papan 8x10 kosong harusnya kolom tengah kiri 4, dapat {res_wide['best_col']}"
    for c in [6, 7, 8, 9]:
        drop_piece(b_wide, 7, c, AI)
    assert check_winner(b_wide) == AI

    print("[PASS] Semua 7 pengujian taktis (termasuk Obstacle, Papan 8x10, & 4-Player Paranoid Minimax) berhasil 100%!")
    print(f"  - Papan kosong -> Kol {res_empty['best_col']} (skor {res_empty['best_score']})")
    print(f"  - Deteksi menang 1 langkah -> Kol {res_win['best_col']} (skor {res_win['best_score']})")
    print(f"  - Blokir ancaman lawan     -> Kol {res_block['best_col']} (skor {res_block['best_score']})")
    print(f"  - Konsistensi Minimax vs Alpha-Beta (d=4): Kol {comp['pure']['best_col']} == Kol {comp['alphabeta']['best_col']} "
          f"(hemat {comp['reduction_pct']}% node)")


run_tactical_tests()

In [ ]:
# 2. Benchmark Minimax Murni vs Alpha-Beta Pruning (Depth 1 s.d. 4)
bench_board = create_board()
sample_opening = [(3, HUMAN), (3, AI), (2, HUMAN), (4, AI), (2, HUMAN), (3, AI)]
for col_m, piece_m in sample_opening:
    drop_piece(bench_board, get_next_open_row(bench_board, col_m), col_m, piece_m)

print("Posisi Papan Benchmark (6 langkah pembuka):")
print_board(bench_board)

rows_bench = []
for d in [1, 2, 3, 4]:
    comp = compare_algorithms(bench_board, max_depth=d, ai_piece=AI)
    rows_bench.append({
        "Max Depth": d,
        "Kolom Dipilih": comp["alphabeta"]["best_col"],
        "Skor Eval": comp["alphabeta"]["best_score"],
        "Node Minimax": comp["pure"]["nodes"],
        "Node Alpha-Beta": comp["alphabeta"]["nodes"],
        "Penghematan Node (%)": comp["reduction_pct"],
        "Waktu Minimax (ms)": comp["pure"]["elapsed_ms"],
        "Waktu Alpha-Beta (ms)": comp["alphabeta"]["elapsed_ms"],
        "Keputusan Sama?": comp["same_decision"],
    })

df_bench = pd.DataFrame(rows_bench)
print("\n=== TABEL BENCHMARK EFISIENSI NODE ===")
print(df_bench.to_string(index=False))

if HAS_MATPLOTLIB:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    depths = df_bench["Max Depth"]

    axes[0].bar(depths - 0.18, df_bench["Node Minimax"], width=0.35, label="Minimax Murni", color="#dc2626")
    axes[0].bar(depths + 0.18, df_bench["Node Alpha-Beta"], width=0.35, label="Alpha-Beta Pruning", color="#1d4ed8")
    axes[0].set_title("Perbandingan Jumlah Node Dievaluasi")
    axes[0].set_xlabel("Max Depth")
    axes[0].set_ylabel("Jumlah Node")
    axes[0].set_xticks(depths)
    axes[0].legend()
    axes[0].grid(axis="y", linestyle="--", alpha=0.4)

    axes[1].bar(depths, df_bench["Penghematan Node (%)"], width=0.45, color="#15803d")
    axes[1].set_title("Persentase Penghematan Node Alpha-Beta")
    axes[1].set_xlabel("Max Depth")
    axes[1].set_ylabel("Penghematan (%)")
    axes[1].set_ylim(0, 100)
    axes[1].set_xticks(depths)
    axes[1].grid(axis="y", linestyle="--", alpha=0.4)

    plt.tight_layout()
    plt.show()

In [ ]:
# 3. Simulasi Turnamen Otomatis Bot vs Bot (5 Pertandingan per Skenario)
def simulate_match(p1_cfg, p2_cfg, seed=None, opening=None):
    if seed is not None:
        random.seed(seed)
    board = create_board()
    moves = 0
    nodes_p1, nodes_p2 = [], []

    if opening:
        for idx, col in enumerate(opening):
            piece = HUMAN if idx % 2 == 0 else AI
            drop_piece(board, get_next_open_row(board, col), col, piece)
            moves += 1
        turn = HUMAN if len(opening) % 2 == 0 else AI
    else:
        turn = HUMAN  # HUMAN = Pemain 1, AI = Pemain 2

    while True:
        cfg = p1_cfg if turn == HUMAN else p2_cfg
        res = choose_move(
            board,
            difficulty=cfg["difficulty"],
            ai_piece=turn,
            use_pruning=cfg.get("use_pruning"),
            custom_depth=cfg.get("custom_depth"),
        )
        col = res["best_col"]
        drop_piece(board, get_next_open_row(board, col), col, turn)
        moves += 1
        if turn == HUMAN:
            nodes_p1.append(res["nodes"])
        else:
            nodes_p2.append(res["nodes"])

        winner = check_winner(board)
        if winner is not None:
            return {
                "winner": winner,
                "moves": moves,
                "avg_nodes_p1": round(sum(nodes_p1) / max(1, len(nodes_p1)), 1),
                "avg_nodes_p2": round(sum(nodes_p2) / max(1, len(nodes_p2)), 1),
            }
        turn = AI if turn == HUMAN else HUMAN


varied_openings = [(3, 3), (2, 3), (4, 3), (3, 2), (3, 4)]

scenarios = [
    ("Easy (Random) vs Medium (Depth 3 Minimax)", {"difficulty": "Easy"}, {"difficulty": "Medium"}, None),
    ("Easy (Random) vs Hard (Depth 5 Alpha-Beta)", {"difficulty": "Easy"}, {"difficulty": "Hard"}, None),
    ("Medium (Depth 3) vs Hard (Depth 5 Alpha-Beta)", {"difficulty": "Medium"}, {"difficulty": "Hard"}, varied_openings),
]

summary_rows = []
for name, p1_c, p2_c, openings in scenarios:
    p1_wins = p2_wins = draws = 0
    total_moves = []
    avg_n2 = []
    for idx in range(5):
        op = openings[idx] if openings else None
        out = simulate_match(p1_c, p2_c, seed=100 + idx, opening=op)
        total_moves.append(out["moves"])
        avg_n2.append(out["avg_nodes_p2"])
        if out["winner"] == HUMAN:
            p1_wins += 1
        elif out["winner"] == AI:
            p2_wins += 1
        else:
            draws += 1

    summary_rows.append({
        "Skenario (P1 vs P2)": name,
        "P1 Menang": p1_wins,
        "P2 Menang": p2_wins,
        "Seri": draws,
        "Rata-rata Langkah": round(sum(total_moves) / len(total_moves), 1),
        "Rata-rata Node P2": round(sum(avg_n2) / len(avg_n2), 1),
    })

df_sim = pd.DataFrame(summary_rows)
print("=== HASIL SIMULASI 5 GAME PER SKENARIO ===")
print(df_sim.to_string(index=False))

---
## Bagian 6A: Arena Teks Konsol Cadangan (`play_game`)
Mode teks konsol Python murni jika ingin bermain lewat terminal/input teks. Ubah `JALANKAN_GAME_INTERAKTIF = True` bila ingin memainkannya lewat `input()`.

In [ ]:
def play_game(difficulty="Hard", human_first=True, custom_depth=None, use_pruning=None, board_size="6x7"):
    """Arena interaktif Manusia ('O') vs Bot AI ('X') mode konsol."""
    rows, cols = BOARD_SIZES.get(board_size, (ROWS, COLS))
    board = create_board(rows, cols)
    current_turn = HUMAN if human_first else AI
    step = 0

    print("=== ARENA INTERAKTIF CONNECT-FOUR AI (TIM 12) ===")
    print(f"Ukuran Papan      : {rows}x{cols}")
    print(f"Tingkat Kesulitan : {difficulty}")
    print("Simbol Pemain     : Manusia = 'O' | Bot AI = 'X'")
    print(f"Masukkan nomor kolom 0 s.d. {cols - 1} (atau 'q' untuk keluar).\n")

    while True:
        print_board(board)
        print(f"Skor Evaluasi Papan Statis (sudut pandang AI): {evaluate_board(board, AI)}")

        if current_turn == HUMAN:
            valid = [c for c in range(cols) if is_valid_location(board, c)]
            while True:
                raw = input(f"[Giliran Manusia 'O'] Pilih kolom {valid} (atau 'q'): ").strip()
                if raw.lower() == "q":
                    print("Permainan dihentikan oleh pemain.")
                    return
                if raw.isdigit() and int(raw) in valid:
                    col = int(raw)
                    break
                print("Input tidak valid! Pilih kolom yang masih tersedia.")

            row = get_next_open_row(board, col)
            drop_piece(board, row, col, HUMAN)
            step += 1
            print(f"-> Manusia menjatuhkan bidak di Kolom {col}, Baris {row}.\n")
        else:
            print("[Giliran Bot AI 'X'] Menghitung langkah terbaik...")
            res = choose_move(
                board,
                difficulty=difficulty,
                ai_piece=AI,
                use_pruning=use_pruning,
                custom_depth=custom_depth,
            )
            col = res["best_col"]
            row = get_next_open_row(board, col)
            drop_piece(board, row, col, AI)
            step += 1

            scores_str = " | ".join(
                f"K{c}:{res['column_scores'][c]}" if res["column_scores"][c] is not None else f"K{c}:Penuh"
                for c in range(cols)
            )
            print(f"-> Bot memilih Kolom {col} (Baris {row}) | Skor: {res['best_score']} "
                  f"| Node: {res['nodes']} | Waktu: {res['elapsed_ms']} ms")
            print(f"   Skor Evaluasi Kolom: [{scores_str}]\n")

        winner, win_cells = get_winner_info(board)
        if winner is not None:
            print_board(board)
            if winner == "DRAW":
                print(f"=== HASIL AKHIR: SERI setelah {step} langkah! ===")
            elif winner == HUMAN:
                print(f"=== SELAMAT! Manusia ('O') MENANG dalam {step} langkah! Koordinat: {win_cells} ===")
            else:
                print(f"=== BOT AI ('X') MENANG dalam {step} langkah! Koordinat: {win_cells} ===")
            break

        current_turn = AI if current_turn == HUMAN else HUMAN


# Ubah menjadi True jika ingin bermain lewat konsol teks
JALANKAN_GAME_INTERAKTIF = False

if JALANKAN_GAME_INTERAKTIF:
    play_game(difficulty="Hard", human_first=True)
else:
    print("Mode konsol standby (aman untuk Run All). Scroll ke sel paling bawah (Bagian 6B) untuk UI Retro Interaktif!")

---
## Bagian 6B: Arena UI Retro 3-Layar Interaktif di Google Colab (`Python Kernel` + Web UI Identik)
Sel terakhir ini merender **UI Retro Connect-Four 3-Layar** (*Main Menu → Arena Bersih Beranimasi → Tabel Analisis Pasca-Main*) yang **sama persis dengan Demo GitHub Pages**, sekaligus menghubungkan setiap langkah Bot AI & Benchmark langsung ke fungsi **Python (`choose_move` & `compare_algorithms`)** di kernel Google Colab!

> **Penting:** Pastikan semua sel di atas sudah dijalankan (**`Runtime → Run all` / `▶ Jalankan semua`**) sebelum memainkan sel ini.

In [ ]:
import json

RETRO_UI_HTML = "<style>\n/* Connect-Four AI — Neo-Brutalist Game Lobby UI (8pt scale, Flat Solid, 100dvh Desktop) */\n\n:root {\n  --bg-paper: #FFC25E;\n  --surface: #fffdf7;\n  --surface-soft: #f5efe0;\n  --surface-blue: #eaf5fc;\n  --ink: #111111;\n  --ink-muted: #4b5563;\n\n  --coral: #fd5a46;\n  --yellow: #ffc567;\n  --green: #00995e;\n  --green-aa: #007a4b;\n  --blue: #058cd7;\n  --purple: #552cb7;\n  --purple-dark: #432098;\n  --stone: #475569;\n\n  --border: 2.5px solid var(--ink);\n  --border-thick: 3px solid var(--ink);\n  --shadow-sm: 2px 2px 0 var(--ink);\n  --shadow-md: 4px 4px 0 var(--ink);\n  --shadow-lg: 6px 6px 0 var(--ink);\n\n  --radius-sm: 4px;\n  --radius-md: 6px;\n  --radius-lg: 8px;\n\n  --font-main: \"Space Grotesk\", \"Segoe UI\", system-ui, -apple-system, sans-serif;\n  --ease-pop: cubic-bezier(0.22, 1, 0.36, 1);\n}\n\n*,\n*::before,\n*::after {\n  box-sizing: border-box;\n}\n\nhtml,\nbody {\n  margin: 0;\n  padding: 0;\n  min-height: 100%;\n  font-family: var(--font-main);\n  color: var(--ink);\n  background-color: var(--bg-paper);\n  background-image:\n    linear-gradient(to right, rgba(17, 17, 17, 0.085) 1px, transparent 1px),\n    linear-gradient(to bottom, rgba(17, 17, 17, 0.085) 1px, transparent 1px);\n  background-size: 32px 32px;\n  line-height: 1.45;\n  -webkit-font-smoothing: antialiased;\n}\n\n@media (min-width: 900px) {\n  html,\n  body {\n    height: 100dvh;\n    overflow: hidden;\n  }\n\n  body {\n    display: flex;\n    flex-direction: column;\n  }\n}\n\nbutton,\ninput,\nselect {\n  font-family: inherit;\n  color: inherit;\n}\n\nbutton {\n  cursor: pointer;\n}\n\n:focus-visible {\n  outline: 3px solid var(--ink);\n  outline-offset: 2px;\n}\n\n/* =========================================================\n   TOPBAR & CORNER SFX TOGGLE\n   ========================================================= */\n\n.topbar {\n  flex: 0 0 auto;\n  background-color: var(--surface);\n  border-bottom: var(--border-thick);\n  position: relative;\n  z-index: 40;\n}\n\n.topbar-inner {\n  max-width: 1200px;\n  margin: 0 auto;\n  min-height: 48px;\n  padding: 6px 24px;\n  display: flex;\n  align-items: center;\n  justify-content: space-between;\n  gap: 16px;\n}\n\n.brand-lockup {\n  display: inline-flex;\n  align-items: center;\n  gap: 10px;\n}\n\n.brand-dots {\n  display: inline-flex;\n  gap: 4px;\n}\n\n.b-dot {\n  width: 12px;\n  height: 12px;\n  border-radius: 50%;\n  border: 2px solid var(--ink);\n  display: inline-block;\n}\n\n.b-dot.p1 { background-color: var(--coral); }\n.b-dot.p2 { background-color: var(--yellow); }\n\n.brand-title {\n  font-size: 1rem;\n  font-weight: 800;\n  letter-spacing: -0.01em;\n}\n\n.topbar-actions {\n  display: flex;\n  align-items: center;\n  gap: 10px;\n}\n\n.top-link-btn {\n  display: inline-flex;\n  align-items: center;\n  justify-content: center;\n  min-height: 44px;\n  padding: 6px 12px;\n  border: var(--border);\n  border-radius: var(--radius-sm);\n  background-color: var(--yellow);\n  color: var(--ink);\n  font-size: 0.85rem;\n  font-weight: 700;\n  text-decoration: none;\n  box-shadow: var(--shadow-sm);\n  transition: transform 140ms var(--ease-pop), box-shadow 140ms var(--ease-pop);\n}\n\n.top-link-btn.dark {\n  background-color: var(--purple);\n  color: #ffffff;\n}\n\n.top-link-btn:hover {\n  transform: translate(-1px, -1px);\n  box-shadow: 3px 3px 0 var(--ink);\n}\n\n.top-link-btn:active {\n  transform: translate(2px, 2px);\n  box-shadow: 1px 1px 0 var(--ink);\n}\n\n/* Topbar WebAudio Toggle */\n.sfx-corner-btn {\n  min-height: 44px;\n  min-width: 44px;\n  padding: 6px 12px;\n  display: inline-flex;\n  align-items: center;\n  justify-content: center;\n  gap: 6px;\n  background-color: var(--surface-soft);\n  color: var(--ink);\n  border: var(--border);\n  border-radius: var(--radius-sm);\n  box-shadow: var(--shadow-sm);\n  font-size: 0.82rem;\n  font-weight: 700;\n  transition: transform 140ms var(--ease-pop), box-shadow 140ms var(--ease-pop), background-color 140ms ease;\n}\n\n.sfx-corner-btn .sfx-on {\n  display: none;\n}\n\n.sfx-corner-btn.is-active {\n  background-color: var(--green-aa);\n  color: #ffffff;\n}\n\n.sfx-corner-btn.is-active .sfx-off {\n  display: none;\n}\n\n.sfx-corner-btn.is-active .sfx-on {\n  display: inline-block;\n}\n\n.sfx-corner-btn:hover {\n  transform: translate(-1px, -1px);\n  box-shadow: 3px 3px 0 var(--ink);\n}\n\n.sfx-corner-btn:active {\n  transform: translate(2px, 2px);\n  box-shadow: 1px 1px 0 var(--ink);\n}\n\n/* =========================================================\n   APP STAGE & VIEW TRANSITIONS\n   ========================================================= */\n\n.app-stage {\n  flex: 1 1 auto;\n  min-height: 0;\n  position: relative;\n  width: 100%;\n}\n\n.view-shell {\n  width: 100%;\n  padding: clamp(12px, 2vh, 24px) clamp(16px, 2.5vw, 24px);\n}\n\n@media (min-width: 900px) {\n  .view-shell {\n    height: 100%;\n  }\n}\n\n.view-shell.home-shell,\n.view-shell.setup-shell {\n  display: flex;\n  align-items: center;\n  justify-content: center;\n}\n\n.view-shell.arena-view {\n  height: 100%;\n}\n\n.view-entering {\n  animation: viewSlideIn 250ms var(--ease-pop) both;\n}\n\n.view-leaving {\n  position: absolute !important;\n  inset: 0;\n  pointer-events: none;\n  z-index: 1;\n  animation: viewSlideOut 180ms ease-in both;\n}\n\n.animate-pop-in {\n  animation: stagePop 240ms var(--ease-pop) both;\n}\n\n@keyframes viewSlideIn {\n  from {\n    opacity: 0;\n    transform: translateY(10px) scale(0.985);\n  }\n  to {\n    opacity: 1;\n    transform: translateY(0) scale(1);\n  }\n}\n\n@keyframes viewSlideOut {\n  from {\n    opacity: 1;\n    transform: translateY(0) scale(1);\n  }\n  to {\n    opacity: 0;\n    transform: translateY(-8px) scale(0.985);\n  }\n}\n\n@keyframes stagePop {\n  from {\n    opacity: 0;\n    transform: scale(0.985);\n  }\n  to {\n    opacity: 1;\n    transform: scale(1);\n  }\n}\n\n/* Shared Surface & Buttons */\n.surface-card {\n  background-color: var(--surface);\n  border: var(--border-thick);\n  border-radius: var(--radius-lg);\n  box-shadow: var(--shadow-lg);\n}\n\n.btn-primary {\n  display: inline-flex;\n  align-items: center;\n  justify-content: center;\n  gap: 10px;\n  min-height: 48px;\n  padding: 10px 22px;\n  border: var(--border-thick);\n  border-radius: var(--radius-md);\n  background-color: var(--purple);\n  color: #ffffff;\n  font-size: 1rem;\n  font-weight: 800;\n  box-shadow: var(--shadow-md);\n  transition: transform 140ms var(--ease-pop), box-shadow 140ms var(--ease-pop), background-color 140ms ease;\n}\n\n.btn-primary:hover {\n  background-color: var(--purple-dark);\n  animation: ctaWiggle 360ms ease-in-out;\n  box-shadow: 5px 5px 0 var(--ink);\n}\n\n.btn-primary:active {\n  transform: translate(3px, 3px);\n  box-shadow: 1px 1px 0 var(--ink);\n}\n\n.btn-primary.compact {\n  min-height: 44px;\n  padding: 8px 16px;\n  font-size: 0.9rem;\n}\n\n.btn-nav {\n  display: inline-flex;\n  align-items: center;\n  justify-content: center;\n  gap: 6px;\n  min-height: 44px;\n  padding: 8px 14px;\n  border: var(--border);\n  border-radius: var(--radius-sm);\n  background-color: var(--surface);\n  color: var(--ink);\n  font-size: 0.88rem;\n  font-weight: 700;\n  box-shadow: var(--shadow-sm);\n  transition: transform 130ms var(--ease-pop), box-shadow 130ms var(--ease-pop), background-color 130ms ease;\n}\n\n.btn-nav:hover:not(:disabled) {\n  transform: translateY(-2px);\n  background-color: var(--yellow);\n  box-shadow: 3px 3px 0 var(--ink);\n}\n\n.btn-nav:active:not(:disabled) {\n  transform: translate(2px, 2px);\n  box-shadow: 0 0 0 var(--ink);\n}\n\n.btn-nav:disabled {\n  opacity: 0.45;\n  cursor: not-allowed;\n}\n\n.btn-header {\n  display: inline-flex;\n  align-items: center;\n  justify-content: center;\n  min-height: 40px;\n  padding: 6px 14px;\n  border: var(--border);\n  border-radius: var(--radius-sm);\n  background-color: var(--surface);\n  color: var(--ink);\n  font-size: 0.88rem;\n  font-weight: 700;\n  box-shadow: var(--shadow-sm);\n  transition: transform 130ms var(--ease-pop), box-shadow 130ms var(--ease-pop);\n}\n\n.btn-header:hover {\n  transform: translateY(-2px);\n  box-shadow: 3px 3px 0 var(--ink);\n}\n\n.btn-header:active {\n  transform: translate(2px, 2px);\n  box-shadow: 0 0 0 var(--ink);\n}\n\n@keyframes ctaWiggle {\n  0%, 100% { transform: rotate(0deg) translateY(-1px); }\n  30% { transform: rotate(-1.2deg) translateY(-2px); }\n  65% { transform: rotate(1.2deg) translateY(-2px); }\n}\n\n/* =========================================================\n   VIEW 1: LANDING HERO\n   ========================================================= */\n\n.home-card {\n  width: 100%;\n  max-width: 860px;\n  padding: clamp(20px, 3vw, 36px);\n  display: grid;\n  grid-template-columns: auto minmax(0, 1fr);\n  align-items: center;\n  gap: clamp(24px, 4vw, 44px);\n}\n\n.home-showcase {\n  display: flex;\n  justify-content: center;\n}\n\n.showcase-board {\n  display: grid;\n  grid-template-columns: repeat(5, clamp(34px, 5vw, 44px));\n  gap: 8px;\n  padding: 12px;\n  background-color: var(--blue);\n  border: var(--border-thick);\n  border-radius: var(--radius-md);\n  box-shadow: var(--shadow-md);\n  overflow: hidden;\n}\n\n.sc-slot {\n  width: clamp(34px, 5vw, 44px);\n  height: clamp(34px, 5vw, 44px);\n  display: flex;\n  align-items: center;\n  justify-content: center;\n}\n\n.sc-disc {\n  width: 100%;\n  height: 100%;\n  border-radius: 50%;\n  border: var(--border);\n  background-color: var(--surface);\n  display: block;\n}\n\n.sc-disc.is-empty {\n  background-color: var(--surface);\n}\n\n.sc-disc.p1 { background-color: var(--coral); }\n.sc-disc.p2 { background-color: var(--yellow); }\n.sc-disc.p3 { background-color: var(--green); }\n.sc-disc.p4 { background-color: var(--purple); }\n\n.sc-disc.loop-drop {\n  animation: demoLoopDrop 3.8s cubic-bezier(0.22, 1, 0.36, 1) infinite;\n}\n.sc-disc.d1 { animation-delay: 0s; }\n.sc-disc.d2 { animation-delay: 0.75s; }\n.sc-disc.d3 { animation-delay: 1.5s; }\n\n.sc-disc.is-win-demo {\n  animation: winPulseSoft 1.8s ease-in-out infinite;\n}\n\n@keyframes demoLoopDrop {\n  0%, 10% {\n    transform: translateY(-220%);\n    opacity: 0;\n  }\n  18% {\n    opacity: 1;\n  }\n  34% {\n    transform: translateY(0%);\n  }\n  40% {\n    transform: translateY(-16%);\n  }\n  47% {\n    transform: translateY(0%);\n  }\n  51% {\n    transform: translateY(-6%);\n  }\n  56%, 86% {\n    transform: translateY(0%);\n    opacity: 1;\n  }\n  96%, 100% {\n    transform: translateY(14px);\n    opacity: 0;\n  }\n}\n\n@keyframes winPulseSoft {\n  0%, 100% { transform: scale(1); }\n  50% { transform: scale(1.07); }\n}\n\n.home-content {\n  display: flex;\n  flex-direction: column;\n  gap: 16px;\n}\n\n.home-title {\n  margin: 0;\n  font-size: clamp(1.9rem, 3.6vw, 2.7rem);\n  font-weight: 800;\n  line-height: 1.1;\n  letter-spacing: -0.02em;\n  transform-origin: left center;\n  animation: titleBreathe 4.2s ease-in-out infinite;\n}\n\n@keyframes titleBreathe {\n  0%, 100% { transform: scale(1); }\n  50% { transform: scale(1.018); }\n}\n\n.home-subtitle {\n  margin: 0;\n  font-size: clamp(0.98rem, 1.3vw, 1.12rem);\n  color: var(--ink-muted);\n  font-weight: 500;\n  max-width: 40ch;\n}\n\n.home-actions {\n  display: flex;\n  flex-direction: column;\n  align-items: flex-start;\n  gap: 14px;\n  margin-top: 6px;\n}\n\n.hero-cta {\n  width: 100%;\n  max-width: 280px;\n  min-height: 54px;\n  font-size: 1.08rem;\n}\n\n.home-links {\n  display: flex;\n  align-items: center;\n  gap: 12px;\n}\n\n.text-link-btn {\n  min-height: 44px;\n  padding: 6px 4px;\n  border: none;\n  background: transparent;\n  font-size: 0.94rem;\n  font-weight: 700;\n  color: var(--ink);\n  text-decoration: underline;\n  text-underline-offset: 4px;\n  text-decoration-thickness: 2px;\n  transition: color 120ms ease, transform 120ms ease;\n}\n\n.text-link-btn:hover {\n  color: var(--purple);\n  transform: translateY(-1px);\n}\n\n.link-sep {\n  color: var(--ink-muted);\n  font-weight: 800;\n}\n\n/* =========================================================\n   VIEW 2: SETUP AS 2-COLUMN GAME LOBBY\n   ========================================================= */\n\n.lobby-card {\n  width: 100%;\n  max-width: 1180px;\n  overflow: hidden;\n  display: flex;\n  flex-direction: column;\n  max-height: 100%;\n}\n\n.card-header.lobby-header {\n  display: grid;\n  grid-template-columns: auto 1fr auto;\n  align-items: center;\n  gap: 16px;\n  padding: 10px clamp(16px, 2.2vw, 24px);\n  background-color: var(--coral);\n  border-bottom: var(--border-thick);\n}\n\n.header-title {\n  margin: 0;\n  text-align: center;\n  font-size: clamp(1rem, 1.45vw, 1.2rem);\n  font-weight: 800;\n  color: var(--ink);\n}\n\n.status-chip {\n  display: inline-flex;\n  align-items: center;\n  padding: 6px 12px;\n  background-color: var(--yellow);\n  border: var(--border);\n  border-radius: var(--radius-sm);\n  font-size: 0.84rem;\n  font-weight: 800;\n  white-space: nowrap;\n}\n\n.lobby-body {\n  display: grid;\n  grid-template-columns: minmax(0, 1.04fr) minmax(0, 1fr);\n  gap: clamp(20px, 3vw, 36px);\n  padding: clamp(16px, 2.4vh, 24px) clamp(20px, 2.8vw, 32px);\n  align-items: stretch;\n}\n\n.lobby-col {\n  display: flex;\n  flex-direction: column;\n  justify-content: space-between;\n  gap: clamp(16px, 2.4vh, 24px);\n  min-width: 0;\n}\n\n.lobby-controls-stack {\n  display: flex;\n  flex-direction: column;\n  gap: clamp(16px, 2.4vh, 24px);\n}\n\n.lobby-group {\n  display: flex;\n  flex-direction: column;\n  gap: 10px;\n}\n\n.group-label {\n  display: block;\n  font-size: 0.92rem;\n  font-weight: 800;\n  color: var(--ink);\n}\n\n/* Uniform Choice Cards */\n.choice-grid {\n  display: grid;\n  gap: 12px;\n}\n\n.choice-grid.cols-2 {\n  grid-template-columns: repeat(2, minmax(0, 1fr));\n}\n\n.choice-grid.cols-4 {\n  grid-template-columns: repeat(4, minmax(0, 1fr));\n}\n\n.choice-grid.cols-3-always {\n  grid-template-columns: repeat(3, minmax(0, 1fr));\n}\n\n.choice-grid.cols-5-first {\n  grid-template-columns: repeat(5, minmax(0, 1fr));\n}\n\n.choice-card {\n  position: relative;\n  display: flex;\n  align-items: center;\n  gap: 12px;\n  min-height: clamp(54px, 7vh, 66px);\n  padding: clamp(10px, 1.4vh, 14px) clamp(12px, 1.4vw, 16px);\n  background-color: var(--surface);\n  border: var(--border);\n  border-radius: var(--radius-md);\n  box-shadow: var(--shadow-sm);\n  text-align: left;\n  transition: transform 150ms var(--ease-pop), box-shadow 150ms var(--ease-pop), background-color 150ms ease;\n}\n\n.choice-card:hover {\n  transform: translateY(-2px);\n  box-shadow: var(--shadow-md);\n}\n\n.choice-card:active {\n  transform: translate(2px, 2px);\n  box-shadow: 0 0 0 var(--ink);\n}\n\n.choice-card.active {\n  background-color: var(--yellow);\n  box-shadow: var(--shadow-md);\n}\n\n.choice-card.just-picked,\n.size-btn.just-picked,\n.seat-btn.just-picked {\n  animation: cardPop 220ms var(--ease-pop);\n}\n\n@keyframes cardPop {\n  0% { transform: scale(0.98); }\n  55% { transform: scale(1.04); }\n  100% { transform: scale(1); }\n}\n\n.choice-icon {\n  width: 38px;\n  height: 38px;\n  flex: 0 0 auto;\n  display: inline-flex;\n  align-items: center;\n  justify-content: center;\n  border: 2px solid var(--ink);\n  border-radius: var(--radius-sm);\n  background-color: var(--surface-soft);\n  color: var(--ink);\n}\n\n.choice-icon.coral { background-color: var(--coral); }\n.choice-icon.yellow { background-color: var(--yellow); }\n.choice-icon.green { background-color: var(--green); color: #ffffff; }\n.choice-icon.purple { background-color: var(--purple); color: #ffffff; }\n.choice-icon.blue { background-color: var(--blue); color: #ffffff; }\n.choice-icon.dark { background-color: var(--stone); color: #ffffff; }\n.choice-icon.compact { width: 28px; height: 28px; }\n\n.choice-title {\n  font-size: clamp(0.88rem, 1.05vw, 0.96rem);\n  font-weight: 800;\n  line-height: 1.2;\n}\n\n.choice-text {\n  display: flex;\n  flex-direction: column;\n  gap: 2px;\n  min-width: 0;\n}\n\n.choice-sub {\n  font-size: 0.78rem;\n  font-weight: 600;\n  color: var(--ink-muted);\n}\n\n.choice-check {\n  margin-left: auto;\n  width: 18px;\n  height: 18px;\n  flex: 0 0 auto;\n  border-radius: 50%;\n  border: 2px solid var(--ink);\n  background-color: var(--surface);\n  position: relative;\n  opacity: 0.35;\n  transition: opacity 150ms ease, transform 180ms var(--ease-pop), background-color 150ms ease;\n}\n\n.choice-card.active .choice-check {\n  opacity: 1;\n  background-color: var(--purple);\n  transform: scale(1.08);\n}\n\n.choice-card.active .choice-check::after {\n  content: \"\";\n  position: absolute;\n  left: 5px;\n  top: 2px;\n  width: 4px;\n  height: 8px;\n  border: solid #ffffff;\n  border-width: 0 2px 2px 0;\n  transform: rotate(45deg);\n}\n\n/* Difficulty Vertical Card Layout (4 cols on 1 row) */\n.choice-card.diff-card {\n  flex-direction: column;\n  justify-content: center;\n  align-items: center;\n  gap: 8px;\n  min-height: clamp(64px, 8vh, 76px);\n  padding: 10px 8px;\n  text-align: center;\n}\n\n.choice-card.diff-card .choice-check {\n  position: absolute;\n  top: 6px;\n  right: 6px;\n  margin-left: 0;\n  width: 14px;\n  height: 14px;\n}\n\n.choice-card.diff-card.active .choice-check::after {\n  left: 3.5px;\n  top: 1px;\n  width: 3px;\n  height: 6px;\n}\n\n.signal-bars {\n  display: inline-flex;\n  align-items: flex-end;\n  gap: 3px;\n  height: 22px;\n}\n\n.s-bar {\n  width: 6px;\n  border: 1.8px solid var(--ink);\n  border-radius: 2px;\n  background-color: var(--surface-soft);\n}\n\n.s-bar:nth-child(1) { height: 9px; }\n.s-bar:nth-child(2) { height: 15px; }\n.s-bar:nth-child(3) { height: 21px; }\n\n.signal-bars.lvl-1 .s-bar.active { background-color: var(--green); }\n.signal-bars.lvl-2 .s-bar.active { background-color: var(--blue); }\n.signal-bars.lvl-3 .s-bar.active { background-color: var(--coral); }\n\n/* Segmented Bar (Board Size & 4P Presets) */\n.board-config-stack {\n  display: flex;\n  flex-direction: column;\n  gap: 12px;\n}\n\n.segmented-bar {\n  display: grid;\n  border: var(--border);\n  border-radius: var(--radius-md);\n  overflow: hidden;\n  background-color: var(--surface);\n  box-shadow: var(--shadow-sm);\n}\n\n.segmented-bar.cols-4 {\n  grid-template-columns: repeat(4, minmax(0, 1fr));\n}\n\n.segmented-bar.compact {\n  grid-template-columns: repeat(4, auto);\n}\n\n.size-btn,\n.preset-chip {\n  min-height: 44px;\n  padding: 8px 10px;\n  border: none;\n  border-right: var(--border);\n  background-color: var(--surface);\n  font-size: 0.9rem;\n  font-weight: 800;\n  transition: background-color 140ms ease;\n}\n\n.size-btn:last-child,\n.preset-chip:last-child {\n  border-right: none;\n}\n\n.size-btn:hover,\n.preset-chip:hover {\n  background-color: var(--surface-soft);\n}\n\n.size-btn.active,\n.preset-chip.active {\n  background-color: var(--yellow);\n}\n\n/* Live Preview Panel (Lobby Left Bottom) */\n.preview-group {\n  flex: 1 1 auto;\n  display: flex;\n  flex-direction: column;\n}\n\n.preview-meta-row {\n  display: flex;\n  align-items: center;\n  justify-content: space-between;\n  gap: 12px;\n}\n\n.preview-meta-pill {\n  font-size: 0.8rem;\n  font-weight: 700;\n  color: var(--ink);\n  background-color: var(--surface-soft);\n  border: 2px solid var(--ink);\n  border-radius: var(--radius-sm);\n  padding: 2px 8px;\n}\n\n.lobby-preview-box {\n  flex: 1 1 auto;\n  display: flex;\n  flex-direction: column;\n  justify-content: space-between;\n  gap: 10px;\n  padding: clamp(12px, 1.8vh, 16px);\n  background-color: var(--surface-blue);\n  border: var(--border);\n  border-radius: var(--radius-md);\n  box-shadow: var(--shadow-sm);\n}\n\n.preview-player-chips {\n  display: flex;\n  flex-wrap: wrap;\n  gap: 8px;\n  align-items: center;\n}\n\n.pv-chip {\n  display: inline-flex;\n  align-items: center;\n  gap: 6px;\n  padding: 3px 8px;\n  background-color: var(--surface);\n  border: 1.8px solid var(--ink);\n  border-radius: var(--radius-sm);\n  font-size: 0.76rem;\n  font-weight: 700;\n}\n\n.pv-dot {\n  width: 10px;\n  height: 10px;\n  border-radius: 50%;\n  border: 1.5px solid var(--ink);\n}\n\n.pv-dot.p1 { background-color: var(--coral); }\n.pv-dot.p2 { background-color: var(--yellow); }\n.pv-dot.p3 { background-color: var(--green); }\n.pv-dot.p4 { background-color: var(--purple); }\n.pv-dot.obs { background-color: var(--stone); border-radius: 2px; }\n\n.lobby-preview-stage {\n  display: flex;\n  align-items: center;\n  justify-content: center;\n  min-height: clamp(120px, 19vh, 180px);\n}\n\n.lobby-preview-board {\n  display: grid;\n  grid-template-columns: repeat(var(--p-cols, 7), minmax(0, 1fr));\n  gap: clamp(3px, 0.5vw, 6px);\n  padding: 8px;\n  background-color: var(--blue);\n  border: var(--border);\n  border-radius: var(--radius-md);\n  box-shadow: var(--shadow-sm);\n  width: min(100%, calc(var(--p-cols, 7) * 26px + 32px));\n}\n\n.p-cell {\n  aspect-ratio: 1;\n  border-radius: 50%;\n  border: 1.8px solid var(--ink);\n  background-color: var(--surface);\n  transition: transform 180ms var(--ease-pop), background-color 180ms ease;\n}\n\n.p-cell.p1 { background-color: var(--coral); }\n.p-cell.p2 { background-color: var(--yellow); }\n.p-cell.p3 { background-color: var(--green); }\n.p-cell.p4 { background-color: var(--purple); }\n\n.p-cell.is-stone {\n  border-radius: 3px;\n  background-color: var(--stone);\n  position: relative;\n}\n\n.p-cell.is-stone::after {\n  content: \"\";\n  position: absolute;\n  inset: 3px;\n  border: 1.5px solid rgba(255, 255, 255, 0.55);\n  border-radius: 2px;\n}\n\n/* Lobby Right Footer Actions */\n.lobby-actions-footer {\n  display: flex;\n  flex-direction: column;\n  gap: 12px;\n}\n\n.advanced-trigger-btn {\n  display: flex;\n  align-items: center;\n  justify-content: space-between;\n  gap: 12px;\n  width: 100%;\n  min-height: 46px;\n  padding: 8px 14px;\n  background-color: var(--surface-soft);\n  border: var(--border);\n  border-radius: var(--radius-md);\n  box-shadow: var(--shadow-sm);\n  font-size: 0.9rem;\n  font-weight: 800;\n  transition: transform 140ms var(--ease-pop), background-color 140ms ease, box-shadow 140ms ease;\n}\n\n.advanced-trigger-btn:hover {\n  transform: translateY(-2px);\n  background-color: var(--surface);\n  box-shadow: var(--shadow-md);\n}\n\n.advanced-trigger-btn.has-highlight {\n  animation: cardPop 260ms var(--ease-pop);\n  background-color: var(--yellow);\n}\n\n.adv-btn-left {\n  display: inline-flex;\n  align-items: center;\n  gap: 8px;\n}\n\n.adv-summary-pill {\n  padding: 3px 9px;\n  background-color: var(--surface);\n  border: 2px solid var(--ink);\n  border-radius: var(--radius-sm);\n  font-size: 0.78rem;\n  font-weight: 700;\n  white-space: nowrap;\n}\n\n.start-cta {\n  width: 100%;\n  min-height: clamp(48px, 6.5vh, 56px);\n  font-size: 1.05rem;\n}\n\n/* =========================================================\n   OVERLAY DRAWER (OPSI LANJUTAN) & MODALS\n   ========================================================= */\n\n.drawer-backdrop,\n.modal-backdrop {\n  position: fixed;\n  inset: 0;\n  z-index: 120;\n  background-color: rgba(17, 17, 17, 0.58);\n  display: flex;\n  align-items: center;\n  justify-content: center;\n  padding: 20px;\n  animation: backdropIn 180ms ease both;\n}\n\n.drawer-backdrop.is-closing,\n.modal-backdrop.is-closing {\n  animation: backdropOut 180ms ease both;\n}\n\n.drawer-panel,\n.modal-card {\n  width: 100%;\n  max-width: 700px;\n  max-height: min(90dvh, 720px);\n  display: flex;\n  flex-direction: column;\n  background-color: var(--surface);\n  border: var(--border-thick);\n  border-radius: var(--radius-lg);\n  box-shadow: var(--shadow-lg);\n  overflow: hidden;\n  animation: modalCardIn 220ms var(--ease-pop) both;\n}\n\n.drawer-backdrop.is-closing .drawer-panel,\n.modal-backdrop.is-closing .modal-card {\n  animation: modalCardOut 180ms ease both;\n}\n\n@keyframes backdropIn {\n  from { background-color: rgba(17, 17, 17, 0); }\n  to { background-color: rgba(17, 17, 17, 0.58); }\n}\n@keyframes backdropOut {\n  from { background-color: rgba(17, 17, 17, 0.58); }\n  to { background-color: rgba(17, 17, 17, 0); }\n}\n@keyframes modalCardIn {\n  from { opacity: 0; transform: translateY(14px) scale(0.97); }\n  to { opacity: 1; transform: translateY(0) scale(1); }\n}\n@keyframes modalCardOut {\n  from { opacity: 1; transform: translateY(0) scale(1); }\n  to { opacity: 0; transform: translateY(10px) scale(0.97); }\n}\n\n.drawer-header,\n.modal-header {\n  display: flex;\n  align-items: center;\n  justify-content: space-between;\n  gap: 16px;\n  padding: 16px 24px;\n  background-color: var(--yellow);\n  border-bottom: var(--border-thick);\n}\n\n.drawer-title,\n.modal-title {\n  margin: 0;\n  font-size: 1.15rem;\n  font-weight: 800;\n}\n\n.modal-close-btn {\n  width: 44px;\n  height: 44px;\n  display: inline-flex;\n  align-items: center;\n  justify-content: center;\n  background-color: var(--surface);\n  border: var(--border);\n  border-radius: var(--radius-sm);\n  box-shadow: var(--shadow-sm);\n  font-size: 1.4rem;\n  font-weight: 800;\n  line-height: 1;\n}\n\n.modal-close-btn:hover {\n  background-color: var(--coral);\n}\n\n.drawer-body,\n.modal-body {\n  padding: 20px 24px;\n  overflow-y: auto;\n  display: flex;\n  flex-direction: column;\n  gap: 24px;\n}\n\n.drawer-section {\n  display: flex;\n  flex-direction: column;\n  gap: 12px;\n}\n\n.drawer-section-head {\n  display: flex;\n  align-items: center;\n  justify-content: space-between;\n  flex-wrap: wrap;\n  gap: 12px;\n}\n\n.seats-grid {\n  display: grid;\n  grid-template-columns: repeat(4, minmax(0, 1fr));\n  gap: 12px;\n}\n\n.seat-btn {\n  display: flex;\n  flex-direction: column;\n  align-items: center;\n  gap: 8px;\n  min-height: 44px;\n  padding: 14px 10px;\n  background-color: var(--surface);\n  border: var(--border);\n  border-radius: var(--radius-md);\n  box-shadow: var(--shadow-sm);\n  transition: transform 140ms var(--ease-pop), box-shadow 140ms ease;\n}\n\n.seat-btn:hover {\n  transform: translateY(-2px);\n  box-shadow: var(--shadow-md);\n}\n\n.seat-name {\n  font-size: 0.84rem;\n  font-weight: 800;\n}\n\n.seat-role-tag {\n  padding: 2px 10px;\n  border: 2px solid var(--ink);\n  border-radius: var(--radius-sm);\n  background-color: var(--yellow);\n  font-size: 0.76rem;\n  font-weight: 800;\n}\n\n.seat-role-tag.is-bot {\n  background-color: var(--purple);\n  color: #ffffff;\n}\n\n.disc-badge {\n  width: 24px;\n  height: 24px;\n  border-radius: 50%;\n  border: 2px solid var(--ink);\n  flex: 0 0 auto;\n  display: inline-flex;\n  align-items: center;\n  justify-content: center;\n  font-size: 0.78rem;\n  font-weight: 800;\n}\n\n.disc-badge.p1 { background-color: var(--coral); }\n.disc-badge.p2 { background-color: var(--yellow); }\n.disc-badge.p3 { background-color: var(--green); }\n.disc-badge.p4 { background-color: var(--purple); }\n.disc-badge.random { background-color: var(--surface-soft); }\n\n.choice-card.first-card {\n  flex-direction: column;\n  align-items: center;\n  justify-content: center;\n  text-align: center;\n  padding: 12px 8px;\n  gap: 8px;\n  min-height: 76px;\n}\n\n.custom-select-row {\n  display: grid;\n  grid-template-columns: repeat(2, minmax(0, 1fr));\n  gap: 16px;\n}\n\n.field-group {\n  display: flex;\n  flex-direction: column;\n  gap: 6px;\n  font-size: 0.86rem;\n  font-weight: 700;\n}\n\n.field-select {\n  min-height: 44px;\n  padding: 8px 12px;\n  border: var(--border);\n  border-radius: var(--radius-sm);\n  background-color: var(--surface);\n  font-size: 0.9rem;\n  font-weight: 700;\n}\n\n.field-select.compact {\n  min-height: 40px;\n  padding: 6px 10px;\n}\n\n.drawer-footer {\n  padding: 14px 24px;\n  border-top: var(--border);\n  background-color: var(--surface-soft);\n  display: flex;\n  justify-content: flex-end;\n}\n\n/* Modal Help Tabs & Rows */\n.tab-bar {\n  display: grid;\n  grid-template-columns: repeat(2, minmax(0, 1fr));\n  border-bottom: var(--border);\n  background-color: var(--surface-soft);\n}\n\n.tab-btn {\n  min-height: 44px;\n  padding: 10px 14px;\n  border: none;\n  border-right: var(--border);\n  background-color: transparent;\n  font-size: 0.92rem;\n  font-weight: 800;\n  transition: background-color 140ms ease;\n}\n\n.tab-btn:last-child {\n  border-right: none;\n}\n\n.tab-btn.active {\n  background-color: var(--yellow);\n}\n\n.help-panel {\n  display: flex;\n  flex-direction: column;\n  gap: 14px;\n}\n\n.help-list {\n  margin: 0;\n  padding-left: 20px;\n  display: flex;\n  flex-direction: column;\n  gap: 12px;\n  font-size: 0.94rem;\n}\n\n.help-intro {\n  margin: 0;\n  font-size: 0.94rem;\n}\n\n.legend-rows {\n  display: flex;\n  flex-direction: column;\n  gap: 10px;\n}\n\n.legend-row {\n  display: flex;\n  align-items: center;\n  justify-content: space-between;\n  gap: 12px;\n  padding: 10px 12px;\n  background-color: var(--surface-soft);\n  border: 2px solid var(--ink);\n  border-radius: var(--radius-sm);\n}\n\n.legend-row.positive {\n  background-color: #eafaf1;\n}\n\n.legend-row.negative {\n  background-color: #feefec;\n}\n\n.legend-left {\n  display: flex;\n  align-items: center;\n  gap: 10px;\n  font-size: 0.84rem;\n  font-weight: 700;\n}\n\n.mini-tiles {\n  display: inline-flex;\n  gap: 3px;\n}\n\n.mini-tile {\n  width: 18px;\n  height: 18px;\n  border: 1.5px solid var(--ink);\n  border-radius: 3px;\n  background-color: var(--blue);\n  display: inline-flex;\n  align-items: center;\n  justify-content: center;\n}\n\n.mini-dot {\n  width: 11px;\n  height: 11px;\n  border-radius: 50%;\n  border: 1.5px solid var(--ink);\n}\n\n.mini-dot.ai { background-color: var(--yellow); }\n.mini-dot.hu { background-color: var(--coral); }\n.mini-dot.em { background-color: var(--surface); }\n\n.legend-right {\n  display: flex;\n  align-items: center;\n  gap: 8px;\n  font-size: 0.82rem;\n  font-weight: 800;\n}\n\n.weight-pill {\n  padding: 2px 7px;\n  background-color: var(--surface);\n  border: 1.8px solid var(--ink);\n  border-radius: var(--radius-sm);\n}\n\n/* =========================================================\n   VIEW 3: ARENA GAME + POST-GAME SPLIT\n   ========================================================= */\n\n.confetti-canvas {\n  position: fixed;\n  inset: 0;\n  width: 100%;\n  height: 100%;\n  pointer-events: none;\n  z-index: 80;\n}\n\n.arena-shell {\n  width: 100%;\n  max-width: 820px;\n  margin: 0 auto;\n  display: flex;\n  flex-direction: column;\n  max-height: 100%;\n  overflow: hidden;\n  transition: max-width 280ms var(--ease-pop);\n}\n\n.arena-shell.is-wide-arena {\n  max-width: 920px;\n}\n\n.arena-shell.has-analysis {\n  max-width: 1180px;\n  height: 100%;\n}\n\n/* Minimal Arena Top Bar */\n.arena-topbar {\n  flex: 0 0 auto;\n  display: grid;\n  grid-template-columns: auto 1fr auto;\n  align-items: center;\n  gap: 12px;\n  padding: 10px 18px;\n  background-color: var(--surface-soft);\n  border-bottom: var(--border-thick);\n}\n\n.arena-top-left,\n.arena-top-right {\n  display: flex;\n  align-items: center;\n  gap: 10px;\n}\n\n.arena-top-center {\n  display: flex;\n  justify-content: center;\n  min-width: 0;\n}\n\n.turn-pill {\n  display: inline-flex;\n  align-items: center;\n  gap: 8px;\n  min-height: 40px;\n  padding: 6px 14px;\n  background-color: var(--surface);\n  border: var(--border);\n  border-radius: var(--radius-md);\n  box-shadow: var(--shadow-sm);\n  font-size: 0.92rem;\n  font-weight: 800;\n  max-width: 100%;\n}\n\n.turn-pill.turn-chip-pop {\n  animation: chipPop 240ms var(--ease-pop);\n}\n\n@keyframes chipPop {\n  0% { transform: scale(0.95); }\n  55% { transform: scale(1.05); }\n  100% { transform: scale(1); }\n}\n\n.turn-pill.turn-p1 { background-color: #ffe9e6; }\n.turn-pill.turn-p2 { background-color: #fff3d6; }\n.turn-pill.turn-p3 { background-color: #e6f9ef; }\n.turn-pill.turn-p4 { background-color: #efe9fc; }\n.turn-pill.game-win { background-color: var(--yellow); }\n.turn-pill.game-draw { background-color: var(--surface-soft); }\n\n.status-dot {\n  width: 16px;\n  height: 16px;\n  border-radius: 50%;\n  border: 2px solid var(--ink);\n  flex: 0 0 auto;\n}\n\n.status-dot.p1 { background-color: var(--coral); }\n.status-dot.p2 { background-color: var(--yellow); }\n.status-dot.p3 { background-color: var(--green); }\n.status-dot.p4 { background-color: var(--purple); }\n.status-dot.neutral { background-color: var(--stone); }\n\n.status-label {\n  white-space: nowrap;\n  overflow: hidden;\n  text-overflow: ellipsis;\n}\n\n.thinking-dots {\n  display: none;\n  align-items: center;\n  gap: 3px;\n}\n\n.thinking-dots.active {\n  display: inline-flex;\n}\n\n.thinking-dots span {\n  width: 6px;\n  height: 6px;\n  border-radius: 50%;\n  background-color: var(--ink);\n  animation: dotBounce 1s infinite ease-in-out;\n}\n\n.thinking-dots span:nth-child(2) { animation-delay: 0.16s; }\n.thinking-dots span:nth-child(3) { animation-delay: 0.32s; }\n\n@keyframes dotBounce {\n  0%, 80%, 100% { transform: translateY(0); opacity: 0.4; }\n  40% { transform: translateY(-4px); opacity: 1; }\n}\n\n.turn-count-chip,\n.last-move-chip {\n  padding: 2px 7px;\n  border: 1.8px solid var(--ink);\n  border-radius: var(--radius-sm);\n  background-color: var(--surface);\n  font-size: 0.76rem;\n  font-weight: 800;\n  white-space: nowrap;\n}\n\n/* Overflow \"⋯\" Popover Menu */\n.overflow-menu {\n  position: relative;\n}\n\n.overflow-trigger {\n  list-style: none;\n  width: 44px;\n  height: 44px;\n  display: inline-flex;\n  align-items: center;\n  justify-content: center;\n  background-color: var(--surface);\n  border: var(--border);\n  border-radius: var(--radius-sm);\n  box-shadow: var(--shadow-sm);\n  font-size: 1.35rem;\n  font-weight: 800;\n  cursor: pointer;\n}\n\n.overflow-trigger::-webkit-details-marker {\n  display: none;\n}\n\n.overflow-trigger:hover {\n  background-color: var(--yellow);\n}\n\n.overflow-popover {\n  position: absolute;\n  top: calc(100% + 8px);\n  right: 0;\n  width: 260px;\n  z-index: 60;\n  padding: 14px;\n  background-color: var(--surface);\n  border: var(--border-thick);\n  border-radius: var(--radius-md);\n  box-shadow: var(--shadow-md);\n  display: flex;\n  flex-direction: column;\n  gap: 10px;\n}\n\n.popover-meta {\n  display: flex;\n  flex-direction: column;\n  gap: 3px;\n  padding-bottom: 8px;\n  border-bottom: 2px solid var(--surface-soft);\n}\n\n.popover-mode-tag {\n  font-size: 0.78rem;\n  font-weight: 800;\n}\n\n.popover-cfg-tag {\n  font-size: 0.76rem;\n  font-weight: 600;\n  color: var(--ink-muted);\n}\n\n.popover-row {\n  display: flex;\n  align-items: center;\n  justify-content: space-between;\n  gap: 8px;\n  font-size: 0.85rem;\n  font-weight: 700;\n}\n\n.popover-bvb {\n  display: flex;\n  gap: 8px;\n}\n\n.btn-soft,\n.btn-green,\n.btn-popover-item {\n  min-height: 40px;\n  padding: 8px 12px;\n  border: var(--border);\n  border-radius: var(--radius-sm);\n  font-size: 0.86rem;\n  font-weight: 800;\n  text-align: left;\n  background-color: var(--surface-soft);\n}\n\n.btn-soft {\n  flex: 1;\n  text-align: center;\n}\n\n.btn-green {\n  flex: 1;\n  text-align: center;\n  background-color: var(--green-aa);\n  color: #ffffff;\n}\n\n.btn-popover-item:hover {\n  background-color: var(--yellow);\n}\n\n.btn-popover-item.subtle {\n  background-color: var(--surface);\n}\n\n/* Arena Workspace (Board Centered -> Split on Post-Game) */\n.arena-workspace {\n  flex: 1 1 auto;\n  min-height: 0;\n  padding: clamp(10px, 1.8vh, 18px) clamp(14px, 2.2vw, 22px);\n  display: flex;\n  flex-direction: column;\n  align-items: center;\n  justify-content: center;\n  gap: 16px;\n  overflow-y: auto;\n}\n\n@media (min-width: 960px) {\n  .arena-shell.has-analysis .arena-workspace {\n    display: grid;\n    grid-template-columns: auto minmax(340px, 470px);\n    align-items: stretch;\n    justify-content: center;\n    gap: 28px;\n  }\n}\n\n.board-column {\n  --cell-size: min(\n    calc((100dvh - 430px) / var(--board-rows, 6)),\n    calc((min(90vw, 660px) - 88px) / var(--board-cols, 7)),\n    64px\n  );\n  display: flex;\n  flex-direction: column;\n  align-items: center;\n  justify-content: center;\n  gap: 6px;\n}\n\n.arena-shell.has-analysis .board-column {\n  --cell-size: min(\n    calc((100dvh - 425px) / var(--board-rows, 6)),\n    calc(520px / var(--board-cols, 7)),\n    56px\n  );\n}\n\n.board-stage {\n  display: flex;\n  flex-direction: column;\n  align-items: center;\n  gap: 6px;\n}\n\n/* Hover Ghost Row */\n.hover-preview-row {\n  display: grid;\n  grid-template-columns: repeat(var(--board-cols, 7), var(--cell-size));\n  gap: clamp(5px, 0.85vh, 9px);\n  padding: 0 clamp(10px, 1.4vh, 14px);\n  height: calc(var(--cell-size) * 0.68);\n  min-height: 26px;\n  transition: opacity 150ms ease;\n}\n\n.hover-preview-row.is-idle {\n  opacity: 0;\n  pointer-events: none;\n}\n\n.ghost-slot {\n  display: flex;\n  align-items: center;\n  justify-content: center;\n}\n\n.ghost-disc {\n  width: calc(var(--cell-size) * 0.68);\n  height: calc(var(--cell-size) * 0.68);\n  border-radius: 50%;\n  border: 2px solid var(--ink);\n  opacity: 0;\n  transform: translateY(-6px) scale(0.85);\n  transition: opacity 140ms ease, transform 180ms var(--ease-pop);\n}\n\n.ghost-disc.visible {\n  opacity: 0.92;\n  transform: translateY(0) scale(1);\n}\n\n.ghost-disc.p1 { background-color: var(--coral); }\n.ghost-disc.p2 { background-color: var(--yellow); }\n.ghost-disc.p3 { background-color: var(--green); }\n.ghost-disc.p4 { background-color: var(--purple); }\n\n/* Board Frame & SVG Win Line Layer */\n.board-frame {\n  position: relative;\n  display: inline-block;\n}\n\n.board-grid {\n  position: relative;\n  display: grid;\n  grid-template-columns: repeat(var(--board-cols, 7), var(--cell-size));\n  gap: clamp(5px, 0.85vh, 9px);\n  padding: clamp(10px, 1.4vh, 14px);\n  background-color: var(--blue);\n  border: var(--border-thick);\n  border-radius: var(--radius-lg);\n  box-shadow: var(--shadow-lg);\n}\n\n.win-line-svg {\n  position: absolute;\n  inset: 0;\n  width: 100%;\n  height: 100%;\n  pointer-events: none;\n  z-index: 6;\n  overflow: visible;\n}\n\n.win-beam-glow {\n  stroke: #ffffff;\n  stroke-width: 14px;\n  stroke-linecap: round;\n  opacity: 0.55;\n  animation: winBeamDraw 420ms var(--ease-pop) both;\n}\n\n.win-beam-core {\n  stroke: var(--ink);\n  stroke-width: 6px;\n  stroke-linecap: round;\n  animation: winBeamDraw 420ms var(--ease-pop) both;\n}\n\n@keyframes winBeamDraw {\n  from {\n    stroke-dasharray: 900;\n    stroke-dashoffset: 900;\n  }\n  to {\n    stroke-dasharray: 900;\n    stroke-dashoffset: 0;\n  }\n}\n\n.cell {\n  width: var(--cell-size);\n  height: var(--cell-size);\n  border-radius: 50%;\n  border: var(--border);\n  background-color: var(--surface);\n  display: flex;\n  align-items: center;\n  justify-content: center;\n  cursor: pointer;\n  position: relative;\n  transition: background-color 120ms ease, transform 120ms ease;\n}\n\n.cell.disabled {\n  cursor: default;\n}\n\n.cell.col-active-hover {\n  background-color: #fff6de;\n}\n\n.cell.target-landing {\n  background-color: var(--yellow);\n  transform: scale(0.96);\n}\n\n.cell.is-blocked {\n  border-radius: var(--radius-sm);\n  background-color: var(--stone);\n  cursor: not-allowed;\n}\n\n.cell.is-blocked::after {\n  content: \"\";\n  width: 44%;\n  height: 44%;\n  border: 2px solid rgba(255, 255, 255, 0.55);\n  border-radius: 2px;\n}\n\n.disc {\n  width: 88%;\n  height: 88%;\n  border-radius: 50%;\n  border: var(--border);\n  position: relative;\n}\n\n.disc.p1 { background-color: var(--coral); }\n.disc.p2 { background-color: var(--yellow); }\n.disc.p3 { background-color: var(--green); }\n.disc.p4 { background-color: var(--purple); }\n\n.disc.is-last-disc::after {\n  content: \"\";\n  position: absolute;\n  top: 50%;\n  left: 50%;\n  width: 26%;\n  height: 26%;\n  transform: translate(-50%, -50%);\n  border-radius: 50%;\n  background-color: var(--surface);\n  border: 2px solid var(--ink);\n}\n\n.disc.drop-anim {\n  animation: discGravityDrop var(--drop-duration, 320ms) cubic-bezier(0.3, 0, 0.7, 1) both;\n  transform-origin: center bottom;\n}\n\n@keyframes discGravityDrop {\n  0% {\n    transform: translateY(calc(var(--drop-rows, 1) * -106%)) scale(0.92, 1.08);\n  }\n  68% {\n    transform: translateY(0%) scale(1.18, 0.82);\n  }\n  80% {\n    transform: translateY(-16%) scale(0.95, 1.05);\n  }\n  90% {\n    transform: translateY(0%) scale(1.05, 0.96);\n  }\n  96% {\n    transform: translateY(-4%) scale(0.99, 1.01);\n  }\n  100% {\n    transform: translateY(0%) scale(1, 1);\n  }\n}\n\n/* Board Shake on Impact */\n.board-grid.board-impact {\n  animation: boardImpactShake 160ms var(--ease-pop) both;\n}\n\n@keyframes boardImpactShake {\n  0% { transform: translateY(0); }\n  25% { transform: translateY(var(--shake-px, 1.8px)); }\n  50% { transform: translateY(calc(var(--shake-px, 1.8px) * -0.6)); }\n  75% { transform: translateY(calc(var(--shake-px, 1.8px) * 0.3)); }\n  100% { transform: translateY(0); }\n}\n\n.board-grid.board-draw-shake {\n  animation: boardDrawShake 420ms ease-in-out both;\n}\n\n@keyframes boardDrawShake {\n  0%, 100% { transform: translate(0, 0); }\n  20% { transform: translate(-2px, 1.5px); }\n  40% { transform: translate(2px, -1.5px); }\n  60% { transform: translate(-1.5px, -1px); }\n  80% { transform: translate(1.5px, 1px); }\n}\n\n/* Column Landing Highlight Flash */\n.cell.col-landing-flash,\n.col-btn.col-landing-flash {\n  animation: cellLandingFlash 240ms ease-out both;\n}\n\n@keyframes cellLandingFlash {\n  0% { background-color: #fff9e6; }\n  50% { background-color: #ffe6a0; }\n  100% { background-color: var(--surface); }\n}\n\n/* Landing Dust Particles Layer */\n.board-dust-layer {\n  position: absolute;\n  inset: 0;\n  pointer-events: none;\n  z-index: 10;\n  overflow: visible;\n}\n\n.dust-burst {\n  position: absolute;\n  width: 1px;\n  height: 1px;\n  pointer-events: none;\n}\n\n.dust-particle {\n  position: absolute;\n  border-radius: 50%;\n  background-color: var(--surface);\n  border: 1.5px solid var(--ink);\n  opacity: 1;\n  animation: dustPuff 280ms ease-out forwards;\n}\n\n@keyframes dustPuff {\n  0% {\n    transform: translate(0, 0) scale(1);\n    opacity: 0.95;\n  }\n  100% {\n    transform: translate(var(--dx), var(--dy)) scale(0.2);\n    opacity: 0;\n  }\n}\n\n.cell.win-cell {\n  z-index: 4;\n}\n\n.cell.win-cell .disc {\n  animation: winDiscPulse 1s ease-in-out infinite;\n  box-shadow: 0 0 0 3px var(--surface);\n}\n\n@keyframes winDiscPulse {\n  0%, 100% { transform: scale(1); }\n  50% { transform: scale(1.09); }\n}\n\n/* Quiet Keyboard Hint Numbers */\n.col-selectors {\n  display: grid;\n  grid-template-columns: repeat(var(--board-cols, 7), var(--cell-size));\n  gap: clamp(5px, 0.85vh, 9px);\n  padding: 0 clamp(10px, 1.4vh, 14px);\n}\n\n.col-btn {\n  min-height: 24px;\n  padding: 2px 0;\n  border: 1.5px solid rgba(17, 17, 17, 0.25);\n  border-radius: var(--radius-sm);\n  background-color: rgba(255, 255, 255, 0.65);\n  font-size: 0.72rem;\n  font-weight: 700;\n  color: var(--ink-muted);\n  transition: background-color 120ms ease, color 120ms ease;\n}\n\n.col-btn:hover:not(:disabled),\n.col-btn.col-hover {\n  background-color: var(--yellow);\n  color: var(--ink);\n  border-color: var(--ink);\n}\n\n.col-btn.col-best {\n  border-color: var(--purple);\n  color: var(--purple);\n}\n\n.col-btn:disabled {\n  opacity: 0.45;\n  cursor: default;\n}\n\n/* Board Legend */\n.board-legend {\n  display: flex;\n  flex-wrap: wrap;\n  justify-content: center;\n  gap: 8px;\n}\n\n.legend-item {\n  display: inline-flex;\n  align-items: center;\n  gap: 6px;\n  padding: 4px 10px;\n  background-color: var(--surface-soft);\n  border: 2px solid var(--ink);\n  border-radius: var(--radius-sm);\n  font-size: 0.78rem;\n  font-weight: 700;\n  transition: transform 180ms var(--ease-pop), background-color 180ms ease;\n}\n\n.legend-item.is-turn {\n  background-color: var(--yellow);\n  transform: scale(1.05);\n  box-shadow: var(--shadow-sm);\n}\n\n.legend-swatch {\n  width: 12px;\n  height: 12px;\n  border-radius: 50%;\n  border: 1.8px solid var(--ink);\n}\n\n.legend-swatch.p1 { background-color: var(--coral); }\n.legend-swatch.p2 { background-color: var(--yellow); }\n.legend-swatch.p3 { background-color: var(--green); }\n.legend-swatch.p4 { background-color: var(--purple); }\n.legend-swatch.obs { background-color: var(--stone); border-radius: 2px; }\n\n/* =========================================================\n   POST-GAME ANALYSIS PANEL\n   ========================================================= */\n\n.postgame-panel {\n  display: flex;\n  flex-direction: column;\n  gap: 12px;\n  min-height: 0;\n  height: 100%;\n  overflow: hidden;\n  padding: 14px;\n  background-color: var(--surface-soft);\n  border: var(--border);\n  border-radius: var(--radius-md);\n  animation: stagePop 260ms var(--ease-pop) both;\n}\n\n.postgame-banner {\n  flex: 0 0 auto;\n  display: flex;\n  align-items: center;\n  justify-content: space-between;\n  gap: 12px;\n  padding: 14px 16px;\n  background-color: var(--yellow);\n  border: var(--border);\n  border-radius: var(--radius-md);\n  box-shadow: var(--shadow-sm);\n}\n\n.postgame-banner.is-draw {\n  background-color: var(--surface);\n}\n\n.postgame-banner.is-bot-win {\n  background-color: #ffe3de;\n}\n\n.banner-title {\n  margin: 0;\n  font-size: 1.08rem;\n  font-weight: 800;\n}\n\n.banner-sub {\n  margin: 2px 0 0;\n  font-size: 0.82rem;\n  font-weight: 600;\n  color: var(--ink-muted);\n}\n\n.postgame-actions {\n  display: flex;\n  gap: 8px;\n  flex-shrink: 0;\n}\n\n.postgame-actions .btn-primary {\n  min-height: 44px;\n  padding: 8px 16px;\n  font-size: 0.9rem;\n}\n\n.stats-row {\n  flex: 0 0 auto;\n  display: grid;\n  grid-template-columns: repeat(3, minmax(0, 1fr));\n  gap: 10px;\n}\n\n.stat-box {\n  padding: 10px 12px;\n  background-color: var(--surface);\n  border: 2px solid var(--ink);\n  border-radius: var(--radius-sm);\n  display: flex;\n  flex-direction: column;\n  gap: 2px;\n}\n\n.stat-label {\n  font-size: 0.74rem;\n  font-weight: 700;\n  color: var(--ink-muted);\n}\n\n.stat-val {\n  font-size: 1.15rem;\n  font-weight: 800;\n}\n\n.stat-sub {\n  font-size: 0.72rem;\n  font-weight: 700;\n  color: var(--ink-muted);\n}\n\n.postgame-panel .tab-bar {\n  flex: 0 0 auto;\n  border: var(--border);\n  border-radius: var(--radius-sm);\n  overflow: hidden;\n  background-color: var(--surface);\n}\n\n.tab-panel {\n  flex: 1 1 auto;\n  min-height: 0;\n  overflow: hidden;\n  display: flex;\n  flex-direction: column;\n  gap: 10px;\n  height: 100%;\n}\n\n.table-scroll-box {\n  flex: 1 1 auto;\n  min-height: 160px;\n  overflow-y: auto;\n  border: 2px solid var(--ink);\n  border-radius: var(--radius-sm);\n  background-color: var(--surface);\n}\n\n.log-table {\n  width: 100%;\n  border-collapse: collapse;\n  font-size: 0.8rem;\n}\n\n.log-table th,\n.log-table td {\n  padding: 7px 10px;\n  text-align: left;\n  border-bottom: 1.5px solid #e5e7eb;\n}\n\n.log-table thead th {\n  position: sticky;\n  top: 0;\n  background-color: var(--surface-soft);\n  border-bottom: 2px solid var(--ink);\n  font-weight: 800;\n  z-index: 2;\n}\n\n.log-table tr.row-win {\n  background-color: #fff3d1;\n  font-weight: 800;\n}\n\n.legend-accordion {\n  flex: 0 0 auto;\n  border: 2px solid var(--ink);\n  border-radius: var(--radius-sm);\n  background-color: var(--surface);\n}\n\n.legend-acc-trigger {\n  list-style: none;\n  padding: 10px 12px;\n  min-height: 44px;\n  display: flex;\n  align-items: center;\n  justify-content: space-between;\n  font-size: 0.84rem;\n  font-weight: 800;\n  cursor: pointer;\n}\n\n.legend-acc-body {\n  padding: 12px;\n  border-top: 2px solid var(--ink);\n  max-height: 180px;\n  overflow-y: auto;\n}\n\n.legend-score-row {\n  display: flex;\n  align-items: center;\n  justify-content: space-between;\n  gap: 10px;\n  padding: 8px 10px;\n  background-color: var(--surface);\n  border: 1.5px solid var(--ink);\n  border-radius: var(--radius-sm);\n  font-size: 0.8rem;\n  margin-bottom: 6px;\n}\n\n.legend-score-row:last-child {\n  margin-bottom: 0;\n}\n\n.legend-pattern {\n  display: flex;\n  align-items: center;\n  gap: 8px;\n  font-weight: 700;\n}\n\n.legend-right {\n  display: flex;\n  align-items: center;\n  gap: 8px;\n  font-weight: 800;\n}\n\n.legend-live-count {\n  color: var(--ink-muted);\n  font-size: 0.78rem;\n}\n\n.legend-points {\n  color: var(--ink);\n  min-width: 52px;\n  text-align: right;\n}\n\n.mini-badge {\n  padding: 2px 8px;\n  background-color: var(--yellow);\n  border: 1.5px solid var(--ink);\n  border-radius: var(--radius-sm);\n  font-size: 0.76rem;\n}\n\n.eff-box {\n  padding: 14px;\n  background-color: var(--surface);\n  border: 2px solid var(--ink);\n  border-radius: var(--radius-sm);\n  display: flex;\n  flex-direction: column;\n  gap: 12px;\n  overflow-y: auto;\n}\n\n.eff-desc {\n  margin: 0;\n  font-size: 0.84rem;\n  color: var(--ink-muted);\n}\n\n.eff-controls {\n  display: flex;\n  align-items: center;\n  justify-content: space-between;\n  flex-wrap: wrap;\n  gap: 10px;\n}\n\n.field-inline {\n  display: inline-flex;\n  align-items: center;\n  gap: 8px;\n  font-size: 0.84rem;\n  font-weight: 700;\n}\n\n.compare-result {\n  padding: 12px;\n  background-color: var(--surface-soft);\n  border: 2px solid var(--ink);\n  border-radius: var(--radius-sm);\n  font-size: 0.84rem;\n}\n\n.compare-grid {\n  display: grid;\n  grid-template-columns: repeat(2, minmax(0, 1fr));\n  gap: 10px;\n  margin-bottom: 10px;\n}\n\n.compare-card {\n  padding: 10px;\n  background-color: var(--surface);\n  border: 2px solid var(--ink);\n  border-radius: var(--radius-sm);\n}\n\n.compare-card.ab {\n  background-color: #eafaf1;\n}\n\n.compare-summary {\n  padding: 8px 10px;\n  background-color: var(--yellow);\n  border: 2px solid var(--ink);\n  border-radius: var(--radius-sm);\n}\n\n/* =========================================================\n   COMPACT DESKTOP HEIGHT MEDIA QUERIES (1366x768 & 1280x720)\n   ========================================================= */\n\n@media (min-width: 900px) and (max-height: 780px) {\n  .topbar-inner {\n    min-height: 44px;\n    padding: 4px 20px;\n  }\n\n  .lobby-body {\n    gap: 24px;\n    padding: 14px 24px;\n  }\n\n  .lobby-col,\n  .lobby-controls-stack {\n    gap: 14px;\n  }\n\n  .choice-card {\n    min-height: 50px;\n    padding: 8px 12px;\n  }\n\n  .choice-card.diff-card {\n    min-height: 60px;\n    padding: 8px 6px;\n  }\n\n  .lobby-preview-stage {\n    min-height: 112px;\n  }\n\n  .arena-workspace {\n    padding: 10px 20px;\n  }\n\n  .board-column {\n    --cell-size: min(\n      calc((100dvh - 340px) / var(--board-rows, 6)),\n      calc((min(90vw, 660px) - 88px) / var(--board-cols, 7)),\n      58px\n    );\n  }\n}\n\n@media (min-width: 900px) and (max-height: 760px) {\n  .postgame-panel {\n    padding: 10px 12px;\n    gap: 8px;\n  }\n\n  .postgame-banner {\n    padding: 10px 12px;\n  }\n\n  .stat-box {\n    padding: 6px 10px;\n  }\n\n  .legend-acc-trigger {\n    min-height: 38px;\n    padding: 6px 10px;\n  }\n}\n\n@media (min-width: 900px) and (max-height: 700px) {\n  .view-shell {\n    padding: 8px 18px;\n  }\n\n  .card-header.lobby-header {\n    padding: 8px 16px;\n  }\n\n  .lobby-body {\n    padding: 12px 20px;\n    gap: 20px;\n  }\n\n  .lobby-group {\n    gap: 6px;\n  }\n\n  .choice-icon {\n    width: 32px;\n    height: 32px;\n  }\n\n  .arena-topbar {\n    padding: 6px 14px;\n  }\n\n  .arena-workspace {\n    padding: 8px 16px;\n  }\n\n  .board-column {\n    --cell-size: min(\n      calc((100dvh - 305px) / var(--board-rows, 6)),\n      calc((min(90vw, 620px) - 88px) / var(--board-cols, 7)),\n      52px\n    );\n  }\n\n  .arena-shell.has-analysis .board-column {\n    --cell-size: min(\n      calc((100dvh - 355px) / var(--board-rows, 6)),\n      calc(460px / var(--board-cols, 7)),\n      48px\n    );\n  }\n\n  .postgame-panel {\n    padding: 8px 10px;\n    gap: 6px;\n  }\n\n  .postgame-banner {\n    padding: 8px 10px;\n  }\n\n  .stat-box {\n    padding: 4px 8px;\n  }\n\n  .stat-val {\n    font-size: 1rem;\n  }\n\n  .tab-btn {\n    min-height: 38px;\n    padding: 6px 10px;\n  }\n\n  .table-scroll-box {\n    min-height: 120px;\n  }\n}\n\n/* =========================================================\n   MOBILE & TABLET RESPONSIVENESS (360px - 899px)\n   ========================================================= */\n\n@media (max-width: 899px) {\n  .topbar-inner {\n    padding: 6px 12px;\n    gap: 8px;\n  }\n\n  .sfx-corner-btn .sfx-label {\n    display: none;\n  }\n\n  .home-card {\n    grid-template-columns: 1fr;\n    text-align: center;\n    padding: 20px;\n  }\n\n  .home-content {\n    align-items: center;\n  }\n\n  .home-actions {\n    width: 100%;\n    align-items: center;\n  }\n\n  .home-title {\n    transform-origin: center;\n  }\n\n  .lobby-card {\n    overflow: visible;\n  }\n\n  .card-header.lobby-header {\n    border-radius: 5px 5px 0 0;\n  }\n\n  .lobby-body {\n    grid-template-columns: 1fr;\n    padding: 16px;\n    gap: 20px;\n  }\n\n  .sticky-start-bar {\n    position: sticky;\n    bottom: 0;\n    z-index: 30;\n    padding: 10px 0 6px;\n    background-color: var(--surface);\n  }\n\n  .arena-topbar {\n    grid-template-columns: auto 1fr auto;\n    padding: 8px 12px;\n    gap: 8px;\n  }\n\n  .arena-workspace {\n    padding: 14px 10px;\n  }\n\n  .postgame-panel {\n    max-height: none;\n  }\n\n  .postgame-banner {\n    flex-direction: column;\n    align-items: stretch;\n  }\n\n  .postgame-actions {\n    display: grid;\n    grid-template-columns: 1fr 1fr;\n  }\n}\n\n@media (max-width: 619px) {\n  .choice-grid.cols-4 {\n    grid-template-columns: repeat(2, minmax(0, 1fr));\n  }\n\n  .choice-grid.cols-5-first {\n    grid-template-columns: repeat(2, minmax(0, 1fr));\n  }\n\n  .choice-grid.cols-5-first .first-random {\n    grid-column: 1 / -1;\n  }\n\n  .seats-grid {\n    grid-template-columns: repeat(2, minmax(0, 1fr));\n  }\n\n  .custom-select-row {\n    grid-template-columns: 1fr;\n  }\n\n  .stats-row {\n    grid-template-columns: 1fr;\n  }\n\n  .choice-grid.cols-3-always {\n    grid-template-columns: 1fr;\n  }\n}\n\n@media (max-width: 480px) {\n  .topbar-inner {\n    padding: 4px 8px;\n    gap: 6px;\n  }\n\n  .brand-lockup {\n    gap: 6px;\n  }\n\n  .brand-title {\n    font-size: 0.78rem;\n    white-space: nowrap;\n  }\n\n  .topbar-actions {\n    gap: 5px;\n  }\n\n  .top-link-btn {\n    padding: 4px 6px;\n    font-size: 0.72rem;\n    white-space: nowrap;\n  }\n\n  .sfx-corner-btn {\n    min-width: 44px;\n    min-height: 44px;\n    padding: 4px 8px;\n  }\n\n  .view-shell {\n    padding: 10px;\n  }\n\n  .card-header.lobby-header {\n    padding: 8px 10px;\n    gap: 8px;\n  }\n\n  .header-title {\n    font-size: 0.92rem;\n  }\n\n  .choice-grid.cols-2 {\n    grid-template-columns: 1fr;\n  }\n\n  .arena-topbar {\n    grid-template-columns: 1fr auto;\n    gap: 8px;\n    padding: 8px 10px;\n  }\n\n  .arena-top-left {\n    order: 1;\n  }\n\n  .arena-top-right {\n    order: 2;\n  }\n\n  .arena-top-center {\n    order: 3;\n    grid-column: 1 / -1;\n    width: 100%;\n  }\n\n  .turn-pill {\n    width: 100%;\n    justify-content: center;\n    min-height: 38px;\n    font-size: 0.84rem;\n  }\n\n  .board-column {\n    --cell-size: min(\n      calc((100vw - 104px) / var(--board-cols, 7)),\n      calc((100dvh - 300px) / var(--board-rows, 6)),\n      42px\n    );\n  }\n\n  .board-grid,\n  .hover-preview-row,\n  .col-selectors {\n    gap: 3px;\n    padding-inline: 8px;\n  }\n\n  .board-grid {\n    padding-block: 8px;\n  }\n\n  .btn-nav {\n    padding: 8px 10px;\n    font-size: 0.82rem;\n  }\n}\n\n/* Volume Slider */\n.volume-slider {\n  accent-color: var(--purple);\n  cursor: pointer;\n  height: 6px;\n  border-radius: 3px;\n  max-width: 110px;\n}\n\n/* PWA Install Button & Update Toast */\n.pwa-install-btn {\n  background-color: var(--green);\n  color: #ffffff;\n  display: inline-flex;\n  align-items: center;\n  gap: 6px;\n}\n\n.pwa-install-btn:hover {\n  background-color: var(--green-aa);\n}\n\n.pwa-toast {\n  position: fixed;\n  bottom: 20px;\n  left: 50%;\n  transform: translateX(-50%);\n  z-index: 150;\n  display: flex;\n  align-items: center;\n  gap: 12px;\n  padding: 10px 16px;\n  background-color: var(--yellow);\n  color: var(--ink);\n  border: var(--border-thick);\n  border-radius: var(--radius-md);\n  box-shadow: var(--shadow-md);\n  font-weight: 700;\n  font-size: 0.9rem;\n  animation: toastSlideUp 240ms var(--ease-pop) both;\n}\n\n@keyframes toastSlideUp {\n  from { opacity: 0; transform: translate(-50%, 14px); }\n  to { opacity: 1; transform: translate(-50%, 0); }\n}\n\n.pwa-reload-btn {\n  padding: 6px 12px;\n  background-color: var(--purple);\n  color: #ffffff;\n  border: 2px solid var(--ink);\n  border-radius: var(--radius-sm);\n  font-weight: 800;\n  font-size: 0.82rem;\n  cursor: pointer;\n  box-shadow: var(--shadow-sm);\n  transition: transform 120ms ease;\n}\n\n.pwa-reload-btn:hover {\n  background-color: var(--purple-dark);\n  transform: translateY(-1px);\n}\n\n/* PWA Floating Install Banner */\n.pwa-banner {\n  position: fixed;\n  bottom: 24px;\n  right: 24px;\n  max-width: 440px;\n  z-index: 140;\n  display: flex;\n  align-items: center;\n  justify-content: space-between;\n  gap: 16px;\n  padding: 12px 16px;\n  background-color: var(--surface);\n  border: var(--border-thick);\n  border-radius: var(--radius-md);\n  box-shadow: var(--shadow-lg);\n  animation: pwaBannerIn 280ms var(--ease-pop) both;\n}\n\n@keyframes pwaBannerIn {\n  from { opacity: 0; transform: translateY(20px) scale(0.96); }\n  to { opacity: 1; transform: translateY(0) scale(1); }\n}\n\n.pwa-banner-content {\n  display: flex;\n  align-items: center;\n  gap: 12px;\n}\n\n.pwa-banner-icon {\n  font-size: 1.8rem;\n  line-height: 1;\n  flex-shrink: 0;\n}\n\n.pwa-banner-text {\n  display: flex;\n  flex-direction: column;\n  gap: 2px;\n}\n\n.pwa-banner-text strong {\n  font-size: 0.92rem;\n  font-weight: 800;\n  color: var(--ink);\n}\n\n.pwa-banner-text p {\n  margin: 0;\n  font-size: 0.78rem;\n  color: var(--ink-muted);\n  line-height: 1.35;\n}\n\n.pwa-banner-actions {\n  display: flex;\n  align-items: center;\n  gap: 8px;\n  flex-shrink: 0;\n}\n\n.btn-pwa-primary {\n  padding: 8px 14px;\n  background-color: var(--green);\n  color: #ffffff;\n  border: 2px solid var(--ink);\n  border-radius: var(--radius-sm);\n  font-size: 0.82rem;\n  font-weight: 800;\n  cursor: pointer;\n  box-shadow: var(--shadow-sm);\n  white-space: nowrap;\n  transition: transform 120ms ease, background-color 120ms ease;\n}\n\n.btn-pwa-primary:hover {\n  background-color: var(--green-aa);\n  transform: translateY(-1px);\n}\n\n.btn-pwa-dismiss {\n  width: 32px;\n  height: 32px;\n  display: flex;\n  align-items: center;\n  justify-content: center;\n  background-color: var(--surface-soft);\n  border: 2px solid var(--ink);\n  border-radius: var(--radius-sm);\n  font-size: 1.1rem;\n  font-weight: 800;\n  color: var(--ink);\n  cursor: pointer;\n  transition: background-color 120ms ease;\n}\n\n.btn-pwa-dismiss:hover {\n  background-color: #ffd8d3;\n}\n\n/* PWA Guide Modal Content */\n.pwa-guide-body {\n  font-size: 0.9rem;\n  display: flex;\n  flex-direction: column;\n  gap: 16px;\n}\n\n.pwa-guide-card {\n  padding: 12px 14px;\n  background-color: #fff9e6;\n  border: 2px solid var(--ink);\n  border-radius: var(--radius-sm);\n}\n\n.pwa-guide-card h4,\n.pwa-guide-steps h4 {\n  margin: 0 0 6px;\n  font-size: 0.92rem;\n  font-weight: 800;\n}\n\n.pwa-guide-card p {\n  margin: 0;\n  font-size: 0.84rem;\n  line-height: 1.45;\n  color: var(--ink-muted);\n}\n\n.pwa-guide-steps ol {\n  margin: 0;\n  padding-left: 20px;\n  display: flex;\n  flex-direction: column;\n  gap: 8px;\n  font-size: 0.84rem;\n}\n\n.pwa-guide-steps pre {\n  margin: 4px 0;\n  padding: 4px 8px;\n  background-color: var(--ink);\n  color: #a7f3d0;\n  border-radius: var(--radius-sm);\n  font-family: monospace;\n  font-size: 0.8rem;\n  display: inline-block;\n}\n\n@media (max-width: 619px) {\n  .pwa-banner {\n    left: 12px;\n    right: 12px;\n    bottom: 12px;\n    max-width: none;\n    flex-direction: column;\n    align-items: stretch;\n    gap: 10px;\n  }\n  .pwa-banner-actions {\n    justify-content: flex-end;\n  }\n}\n\n/* =========================================================\n   REDUCED MOTION ACCESSIBILITY\n   ========================================================= */\n\n@media (prefers-reduced-motion: reduce) {\n  *,\n  *::before,\n  *::after {\n    animation-duration: 0.01ms !important;\n    animation-iteration-count: 1 !important;\n    transition-duration: 0.01ms !important;\n  }\n\n  .board-grid.board-impact,\n  .board-grid.board-draw-shake,\n  .dust-particle {\n    animation: none !important;\n    display: none !important;\n  }\n}\n\n\nbody {\n  min-height: 760px !important;\n  padding-bottom: 24px !important;\n}\n.modal-backdrop {\n  align-items: flex-start;\n  padding-top: 32px;\n}\n\n</style>\n<!-- VIEW 1: LANDING -->\n  <div id=\"appStage\" class=\"app-stage\">\n\n    <main id=\"homeView\" class=\"view-shell home-shell\">\n      <section class=\"surface-card home-card animate-pop-in\">\n        <div class=\"home-showcase\" aria-hidden=\"true\">\n          <div class=\"showcase-board\">\n            <div class=\"sc-slot\"><span class=\"sc-disc is-empty\"></span></div>\n            <div class=\"sc-slot\"><span class=\"sc-disc p2 loop-drop d1\"></span></div>\n            <div class=\"sc-slot\"><span class=\"sc-disc is-empty\"></span></div>\n            <div class=\"sc-slot\"><span class=\"sc-disc is-empty\"></span></div>\n            <div class=\"sc-slot\"><span class=\"sc-disc p1 loop-drop d2\"></span></div>\n\n            <div class=\"sc-slot\"><span class=\"sc-disc is-empty\"></span></div>\n            <div class=\"sc-slot\"><span class=\"sc-disc p1\"></span></div>\n            <div class=\"sc-slot\"><span class=\"sc-disc p2 loop-drop d3\"></span></div>\n            <div class=\"sc-slot\"><span class=\"sc-disc p3\"></span></div>\n            <div class=\"sc-slot\"><span class=\"sc-disc p1\"></span></div>\n\n            <div class=\"sc-slot\"><span class=\"sc-disc p2\"></span></div>\n            <div class=\"sc-slot\"><span class=\"sc-disc p1\"></span></div>\n            <div class=\"sc-slot\"><span class=\"sc-disc p1\"></span></div>\n            <div class=\"sc-slot\"><span class=\"sc-disc p4\"></span></div>\n            <div class=\"sc-slot\"><span class=\"sc-disc p2\"></span></div>\n\n            <div class=\"sc-slot\"><span class=\"sc-disc p1 is-win-demo\"></span></div>\n            <div class=\"sc-slot\"><span class=\"sc-disc p1 is-win-demo\"></span></div>\n            <div class=\"sc-slot\"><span class=\"sc-disc p1 is-win-demo\"></span></div>\n            <div class=\"sc-slot\"><span class=\"sc-disc p1 is-win-demo\"></span></div>\n            <div class=\"sc-slot\"><span class=\"sc-disc p3\"></span></div>\n          </div>\n        </div>\n\n        <div class=\"home-content\">\n          <h1 class=\"home-title\">Connect-Four AI</h1>\n          <p class=\"home-subtitle\">\n            Hubungkan 4 bidak secara beruntun &mdash; lawan bot AI, teman sekamar, atau 4 pemain sekaligus.\n          </p>\n          <div class=\"home-actions\">\n            <button type=\"button\" id=\"goSetupBtn\" class=\"btn-primary hero-cta\">\n              <svg viewBox=\"0 0 24 24\" width=\"20\" height=\"20\" fill=\"currentColor\" aria-hidden=\"true\">\n                <polygon points=\"6 4 20 12 6 20 6 4\"></polygon>\n              </svg>\n              <span>Main sekarang</span>\n            </button>\n            <div class=\"home-links\">\n              <button type=\"button\" id=\"openRulesBtn\" class=\"text-link-btn\">Aturan</button>\n              <span class=\"link-sep\" aria-hidden=\"true\">&middot;</span>\n              <button type=\"button\" id=\"openInfoBtn\" class=\"text-link-btn\">Cara kerja AI</button>\n            </div>\n          </div>\n        </div>\n      </section>\n    </main>\n\n    <!-- VIEW 2: LOBBY SETUP (2-COLUMN GAME LOBBY) -->\n    <section id=\"setupView\" class=\"view-shell setup-shell\" style=\"display: none;\">\n      <div class=\"surface-card lobby-card animate-pop-in\">\n        <header class=\"card-header lobby-header\">\n          <button type=\"button\" id=\"backHomeBtn\" class=\"btn-header\" aria-label=\"Kembali ke menu utama\">&larr; Kembali</button>\n          <h2 class=\"header-title\">Lobby Pertandingan</h2>\n          <span id=\"setupPreviewBadge\" class=\"status-chip\">2P AI &bull; 6&times;7</span>\n        </header>\n\n        <div class=\"lobby-body\">\n          <!-- LEFT COLUMN: MODE + LIVE PREVIEW -->\n          <div class=\"lobby-col lobby-col-left\">\n            <div class=\"lobby-group\">\n              <span class=\"group-label\">Mode permainan</span>\n              <div id=\"modeChoiceGrid\" class=\"choice-grid cols-2\" role=\"radiogroup\" aria-label=\"Mode permainan\">\n                <button type=\"button\" class=\"choice-card active\" data-mode=\"hvb\">\n                  <span class=\"choice-icon coral\" aria-hidden=\"true\">\n                    <svg viewBox=\"0 0 24 24\" width=\"22\" height=\"22\" fill=\"none\" stroke=\"currentColor\" stroke-width=\"2.2\" stroke-linecap=\"round\" stroke-linejoin=\"round\">\n                      <path d=\"M20 21v-2a4 4 0 0 0-4-4H8a4 4 0 0 0-4 4v2\"></path>\n                      <circle cx=\"12\" cy=\"7\" r=\"4\"></circle>\n                    </svg>\n                  </span>\n                  <span class=\"choice-title\">Manusia vs AI</span>\n                  <span class=\"choice-check\" aria-hidden=\"true\"></span>\n                </button>\n\n                <button type=\"button\" class=\"choice-card\" data-mode=\"pvp\">\n                  <span class=\"choice-icon yellow\" aria-hidden=\"true\">\n                    <svg viewBox=\"0 0 24 24\" width=\"22\" height=\"22\" fill=\"none\" stroke=\"currentColor\" stroke-width=\"2.2\" stroke-linecap=\"round\" stroke-linejoin=\"round\">\n                      <path d=\"M17 21v-2a4 4 0 0 0-4-4H5a4 4 0 0 0-4 4v2\"></path>\n                      <circle cx=\"9\" cy=\"7\" r=\"4\"></circle>\n                      <path d=\"M23 21v-2a4 4 0 0 0-3-3.87\"></path>\n                      <path d=\"M16 3.13a4 4 0 0 1 0 7.75\"></path>\n                    </svg>\n                  </span>\n                  <span class=\"choice-title\">Pemain vs Pemain</span>\n                  <span class=\"choice-check\" aria-hidden=\"true\"></span>\n                </button>\n\n                <button type=\"button\" class=\"choice-card\" data-mode=\"4p\">\n                  <span class=\"choice-icon green\" aria-hidden=\"true\">\n                    <svg viewBox=\"0 0 24 24\" width=\"22\" height=\"22\" fill=\"none\" stroke=\"currentColor\" stroke-width=\"2.2\">\n                      <circle cx=\"8\" cy=\"8\" r=\"3\"></circle>\n                      <circle cx=\"16\" cy=\"8\" r=\"3\"></circle>\n                      <circle cx=\"8\" cy=\"16\" r=\"3\"></circle>\n                      <circle cx=\"16\" cy=\"16\" r=\"3\"></circle>\n                    </svg>\n                  </span>\n                  <span class=\"choice-title\">4-Player Arena</span>\n                  <span class=\"choice-check\" aria-hidden=\"true\"></span>\n                </button>\n\n                <button type=\"button\" class=\"choice-card\" data-mode=\"bvb\">\n                  <span class=\"choice-icon purple\" aria-hidden=\"true\">\n                    <svg viewBox=\"0 0 24 24\" width=\"22\" height=\"22\" fill=\"none\" stroke=\"currentColor\" stroke-width=\"2.2\" stroke-linecap=\"round\" stroke-linejoin=\"round\">\n                      <rect x=\"4\" y=\"7\" width=\"16\" height=\"12\" rx=\"2\"></rect>\n                      <circle cx=\"9\" cy=\"13\" r=\"1.5\" fill=\"currentColor\"></circle>\n                      <circle cx=\"15\" cy=\"13\" r=\"1.5\" fill=\"currentColor\"></circle>\n                      <line x1=\"12\" y1=\"3\" x2=\"12\" y2=\"7\"></line>\n                      <circle cx=\"12\" cy=\"3\" r=\"1\" fill=\"currentColor\"></circle>\n                    </svg>\n                  </span>\n                  <span class=\"choice-title\">Bot vs Bot</span>\n                  <span class=\"choice-check\" aria-hidden=\"true\"></span>\n                </button>\n              </div>\n            </div>\n\n            <div class=\"lobby-group preview-group\">\n              <div class=\"preview-meta-row\">\n                <span class=\"group-label\">Pratinjau papan</span>\n                <span id=\"lobbyPreviewLabel\" class=\"preview-meta-pill\">Klasik &bull; 6&times;7</span>\n              </div>\n              <div class=\"lobby-preview-box\">\n                <div id=\"lobbyPreviewChips\" class=\"preview-player-chips\" aria-label=\"Warna pemain aktif\"></div>\n                <div class=\"lobby-preview-stage\">\n                  <div id=\"lobbyPreviewBoard\" class=\"lobby-preview-board\" aria-hidden=\"true\"></div>\n                </div>\n              </div>\n            </div>\n          </div>\n\n          <!-- RIGHT COLUMN: BOARD + DIFFICULTY + ACTIONS -->\n          <div class=\"lobby-col lobby-col-right\">\n            <div class=\"lobby-controls-stack\">\n              <div class=\"lobby-group\">\n                <span class=\"group-label\">Papan &amp; ukuran</span>\n                <div class=\"board-config-stack\">\n                  <div id=\"boardModeGrid\" class=\"choice-grid cols-2\" role=\"radiogroup\" aria-label=\"Tipe papan\">\n                    <button type=\"button\" class=\"choice-card active\" data-bmode=\"classic\">\n                      <span class=\"choice-icon blue\" aria-hidden=\"true\">\n                        <svg viewBox=\"0 0 24 24\" width=\"20\" height=\"20\" fill=\"none\" stroke=\"currentColor\" stroke-width=\"2.2\">\n                          <rect x=\"3\" y=\"3\" width=\"18\" height=\"18\" rx=\"2\"></rect>\n                          <line x1=\"3\" y1=\"9\" x2=\"21\" y2=\"9\"></line>\n                          <line x1=\"3\" y1=\"15\" x2=\"21\" y2=\"15\"></line>\n                          <line x1=\"9\" y1=\"3\" x2=\"9\" y2=\"21\"></line>\n                          <line x1=\"15\" y1=\"3\" x2=\"15\" y2=\"21\"></line>\n                        </svg>\n                      </span>\n                      <span class=\"choice-title\">Klasik</span>\n                      <span class=\"choice-check\" aria-hidden=\"true\"></span>\n                    </button>\n\n                    <button type=\"button\" class=\"choice-card\" data-bmode=\"obstacle\">\n                      <span class=\"choice-icon dark\" aria-hidden=\"true\">\n                        <svg viewBox=\"0 0 24 24\" width=\"20\" height=\"20\" fill=\"none\" stroke=\"currentColor\" stroke-width=\"2.2\">\n                          <rect x=\"4\" y=\"4\" width=\"16\" height=\"16\" rx=\"2\"></rect>\n                          <line x1=\"8\" y1=\"8\" x2=\"16\" y2=\"16\"></line>\n                          <line x1=\"16\" y1=\"8\" x2=\"8\" y2=\"16\"></line>\n                        </svg>\n                      </span>\n                      <span class=\"choice-title\">Obstacle</span>\n                      <span class=\"choice-check\" aria-hidden=\"true\"></span>\n                    </button>\n                  </div>\n\n                  <div id=\"boardSizeGrid\" class=\"segmented-bar cols-4\" role=\"radiogroup\" aria-label=\"Ukuran grid\">\n                    <button type=\"button\" class=\"size-btn active\" data-size=\"6x7\">6&times;7</button>\n                    <button type=\"button\" class=\"size-btn\" data-size=\"7x8\">7&times;8</button>\n                    <button type=\"button\" class=\"size-btn\" data-size=\"7x9\">7&times;9</button>\n                    <button type=\"button\" class=\"size-btn\" data-size=\"8x10\">8&times;10</button>\n                  </div>\n                </div>\n              </div>\n\n              <div id=\"diffSection\" class=\"lobby-group\">\n                <span class=\"group-label\">Kesulitan AI</span>\n                <div id=\"diffChoiceGrid\" class=\"choice-grid cols-4 diff-grid\" role=\"radiogroup\" aria-label=\"Tingkat kesulitan AI\">\n                  <button type=\"button\" class=\"choice-card diff-card\" data-diff=\"Easy\">\n                    <span class=\"signal-bars lvl-1\" aria-hidden=\"true\">\n                      <span class=\"s-bar active\"></span><span class=\"s-bar\"></span><span class=\"s-bar\"></span>\n                    </span>\n                    <span class=\"choice-title\">Easy</span>\n                    <span class=\"choice-check\" aria-hidden=\"true\"></span>\n                  </button>\n\n                  <button type=\"button\" class=\"choice-card diff-card active\" data-diff=\"Medium\">\n                    <span class=\"signal-bars lvl-2\" aria-hidden=\"true\">\n                      <span class=\"s-bar active\"></span><span class=\"s-bar active\"></span><span class=\"s-bar\"></span>\n                    </span>\n                    <span class=\"choice-title\">Medium</span>\n                    <span class=\"choice-check\" aria-hidden=\"true\"></span>\n                  </button>\n\n                  <button type=\"button\" class=\"choice-card diff-card\" data-diff=\"Hard\">\n                    <span class=\"signal-bars lvl-3\" aria-hidden=\"true\">\n                      <span class=\"s-bar active\"></span><span class=\"s-bar active\"></span><span class=\"s-bar active\"></span>\n                    </span>\n                    <span class=\"choice-title\">Hard</span>\n                    <span class=\"choice-check\" aria-hidden=\"true\"></span>\n                  </button>\n\n                  <button type=\"button\" class=\"choice-card diff-card\" data-diff=\"Custom\">\n                    <span class=\"choice-icon compact\" aria-hidden=\"true\">\n                      <svg viewBox=\"0 0 24 24\" width=\"18\" height=\"18\" fill=\"none\" stroke=\"currentColor\" stroke-width=\"2.2\" stroke-linecap=\"round\">\n                        <line x1=\"4\" y1=\"8\" x2=\"20\" y2=\"8\"></line>\n                        <circle cx=\"9\" cy=\"8\" r=\"2\" fill=\"currentColor\"></circle>\n                        <line x1=\"4\" y1=\"16\" x2=\"20\" y2=\"16\"></line>\n                        <circle cx=\"15\" cy=\"16\" r=\"2\" fill=\"currentColor\"></circle>\n                      </svg>\n                    </span>\n                    <span class=\"choice-title\">Custom</span>\n                    <span class=\"choice-check\" aria-hidden=\"true\"></span>\n                  </button>\n                </div>\n              </div>\n            </div>\n\n            <div class=\"lobby-actions-footer\">\n              <button type=\"button\" id=\"openAdvancedBtn\" class=\"advanced-trigger-btn\">\n                <span class=\"adv-btn-left\">\n                  <svg viewBox=\"0 0 24 24\" width=\"18\" height=\"18\" fill=\"none\" stroke=\"currentColor\" stroke-width=\"2.2\" stroke-linecap=\"round\" stroke-linejoin=\"round\" aria-hidden=\"true\">\n                    <path d=\"M12 20h9\"></path>\n                    <path d=\"M16.5 3.5a2.1 2.1 0 0 1 3 3L7 19l-4 1 1-4Z\"></path>\n                  </svg>\n                  <span>Opsi lanjutan</span>\n                </span>\n                <span id=\"advancedSummaryBadge\" class=\"adv-summary-pill\">Jalan: Manusia</span>\n              </button>\n\n              <div class=\"sticky-start-bar\">\n                <button type=\"button\" id=\"startGameBtn\" class=\"btn-primary start-cta\">\n                  <span>Mulai pertandingan</span>\n                  <svg viewBox=\"0 0 24 24\" width=\"20\" height=\"20\" fill=\"currentColor\" aria-hidden=\"true\">\n                    <polygon points=\"6 4 20 12 6 20 6 4\"></polygon>\n                  </svg>\n                </button>\n              </div>\n            </div>\n          </div>\n        </div>\n      </div>\n    </section>\n\n    <!-- VIEW 3: GAME ARENA + POST-GAME -->\n    <section id=\"gameView\" class=\"view-shell arena-view\" style=\"display: none;\">\n      <canvas id=\"confettiCanvas\" class=\"confetti-canvas\" aria-hidden=\"true\"></canvas>\n\n      <div id=\"arenaShell\" class=\"surface-card arena-shell animate-pop-in\">\n        <header class=\"arena-topbar\">\n          <div class=\"arena-top-left\">\n            <button type=\"button\" id=\"backSetupBtn\" class=\"btn-nav\" aria-label=\"Kembali ke setup\">&larr; Kembali</button>\n          </div>\n\n          <div class=\"arena-top-center\">\n            <div id=\"statusBanner\" class=\"turn-pill\" role=\"status\" aria-live=\"polite\">\n              <span id=\"statusDot\" class=\"status-dot\"></span>\n              <span id=\"statusText\" class=\"status-label\">Giliran Anda</span>\n              <span id=\"thinkingDots\" class=\"thinking-dots\" aria-hidden=\"true\">\n                <span></span><span></span><span></span>\n              </span>\n              <span id=\"turnCounterBadge\" class=\"turn-count-chip\">#0</span>\n              <span id=\"lastMovePill\" class=\"last-move-chip\" style=\"display: none;\">-</span>\n            </div>\n          </div>\n\n          <div class=\"arena-top-right\">\n            <button type=\"button\" id=\"undoBtn\" class=\"btn-nav\">Undo</button>\n            <details id=\"arenaOverflowMenu\" class=\"overflow-menu\">\n              <summary class=\"overflow-trigger\" aria-label=\"Menu kontrol permainan\" title=\"Opsi permainan\">&#8943;</summary>\n              <div class=\"overflow-popover\">\n                <div class=\"popover-meta\">\n                  <span id=\"arenaPill\" class=\"popover-mode-tag\">ARENA 2P AI</span>\n                  <span id=\"activeConfigBadge\" class=\"popover-cfg-tag\">2P AI &bull; Medium</span>\n                </div>\n\n                <label class=\"popover-row\" for=\"botSpeedSelect\">\n                  <span>Kecepatan bot</span>\n                  <select id=\"botSpeedSelect\" class=\"field-select compact\">\n                    <option value=\"fast\">Cepat</option>\n                    <option value=\"normal\" selected>Normal</option>\n                    <option value=\"slow\">Lambat</option>\n                  </select>\n                </label>\n\n                <label class=\"popover-row\" for=\"sfxVolumeSlider\">\n                  <span>Volume efek</span>\n                  <input type=\"range\" id=\"sfxVolumeSlider\" min=\"0\" max=\"1\" step=\"0.05\" value=\"0.6\" class=\"volume-slider compact\" aria-label=\"Volume efek suara\" />\n                </label>\n\n                <div id=\"bvbBar\" class=\"popover-bvb\" style=\"display: none;\">\n                  <button type=\"button\" id=\"botStepBtn\" class=\"btn-soft\">1 Langkah</button>\n                  <button type=\"button\" id=\"autoPlayBtn\" class=\"btn-green\">Auto-Play</button>\n                </div>\n\n                <button type=\"button\" id=\"resetBtn\" class=\"btn-popover-item\">Reset papan</button>\n                <button type=\"button\" id=\"menuInstallPwaBtn\" class=\"btn-popover-item\">Instal aplikasi PWA</button>\n                <button type=\"button\" id=\"menuTestPwaToastBtn\" class=\"btn-popover-item subtle\">Tes notifikasi update</button>\n                <button type=\"button\" id=\"backMenuBtn\" class=\"btn-popover-item subtle\">Ke menu utama</button>\n              </div>\n            </details>\n          </div>\n        </header>\n\n        <div class=\"arena-workspace\">\n          <div class=\"board-column\">\n            <div class=\"board-stage\">\n              <div id=\"hoverPreviewRow\" class=\"hover-preview-row\" aria-hidden=\"true\"></div>\n              <div class=\"board-frame\">\n                <div id=\"boardGrid\" class=\"board-grid\" role=\"grid\" aria-label=\"Papan Connect-Four\"></div>\n                <div id=\"boardDustLayer\" class=\"board-dust-layer\" aria-hidden=\"true\"></div>\n                <svg id=\"winLineSvg\" class=\"win-line-svg\" aria-hidden=\"true\"></svg>\n              </div>\n              <div id=\"colSelectors\" class=\"col-selectors\" aria-label=\"Hint kolom keyboard\"></div>\n            </div>\n            <div id=\"boardLegend\" class=\"board-legend\"></div>\n          </div>\n\n          <section id=\"postGamePanel\" class=\"postgame-panel\" style=\"display: none;\" aria-label=\"Hasil pertandingan dan analisis\">\n            <div id=\"postGameBanner\" class=\"postgame-banner\">\n              <div class=\"banner-text\">\n                <h3 id=\"postGameTitle\" class=\"banner-title\">Hasil Pertandingan</h3>\n                <p id=\"postGameSub\" class=\"banner-sub\">-</p>\n              </div>\n              <div class=\"postgame-actions\">\n                <button type=\"button\" id=\"playAgainBtn\" class=\"btn-primary\">Main lagi</button>\n                <button type=\"button\" id=\"postMenuBtn\" class=\"btn-nav\">Menu</button>\n              </div>\n            </div>\n\n            <div class=\"stats-row\">\n              <div class=\"stat-box\">\n                <span class=\"stat-label\">Total langkah</span>\n                <strong id=\"statTotalMoves\" class=\"stat-val\">0</strong>\n              </div>\n              <div class=\"stat-box\">\n                <span class=\"stat-label\">Langkah bot</span>\n                <strong id=\"statBotMoves\" class=\"stat-val\">0</strong>\n              </div>\n              <div class=\"stat-box\">\n                <span class=\"stat-label\">Rata-rata node</span>\n                <strong id=\"statAvgNodes\" class=\"stat-val\">0</strong>\n                <span id=\"statTotalTime\" class=\"stat-sub\">0 ms</span>\n              </div>\n            </div>\n\n            <div class=\"tab-bar\" role=\"tablist\" aria-label=\"Analisis pertandingan\">\n              <button type=\"button\" class=\"tab-btn active\" role=\"tab\" aria-selected=\"true\" data-analysis-tab=\"steps\">Langkah</button>\n              <button type=\"button\" class=\"tab-btn\" role=\"tab\" aria-selected=\"false\" data-analysis-tab=\"efficiency\">Efisiensi</button>\n            </div>\n\n            <div id=\"analysisTabSteps\" class=\"tab-panel active\">\n              <div class=\"table-scroll-box\">\n                <table class=\"log-table\">\n                  <thead>\n                    <tr>\n                      <th>Step</th>\n                      <th>Pemain</th>\n                      <th>Kolom</th>\n                      <th>Skor</th>\n                      <th>Node</th>\n                      <th>Waktu</th>\n                    </tr>\n                  </thead>\n                  <tbody id=\"postGameTableBody\"></tbody>\n                </table>\n              </div>\n\n              <details class=\"legend-accordion\">\n                <summary class=\"legend-acc-trigger\">\n                  <span>Bobot evaluasi heuristik</span>\n                  <span id=\"legendTotalBadge\" class=\"mini-badge\">Total: 0</span>\n                </summary>\n                <div class=\"legend-acc-body\">\n                  <div id=\"heuristicGrid\" class=\"legend-rows\"></div>\n                </div>\n              </details>\n            </div>\n\n            <div id=\"analysisTabEfficiency\" class=\"tab-panel\" style=\"display: none;\">\n              <div class=\"eff-box\">\n                <p id=\"compareContextText\" class=\"eff-desc\">\n                  Bandingkan jumlah node &amp; waktu Minimax polos vs Alpha-Beta Pruning pada posisi terakhir.\n                </p>\n                <div class=\"eff-controls\">\n                  <label class=\"field-inline\" for=\"compareDepthSelect\">\n                    <span>Kedalaman uji</span>\n                    <select id=\"compareDepthSelect\" class=\"field-select\">\n                      <option value=\"2\">Depth 2</option>\n                      <option value=\"3\" selected>Depth 3</option>\n                      <option value=\"4\">Depth 4</option>\n                      <option value=\"5\">Depth 5</option>\n                    </select>\n                  </label>\n                  <button type=\"button\" id=\"compareBtn\" class=\"btn-primary compact\">Jalankan uji</button>\n                </div>\n                <div id=\"compareResult\" class=\"compare-result\">\n                  Pilih kedalaman lalu tekan tombol untuk melihat efisiensi pemangkasan.\n                </div>\n              </div>\n            </div>\n          </section>\n        </div>\n      </div>\n    </section>\n\n    <!-- OVERLAY DRAWER: OPSI LANJUTAN SETUP -->\n    <div id=\"advancedSetupDetails\" class=\"drawer-backdrop\" role=\"dialog\" aria-modal=\"true\" aria-labelledby=\"drawerHeading\" style=\"display: none;\">\n      <div class=\"drawer-panel\">\n        <div class=\"drawer-header\">\n          <h3 id=\"drawerHeading\" class=\"drawer-title\">Opsi lanjutan</h3>\n          <button type=\"button\" id=\"closeAdvancedBtn\" class=\"modal-close-btn\" aria-label=\"Tutup opsi lanjutan\">&times;</button>\n        </div>\n\n        <div class=\"drawer-body\">\n          <div id=\"fourPlayerBox\" class=\"drawer-section\" style=\"display: none;\">\n            <div class=\"drawer-section-head\">\n              <span class=\"group-label\">Susunan pemain (4P)</span>\n              <div id=\"fourPresetGrid\" class=\"segmented-bar compact\" role=\"group\" aria-label=\"Preset susunan 4 pemain\">\n                <button type=\"button\" class=\"preset-chip\" data-preset=\"4human\">4 Manusia</button>\n                <button type=\"button\" class=\"preset-chip active\" data-preset=\"1h3b\">1P + 3 Bot</button>\n                <button type=\"button\" class=\"preset-chip\" data-preset=\"2h2b\">2P + 2 Bot</button>\n                <button type=\"button\" class=\"preset-chip\" data-preset=\"4bot\">4 Bot</button>\n              </div>\n            </div>\n\n            <div id=\"fourSeatsGrid\" class=\"seats-grid\">\n              <button type=\"button\" class=\"seat-btn\" data-seat=\"1\">\n                <span class=\"disc-badge p1\"></span>\n                <span class=\"seat-name\">Pemain 1</span>\n                <span class=\"seat-role-tag\" data-role-for=\"1\">Manusia</span>\n              </button>\n              <button type=\"button\" class=\"seat-btn\" data-seat=\"2\">\n                <span class=\"disc-badge p2\"></span>\n                <span class=\"seat-name\">Pemain 2</span>\n                <span class=\"seat-role-tag is-bot\" data-role-for=\"2\">Bot</span>\n              </button>\n              <button type=\"button\" class=\"seat-btn\" data-seat=\"3\">\n                <span class=\"disc-badge p3\"></span>\n                <span class=\"seat-name\">Pemain 3</span>\n                <span class=\"seat-role-tag is-bot\" data-role-for=\"3\">Bot</span>\n              </button>\n              <button type=\"button\" class=\"seat-btn\" data-seat=\"4\">\n                <span class=\"disc-badge p4\"></span>\n                <span class=\"seat-name\">Pemain 4</span>\n                <span class=\"seat-role-tag is-bot\" data-role-for=\"4\">Bot</span>\n              </button>\n            </div>\n          </div>\n\n          <div class=\"drawer-section\">\n            <span id=\"firstSectionLabel\" class=\"group-label\">Jalan pertama</span>\n            <div id=\"firstChoiceGrid\" class=\"choice-grid cols-3-always\">\n              <button type=\"button\" class=\"choice-card first-card active\" data-first=\"1\">\n                <span class=\"disc-badge p1\"></span>\n                <span class=\"choice-text\">\n                  <strong id=\"firstP1Title\" class=\"choice-title\">Manusia</strong>\n                  <small id=\"firstP1Sub\" class=\"choice-sub\">Coral</small>\n                </span>\n              </button>\n              <button type=\"button\" class=\"choice-card first-card\" data-first=\"2\">\n                <span class=\"disc-badge p2\"></span>\n                <span class=\"choice-text\">\n                  <strong id=\"firstP2Title\" class=\"choice-title\">Bot AI</strong>\n                  <small id=\"firstP2Sub\" class=\"choice-sub\">Kuning</small>\n                </span>\n              </button>\n              <button type=\"button\" class=\"choice-card first-card\" data-first=\"3\" style=\"display: none;\">\n                <span class=\"disc-badge p3\"></span>\n                <span class=\"choice-text\">\n                  <strong id=\"firstP3Title\" class=\"choice-title\">Bot 3</strong>\n                  <small class=\"choice-sub\">Hijau</small>\n                </span>\n              </button>\n              <button type=\"button\" class=\"choice-card first-card\" data-first=\"4\" style=\"display: none;\">\n                <span class=\"disc-badge p4\"></span>\n                <span class=\"choice-text\">\n                  <strong id=\"firstP4Title\" class=\"choice-title\">Bot 4</strong>\n                  <small class=\"choice-sub\">Ungu</small>\n                </span>\n              </button>\n              <button type=\"button\" class=\"choice-card first-card first-random\" data-first=\"random\">\n                <span class=\"disc-badge random\">?</span>\n                <span class=\"choice-text\">\n                  <strong class=\"choice-title\">Acak</strong>\n                  <small id=\"firstRandomSub\" class=\"choice-sub\">Undi 50:50</small>\n                </span>\n              </button>\n            </div>\n          </div>\n\n          <div id=\"customConfigBox\" class=\"drawer-section\" style=\"display: none;\">\n            <span class=\"group-label\">Algoritma &amp; kedalaman custom</span>\n            <div class=\"custom-select-row\">\n              <label class=\"field-group\" for=\"algorithmSelect\">\n                <span>Algoritma AI</span>\n                <select id=\"algorithmSelect\" class=\"field-select\">\n                  <option value=\"alphabeta\">Alpha-Beta Pruning</option>\n                  <option value=\"minimax\">Minimax Polos</option>\n                </select>\n              </label>\n              <label class=\"field-group\" for=\"depthSelect\">\n                <span>Kedalaman (Depth)</span>\n                <select id=\"depthSelect\" class=\"field-select\">\n                  <option value=\"2\">Depth 2 (Cepat)</option>\n                  <option value=\"3\">Depth 3 (Seimbang)</option>\n                  <option value=\"4\" selected>Depth 4 (Kuat)</option>\n                  <option value=\"5\">Depth 5 (Maksimal)</option>\n                </select>\n              </label>\n            </div>\n          </div>\n        </div>\n\n        <div class=\"drawer-footer\">\n          <button type=\"button\" id=\"applyAdvancedBtn\" class=\"btn-primary\">Selesai</button>\n        </div>\n      </div>\n    </div>\n\n    <!-- UNIFIED HELP MODAL (ATURAN & CARA KERJA AI) -->\n    <div id=\"rulesModal\" class=\"modal-backdrop\" role=\"dialog\" aria-modal=\"true\" aria-labelledby=\"helpModalHeading\" style=\"display: none;\">\n      <div class=\"modal-card\">\n        <div class=\"modal-header\">\n          <h3 id=\"helpModalHeading\" class=\"modal-title\">Panduan &amp; Cara Kerja AI</h3>\n          <button type=\"button\" id=\"closeRulesBtn\" class=\"modal-close-btn\" aria-label=\"Tutup panduan\">&times;</button>\n        </div>\n\n        <div class=\"tab-bar modal-tabs\" role=\"tablist\">\n          <button type=\"button\" class=\"tab-btn active\" role=\"tab\" aria-selected=\"true\" data-help-tab=\"rules\">Aturan main</button>\n          <button type=\"button\" class=\"tab-btn\" role=\"tab\" aria-selected=\"false\" data-help-tab=\"ai\">Cara kerja AI</button>\n        </div>\n\n        <div class=\"modal-body\">\n          <div id=\"helpPanelRules\" class=\"help-panel\">\n            <ul class=\"help-list\">\n              <li>Klik kolom mana saja pada papan atau tekan tombol angka <strong>0&ndash;9</strong> di keyboard untuk menjatuhkan bidak ke baris kosong terbawah.</li>\n              <li>Hubungkan <strong>4 bidak</strong> dengan warna sama secara horizontal, vertikal, atau diagonal untuk langsung menang.</li>\n              <li>Mode <strong>Obstacle</strong> menaruh batu penghalang tetap yang memblokir kolom serta memutus jalur 4 bidak.</li>\n              <li>Mode <strong>4-Player</strong> memutar giliran Coral &rarr; Kuning &rarr; Hijau &rarr; Ungu, dengan komposisi manusia &amp; bot bebas diatur di Opsi lanjutan.</li>\n            </ul>\n          </div>\n\n          <div id=\"helpPanelAi\" class=\"help-panel\" style=\"display: none;\">\n            <p class=\"help-intro\">\n              Agen AI menelusuri cabang langkah hingga batas kedalaman (<em>depth</em>) memakai algoritma\n              <strong>Minimax</strong> dan <strong>Alpha-Beta Pruning</strong> (<em>Paranoid Minimax</em> di 4-Player),\n              lalu menilai setiap jendela 4-petak dengan heuristik berikut:\n            </p>\n            <div id=\"modalLegendList\" class=\"legend-rows\"></div>\n          </div>\n        </div>\n      </div>\n    </div>\n\n    <div id=\"infoModal\" style=\"display: none;\" aria-hidden=\"true\">\n      <button type=\"button\" id=\"closeInfoBtn\" tabindex=\"-1\"></button>\n    </div>\n\n    <div id=\"pwaUpdateToast\" class=\"pwa-toast\" role=\"status\" aria-live=\"polite\" style=\"display: none;\">\n      <span class=\"pwa-toast-text\">Update tersedia!</span>\n      <button type=\"button\" id=\"pwaReloadBtn\" class=\"pwa-reload-btn\">Muat ulang</button>\n    </div>\n\n    <!-- PWA FLOATING NOTIFICATION BANNER -->\n    <div id=\"pwaBanner\" class=\"pwa-banner\" role=\"region\" aria-label=\"Notifikasi instalasi aplikasi\" style=\"display: none;\">\n      <div class=\"pwa-banner-content\">\n        <span class=\"pwa-banner-icon\" aria-hidden=\"true\">📲</span>\n        <div class=\"pwa-banner-text\">\n          <strong>Pasang Connect-Four AI</strong>\n          <p>Mainkan lebih cepat, responsif, dan offline kapan saja tanpa kuota.</p>\n        </div>\n      </div>\n      <div class=\"pwa-banner-actions\">\n        <button type=\"button\" id=\"pwaBannerInstallBtn\" class=\"btn-pwa-primary\">Pasang sekarang</button>\n        <button type=\"button\" id=\"pwaBannerCloseBtn\" class=\"btn-pwa-dismiss\" aria-label=\"Tutup notifikasi\">&times;</button>\n      </div>\n    </div>\n\n    <!-- PWA GUIDE MODAL -->\n    <div id=\"pwaGuideModal\" class=\"modal-backdrop\" role=\"dialog\" aria-modal=\"true\" aria-labelledby=\"pwaGuideHeading\" style=\"display: none;\">\n      <div class=\"modal-card\">\n        <div class=\"modal-header\">\n          <h3 id=\"pwaGuideHeading\" class=\"modal-title\">Panduan Instalasi PWA</h3>\n          <button type=\"button\" id=\"closePwaGuideBtn\" class=\"modal-close-btn\" aria-label=\"Tutup panduan\">&times;</button>\n        </div>\n        <div class=\"modal-body pwa-guide-body\">\n          <div class=\"pwa-guide-card\">\n            <h4>Mengapa fitur PWA butuh Localhost atau HTTPS?</h4>\n            <p>Aturan keamanan browser mewajibkan <em>Service Worker</em> dijalankan melalui koneksi aman (<strong>HTTPS</strong> atau <strong>http://localhost</strong>). Jika dibuka langsung lewat file (<code>file:///</code>), browser mematikan instalasi otomatis.</p>\n          </div>\n          <div class=\"pwa-guide-steps\">\n            <h4>Langkah Mudah Menjalankan &amp; Memasang PWA:</h4>\n            <ol>\n              <li>Buka terminal di folder project lalu jalankan server lokal:\n                <pre><code>python -m http.server 8080</code></pre>\n                atau\n                <pre><code>npx serve .</code></pre>\n              </li>\n              <li>Buka browser di <strong>http://localhost:8080</strong> (atau buka link GitHub Pages).</li>\n              <li>Klik ikon <strong>⬇ Instal</strong> di pojok kanan address bar browser (atau tombol <strong>⬇ Instal</strong> di bar atas game).</li>\n            </ol>\n          </div>\n        </div>\n      </div>\n    </div>\n  </div>\n<script>\n/**\n * Connect-Four AI Engine — TIM 12 UTS PjBL Kecerdasan Buatan\n * Sinkron 1:1 dengan implementasi Python di Praktikum_AI_UTS_TIM12.ipynb.\n *\n * Fitur:\n * - Adversarial Search: Depth-Limited Minimax & Alpha-Beta Pruning\n * - Dukungan Mode 2-Player (Minimax Klasik) & 4-Player (Paranoid Minimax + Alpha-Beta)\n * - Heuristic Board Evaluation Function (4-cell window + kontrol kolom tengah)\n * - Dukungan variasi papan & ukuran dinamis: 6x7, 7x8, 7x9 + Klasik, Obstacle\n */\n(function (root, factory) {\n  if (typeof module === 'object' && module.exports) {\n    module.exports = factory();\n  } else {\n    root.ConnectFourAI = factory();\n  }\n})(typeof globalThis !== 'undefined' ? globalThis : this, function () {\n  'use strict';\n\n  const ROWS = 6;\n  const COLS = 7;\n  const WINDOW_LENGTH = 4;\n\n  const BLOCKED = -1;\n  const EMPTY = 0;\n  const HUMAN = 1; // Pemain 1 (Coral)\n  const AI = 2;    // Pemain 2 (Kuning)\n  const P3 = 3;    // Pemain 3 (Hijau)\n  const P4 = 4;    // Pemain 4 (Ungu)\n\n  const WIN_SCORE = 100000;\n  const COLUMN_ORDER = [3, 2, 4, 1, 5, 0, 6];\n\n  const BOARD_SIZES = {\n    '6x7': { rows: 6, cols: 7, label: '6×7' },\n    '7x8': { rows: 7, cols: 8, label: '7×8' },\n    '7x9': { rows: 7, cols: 9, label: '7×9' },\n    '8x10': { rows: 8, cols: 10, label: '8×10' },\n  };\n\n  const EVAL_WEIGHTS = {\n    WIN: 100000,\n    AI_THREE: 12,\n    AI_TWO: 3,\n    OPP_THREE: -80,\n    OPP_TWO: -2,\n    CENTER_PER_DISC: 4,\n  };\n\n  const DIFFICULTY_PRESETS = {\n    Easy: { maxDepth: 0, usePruning: false, label: 'Easy (Random Move)' },\n    Medium: { maxDepth: 3, usePruning: false, label: 'Medium (Depth-Limited Minimax, depth=3)' },\n    Hard: { maxDepth: 5, usePruning: true, label: 'Hard (Depth-Limited + Alpha-Beta, depth=5)' },\n  };\n\n  /**\n   * Kembalikan urutan kolom tengah-ke-pinggir untuk jumlah kolom berapa pun.\n   */\n  function getColumnOrder(cols = COLS) {\n    const order = [];\n    if (cols % 2 === 1) {\n      const mid = Math.floor(cols / 2);\n      order.push(mid);\n      for (let d = 1; d <= mid; d++) {\n        order.push(mid - d);\n        order.push(mid + d);\n      }\n    } else {\n      const left = cols / 2 - 1;\n      const right = cols / 2;\n      for (let d = 0; d < cols / 2; d++) {\n        order.push(left - d);\n        order.push(right + d);\n      }\n    }\n    return order;\n  }\n\n  /**\n   * Kembalikan pemain berikutnya secara siklik (1..playerCount).\n   */\n  function getNextPlayer(piece, playerCount = 2) {\n    const total = playerCount >= 4 ? 4 : 2;\n    return piece >= total ? 1 : piece + 1;\n  }\n\n  /**\n   * Buat papan kosong berukuran rows x cols (baris 0 = paling atas).\n   */\n  function createBoard(rows = ROWS, cols = COLS) {\n    const board = [];\n    for (let r = 0; r < rows; r++) {\n      board.push(new Array(cols).fill(EMPTY));\n    }\n    return board;\n  }\n\n  /**\n   * Buat konfigurasi awal papan sesuai mode ('classic', 'obstacle') dan ukuran rows x cols.\n   * Mengembalikan { board, obstacles }.\n   */\n  function createSpecialSetup(boardMode = 'classic', rows = ROWS, cols = COLS) {\n    const board = createBoard(rows, cols);\n    const obstacles = [];\n\n    if (boardMode === 'obstacle') {\n      const centerSet = new Set(\n        cols % 2 === 1 ? [Math.floor(cols / 2)] : [cols / 2 - 1, cols / 2]\n      );\n      const candidateCols = [];\n      for (let c = 0; c < cols; c++) {\n        if (!centerSet.has(c)) candidateCols.push(c);\n      }\n      for (let i = candidateCols.length - 1; i > 0; i--) {\n        const j = Math.floor(Math.random() * (i + 1));\n        const tmp = candidateCols[i];\n        candidateCols[i] = candidateCols[j];\n        candidateCols[j] = tmp;\n      }\n\n      const spots = [\n        [rows - 1, candidateCols[0]],\n        [rows - 1, candidateCols[1]],\n        [rows - 2, candidateCols[1]],\n        [rows - 1, candidateCols[2]],\n      ];\n      if (cols >= 8 && candidateCols.length >= 4) {\n        spots.push([rows - 1, candidateCols[3]]);\n      }\n      if (cols >= 10 && candidateCols.length >= 4) {\n        spots.push([rows - 2, candidateCols[3]]);\n      }\n\n      spots.forEach(([r, c]) => {\n        board[r][c] = BLOCKED;\n        obstacles.push([r, c]);\n      });\n    }\n\n    return { board, obstacles };\n  }\n\n  /**\n   * Salin papan 2D secara mendalam.\n   */\n  function cloneBoard(board) {\n    return board.map((row) => row.slice());\n  }\n\n  /**\n   * Cek apakah kolom masih memiliki ruang kosong di paling atas.\n   */\n  function isValidLocation(board, col) {\n    const cols = board[0].length;\n    return col >= 0 && col < cols && board[0][col] === EMPTY;\n  }\n\n  /**\n   * Kembalikan daftar kolom valid sesuai urutan tengah-ke-pinggir.\n   */\n  function getValidLocations(board) {\n    const order = getColumnOrder(board[0].length);\n    const valid = [];\n    for (let i = 0; i < order.length; i++) {\n      const col = order[i];\n      if (isValidLocation(board, col)) {\n        valid.push(col);\n      }\n    }\n    return valid;\n  }\n\n  /**\n   * Cari baris terbawah yang masih kosong pada kolom tertentu.\n   */\n  function getNextOpenRow(board, col) {\n    for (let r = board.length - 1; r >= 0; r--) {\n      if (board[r][col] === EMPTY) {\n        return r;\n      }\n    }\n    return -1;\n  }\n\n  /**\n   * Jatuhkan bidak ke koordinat (row, col).\n   */\n  function dropPiece(board, row, col, piece) {\n    board[row][col] = piece;\n  }\n\n  /**\n   * Cek pemenang (mendukung pemain 1..4) atau seri beserta koordinat 4 bidak terhubung.\n   * Mengembalikan { winner: 1 | 2 | 3 | 4 | 'DRAW' | null, cells: [[r, c], ...] }.\n   */\n  function getWinnerInfo(board) {\n    const rows = board.length;\n    const cols = board[0].length;\n    const isPlayer = (p) => p >= 1 && p <= 4;\n\n    // Cek horizontal\n    for (let r = 0; r < rows; r++) {\n      for (let c = 0; c <= cols - WINDOW_LENGTH; c++) {\n        const p = board[r][c];\n        if (isPlayer(p) && p === board[r][c + 1] && p === board[r][c + 2] && p === board[r][c + 3]) {\n          return { winner: p, cells: [[r, c], [r, c + 1], [r, c + 2], [r, c + 3]] };\n        }\n      }\n    }\n\n    // Cek vertikal\n    for (let c = 0; c < cols; c++) {\n      for (let r = 0; r <= rows - WINDOW_LENGTH; r++) {\n        const p = board[r][c];\n        if (isPlayer(p) && p === board[r + 1][c] && p === board[r + 2][c] && p === board[r + 3][c]) {\n          return { winner: p, cells: [[r, c], [r + 1, c], [r + 2, c], [r + 3, c]] };\n        }\n      }\n    }\n\n    // Cek diagonal turun ke kanan (\\)\n    for (let r = 0; r <= rows - WINDOW_LENGTH; r++) {\n      for (let c = 0; c <= cols - WINDOW_LENGTH; c++) {\n        const p = board[r][c];\n        if (isPlayer(p) && p === board[r + 1][c + 1] && p === board[r + 2][c + 2] && p === board[r + 3][c + 3]) {\n          return { winner: p, cells: [[r, c], [r + 1, c + 1], [r + 2, c + 2], [r + 3, c + 3]] };\n        }\n      }\n    }\n\n    // Cek diagonal naik ke kanan (/)\n    for (let r = WINDOW_LENGTH - 1; r < rows; r++) {\n      for (let c = 0; c <= cols - WINDOW_LENGTH; c++) {\n        const p = board[r][c];\n        if (isPlayer(p) && p === board[r - 1][c + 1] && p === board[r - 2][c + 2] && p === board[r - 3][c + 3]) {\n          return { winner: p, cells: [[r, c], [r - 1, c + 1], [r - 2, c + 2], [r - 3, c + 3]] };\n        }\n      }\n    }\n\n    // Cek seri jika semua kolom teratas penuh\n    if (getValidLocations(board).length === 0) {\n      return { winner: 'DRAW', cells: [] };\n    }\n\n    return { winner: null, cells: [] };\n  }\n\n  /**\n   * Wrapper sederhana mengembalikan 1..4, 'DRAW', atau null.\n   */\n  function checkWinner(board) {\n    return getWinnerInfo(board).winner;\n  }\n\n  /**\n   * Analisis satu jendela 4-petak untuk 2-Player maupun 4-Player.\n   * Jendela hanya bernilai jika hanya diisi 1 pihak (+ petak kosong).\n   */\n  function classifyWindow(windowCells, aiPiece) {\n    let aiCount = 0;\n    let emptyCount = 0;\n    let soleOpp = 0;\n    let soleOppCount = 0;\n\n    for (let i = 0; i < windowCells.length; i++) {\n      const cell = windowCells[i];\n      if (cell === BLOCKED) return { kind: 'blocked', score: 0 };\n      if (cell === EMPTY) {\n        emptyCount++;\n      } else if (cell === aiPiece) {\n        aiCount++;\n      } else {\n        if (soleOpp === 0) {\n          soleOpp = cell;\n          soleOppCount = 1;\n        } else if (soleOpp === cell) {\n          soleOppCount++;\n        } else {\n          // 2 lawan berbeda bentrok dalam satu jendela -> tidak bisa menjadi 4-in-a-row\n          return { kind: 'mixed', score: 0 };\n        }\n      }\n    }\n\n    if (aiCount > 0 && soleOppCount > 0) {\n      return { kind: 'mixed', score: 0 };\n    }\n    if (aiCount === 4) return { kind: 'ai_win', score: EVAL_WEIGHTS.WIN };\n    if (soleOppCount === 4) return { kind: 'opp_win', score: -EVAL_WEIGHTS.WIN };\n    if (aiCount === 3 && emptyCount === 1) return { kind: 'ai3', score: EVAL_WEIGHTS.AI_THREE };\n    if (aiCount === 2 && emptyCount === 2) return { kind: 'ai2', score: EVAL_WEIGHTS.AI_TWO };\n    if (soleOppCount === 3 && emptyCount === 1) return { kind: 'opp3', score: EVAL_WEIGHTS.OPP_THREE };\n    if (soleOppCount === 2 && emptyCount === 2) return { kind: 'opp2', score: EVAL_WEIGHTS.OPP_TWO };\n    return { kind: 'neutral', score: 0 };\n  }\n\n  /**\n   * Hitung skor satu jendela berisi 4 petak dari sudut pandang aiPiece.\n   */\n  function evaluateWindow(windowCells, aiPiece) {\n    return classifyWindow(windowCells, aiPiece).score;\n  }\n\n  /**\n   * Analisis papan menyeluruh untuk ukuran rows x cols dan 2 atau 4 pemain.\n   */\n  function analyzeBoard(board, aiPiece = AI, playerCount = 2) {\n    const rows = board.length;\n    const cols = board[0].length;\n    const totalPlayers = playerCount >= 4 ? 4 : 2;\n\n    const centerCols = cols % 2 === 1 ? [Math.floor(cols / 2)] : [cols / 2 - 1, cols / 2];\n    const centerWeight = cols % 2 === 1 ? EVAL_WEIGHTS.CENTER_PER_DISC : Math.floor(EVAL_WEIGHTS.CENTER_PER_DISC / 2);\n\n    let centerAi = 0;\n    const oppCenterCounts = { 1: 0, 2: 0, 3: 0, 4: 0 };\n    for (let i = 0; i < centerCols.length; i++) {\n      const cc = centerCols[i];\n      for (let r = 0; r < rows; r++) {\n        const cell = board[r][cc];\n        if (cell === aiPiece) {\n          centerAi++;\n        } else if (cell >= 1 && cell <= totalPlayers) {\n          oppCenterCounts[cell] += 1;\n        }\n      }\n    }\n\n    let centerOpp = 0;\n    for (let p = 1; p <= totalPlayers; p++) {\n      if (p !== aiPiece && oppCenterCounts[p] > centerOpp) {\n        centerOpp = oppCenterCounts[p];\n      }\n    }\n\n    let aiThree = 0;\n    let aiTwo = 0;\n    let oppThree = 0;\n    let oppTwo = 0;\n    let windowScore = 0;\n\n    function processWindow(w) {\n      const res = classifyWindow(w, aiPiece);\n      if (res.kind === 'ai3') aiThree++;\n      else if (res.kind === 'ai2') aiTwo++;\n      else if (res.kind === 'opp3') oppThree++;\n      else if (res.kind === 'opp2') oppTwo++;\n      windowScore += res.score;\n    }\n\n    // Horizontal\n    for (let r = 0; r < rows; r++) {\n      for (let c = 0; c <= cols - WINDOW_LENGTH; c++) {\n        processWindow([board[r][c], board[r][c + 1], board[r][c + 2], board[r][c + 3]]);\n      }\n    }\n\n    // Vertikal\n    for (let c = 0; c < cols; c++) {\n      for (let r = 0; r <= rows - WINDOW_LENGTH; r++) {\n        processWindow([board[r][c], board[r + 1][c], board[r + 2][c], board[r + 3][c]]);\n      }\n    }\n\n    // Diagonal turun (\\)\n    for (let r = 0; r <= rows - WINDOW_LENGTH; r++) {\n      for (let c = 0; c <= cols - WINDOW_LENGTH; c++) {\n        processWindow([board[r][c], board[r + 1][c + 1], board[r + 2][c + 2], board[r + 3][c + 3]]);\n      }\n    }\n\n    // Diagonal naik (/)\n    for (let r = WINDOW_LENGTH - 1; r < rows; r++) {\n      for (let c = 0; c <= cols - WINDOW_LENGTH; c++) {\n        processWindow([board[r][c], board[r - 1][c + 1], board[r - 2][c + 2], board[r - 3][c + 3]]);\n      }\n    }\n\n    const centerScore = (centerAi - centerOpp) * centerWeight;\n    return {\n      totalScore: windowScore + centerScore,\n      centerAi,\n      centerOpp,\n      centerScore,\n      aiThree,\n      aiTwo,\n      oppThree,\n      oppTwo,\n      windowScore,\n    };\n  }\n\n  /**\n   * Fungsi Evaluasi Papan (Evaluation Function) untuk non-terminal state.\n   */\n  function evaluateBoard(board, aiPiece = AI, playerCount = 2) {\n    return analyzeBoard(board, aiPiece, playerCount).totalScore;\n  }\n\n  /**\n   * Depth-Limited Minimax Murni (tanpa Alpha-Beta Pruning).\n   * Pada 2P: bergantian MAX (aiPiece) vs MIN (lawan).\n   * Pada 4P (Paranoid Minimax): giliran berotasi 1->2->3->4 dengan semua non-aiPiece berperan MIN.\n   */\n  function minimax(board, depth, maxDepth, turnOrMax, aiPiece, stats, playerCount = 2) {\n    stats.nodes += 1;\n    const totalPlayers = playerCount >= 4 ? 4 : 2;\n    const turnPiece =\n      typeof turnOrMax === 'boolean'\n        ? (turnOrMax ? aiPiece : getNextPlayer(aiPiece, totalPlayers))\n        : turnOrMax;\n\n    const winner = checkWinner(board);\n    if (winner === aiPiece) return WIN_SCORE - depth;\n    if (winner === 'DRAW') return 0;\n    if (winner !== null) return depth - WIN_SCORE;\n\n    if (depth >= maxDepth) {\n      return evaluateBoard(board, aiPiece, totalPlayers);\n    }\n\n    const validCols = getValidLocations(board);\n    const nextPiece = getNextPlayer(turnPiece, totalPlayers);\n    const isMaximizing = turnPiece === aiPiece;\n\n    if (isMaximizing) {\n      let bestScore = -Infinity;\n      for (let i = 0; i < validCols.length; i++) {\n        const col = validCols[i];\n        const row = getNextOpenRow(board, col);\n        board[row][col] = turnPiece;\n        const score = minimax(board, depth + 1, maxDepth, nextPiece, aiPiece, stats, totalPlayers);\n        board[row][col] = EMPTY;\n        if (score > bestScore) bestScore = score;\n      }\n      return bestScore;\n    }\n\n    let bestScore = Infinity;\n    for (let i = 0; i < validCols.length; i++) {\n      const col = validCols[i];\n      const row = getNextOpenRow(board, col);\n      board[row][col] = turnPiece;\n      const score = minimax(board, depth + 1, maxDepth, nextPiece, aiPiece, stats, totalPlayers);\n      board[row][col] = EMPTY;\n      if (score < bestScore) bestScore = score;\n    }\n    return bestScore;\n  }\n\n  /**\n   * Depth-Limited Minimax dengan optimasi Alpha-Beta Pruning (2P & 4P Paranoid).\n   */\n  function minimaxAlphaBeta(board, depth, maxDepth, alpha, beta, turnOrMax, aiPiece, stats, playerCount = 2) {\n    stats.nodes += 1;\n    const totalPlayers = playerCount >= 4 ? 4 : 2;\n    const turnPiece =\n      typeof turnOrMax === 'boolean'\n        ? (turnOrMax ? aiPiece : getNextPlayer(aiPiece, totalPlayers))\n        : turnOrMax;\n\n    const winner = checkWinner(board);\n    if (winner === aiPiece) return WIN_SCORE - depth;\n    if (winner === 'DRAW') return 0;\n    if (winner !== null) return depth - WIN_SCORE;\n\n    if (depth >= maxDepth) {\n      return evaluateBoard(board, aiPiece, totalPlayers);\n    }\n\n    const validCols = getValidLocations(board);\n    const nextPiece = getNextPlayer(turnPiece, totalPlayers);\n    const isMaximizing = turnPiece === aiPiece;\n\n    if (isMaximizing) {\n      let maxEval = -Infinity;\n      for (let i = 0; i < validCols.length; i++) {\n        const col = validCols[i];\n        const row = getNextOpenRow(board, col);\n        board[row][col] = turnPiece;\n        const evalScore = minimaxAlphaBeta(board, depth + 1, maxDepth, alpha, beta, nextPiece, aiPiece, stats, totalPlayers);\n        board[row][col] = EMPTY;\n        if (evalScore > maxEval) maxEval = evalScore;\n        if (maxEval > alpha) alpha = maxEval;\n        if (alpha >= beta) break;\n      }\n      return maxEval;\n    }\n\n    let minEval = Infinity;\n    for (let i = 0; i < validCols.length; i++) {\n      const col = validCols[i];\n      const row = getNextOpenRow(board, col);\n      board[row][col] = turnPiece;\n      const evalScore = minimaxAlphaBeta(board, depth + 1, maxDepth, alpha, beta, nextPiece, aiPiece, stats, totalPlayers);\n      board[row][col] = EMPTY;\n      if (evalScore < minEval) minEval = evalScore;\n      if (minEval < beta) beta = minEval;\n      if (alpha >= beta) break;\n    }\n    return minEval;\n  }\n\n  /**\n   * Cari langkah terbaik beserta skor evaluasi tiap kolom dan metrik pencarian.\n   */\n  function findBestMove(board, options = {}) {\n    const maxDepth = options.maxDepth !== undefined ? options.maxDepth : 4;\n    const aiPiece = options.aiPiece !== undefined ? options.aiPiece : AI;\n    const usePruning = options.usePruning !== undefined ? options.usePruning : true;\n    const playerCount = options.playerCount !== undefined ? options.playerCount : 2;\n\n    const workingBoard = cloneBoard(board);\n    const cols = workingBoard[0].length;\n    const validCols = getValidLocations(workingBoard);\n    const columnScores = {};\n    for (let c = 0; c < cols; c++) {\n      columnScores[c] = null;\n    }\n\n    if (validCols.length === 0) {\n      return {\n        bestCol: -1,\n        bestScore: 0,\n        columnScores,\n        nodes: 0,\n        elapsedMs: 0,\n        maxDepth,\n        usePruning,\n      };\n    }\n\n    const stats = { nodes: 0 };\n    const t0 = typeof performance !== 'undefined' ? performance.now() : Date.now();\n\n    let bestScore = -Infinity;\n    let bestCol = validCols[0];\n    let alpha = -Infinity;\n    const beta = Infinity;\n    const nextTurn = getNextPlayer(aiPiece, playerCount);\n\n    for (let i = 0; i < validCols.length; i++) {\n      const col = validCols[i];\n      const row = getNextOpenRow(workingBoard, col);\n      workingBoard[row][col] = aiPiece;\n\n      const score = usePruning\n        ? minimaxAlphaBeta(workingBoard, 1, maxDepth, alpha, beta, nextTurn, aiPiece, stats, playerCount)\n        : minimax(workingBoard, 1, maxDepth, nextTurn, aiPiece, stats, playerCount);\n\n      workingBoard[row][col] = EMPTY;\n      columnScores[col] = score;\n\n      if (score > bestScore) {\n        bestScore = score;\n        bestCol = col;\n      }\n      if (usePruning && bestScore > alpha) {\n        alpha = bestScore;\n      }\n    }\n\n    const t1 = typeof performance !== 'undefined' ? performance.now() : Date.now();\n\n    return {\n      bestCol,\n      bestScore,\n      columnScores,\n      nodes: stats.nodes,\n      elapsedMs: Number((t1 - t0).toFixed(2)),\n      maxDepth,\n      usePruning,\n    };\n  }\n\n  /**\n   * Pilih langkah Bot AI berdasarkan tingkat kesulitan ('Easy', 'Medium', 'Hard')\n   * atau konfigurasi depth/pruning kustom.\n   */\n  function chooseMove(board, options = {}) {\n    const difficulty = options.difficulty || 'Hard';\n    const aiPiece = options.aiPiece !== undefined ? options.aiPiece : AI;\n    const playerCount = options.playerCount !== undefined ? options.playerCount : 2;\n    const preset = DIFFICULTY_PRESETS[difficulty] || DIFFICULTY_PRESETS.Hard;\n    const cols = board[0].length;\n\n    if (difficulty === 'Easy' && options.customDepth === undefined) {\n      const validCols = getValidLocations(board);\n      const columnScores = {};\n      for (let c = 0; c < cols; c++) {\n        columnScores[c] = null;\n      }\n      if (validCols.length === 0) {\n        return { bestCol: -1, bestScore: 0, columnScores, nodes: 0, elapsedMs: 0, maxDepth: 0, usePruning: false, isRandom: true };\n      }\n      const workingBoard = cloneBoard(board);\n      const t0 = typeof performance !== 'undefined' ? performance.now() : Date.now();\n      for (let i = 0; i < validCols.length; i++) {\n        const c = validCols[i];\n        const r = getNextOpenRow(workingBoard, c);\n        workingBoard[r][c] = aiPiece;\n        columnScores[c] = evaluateBoard(workingBoard, aiPiece, playerCount);\n        workingBoard[r][c] = EMPTY;\n      }\n      const randomCol = validCols[Math.floor(Math.random() * validCols.length)];\n      const t1 = typeof performance !== 'undefined' ? performance.now() : Date.now();\n      return {\n        bestCol: randomCol,\n        bestScore: columnScores[randomCol],\n        columnScores,\n        nodes: validCols.length,\n        elapsedMs: Number((t1 - t0).toFixed(2)),\n        maxDepth: 0,\n        usePruning: false,\n        isRandom: true,\n      };\n    }\n\n    const defaultDepth = (playerCount >= 4 || cols >= 10) && difficulty === 'Hard' ? 4 : preset.maxDepth;\n    const maxDepth = options.customDepth !== undefined ? options.customDepth : defaultDepth;\n    const usePruning = options.usePruning !== undefined ? options.usePruning : preset.usePruning;\n\n    const result = findBestMove(board, { maxDepth, aiPiece, usePruning, playerCount });\n    result.isRandom = false;\n    return result;\n  }\n\n  /**\n   * Jalankan perbandingan langsung Minimax Murni vs Alpha-Beta Pruning pada papan saat ini.\n   */\n  function compareAlgorithms(board, maxDepth = 4, aiPiece = AI, playerCount = 2) {\n    const pureMinimax = findBestMove(board, { maxDepth, aiPiece, usePruning: false, playerCount });\n    const alphaBeta = findBestMove(board, { maxDepth, aiPiece, usePruning: true, playerCount });\n\n    const reductionPct =\n      pureMinimax.nodes > 0\n        ? Number((((pureMinimax.nodes - alphaBeta.nodes) / pureMinimax.nodes) * 100).toFixed(1))\n        : 0;\n\n    return {\n      maxDepth,\n      pureMinimax,\n      alphaBeta,\n      reductionPct,\n      sameDecision: pureMinimax.bestCol === alphaBeta.bestCol,\n    };\n  }\n\n  return {\n    ROWS,\n    COLS,\n    WINDOW_LENGTH,\n    BLOCKED,\n    EMPTY,\n    HUMAN,\n    AI,\n    P3,\n    P4,\n    WIN_SCORE,\n    COLUMN_ORDER,\n    BOARD_SIZES,\n    EVAL_WEIGHTS,\n    DIFFICULTY_PRESETS,\n    getColumnOrder,\n    getNextPlayer,\n    createBoard,\n    createSpecialSetup,\n    cloneBoard,\n    isValidLocation,\n    getValidLocations,\n    getNextOpenRow,\n    dropPiece,\n    getWinnerInfo,\n    checkWinner,\n    evaluateWindow,\n    analyzeBoard,\n    evaluateBoard,\n    minimax,\n    minimaxAlphaBeta,\n    findBestMove,\n    chooseMove,\n    compareAlgorithms,\n  };\n});\n\n\n(function () {\n  if (typeof google !== 'undefined' && google.colab && google.colab.kernel) {\n    window.ConnectFourBridge = {\n      label: 'Engine: Python Kernel Aktif (Google Colab • google.colab.output)',\n      async chooseMove(board, options) {\n        try {\n          const res = await google.colab.kernel.invokeFunction(\n            'connect4.bot_move',\n            [\n              JSON.stringify(board),\n              options.difficulty || 'Hard',\n              options.aiPiece !== undefined ? options.aiPiece : 2,\n              options.usePruning !== undefined ? options.usePruning : null,\n              options.customDepth !== undefined ? options.customDepth : null,\n              options.playerCount !== undefined ? options.playerCount : 2,\n            ],\n            {}\n          );\n          const data = res && res.data && res.data['application/json'];\n          if (data) return data;\n        } catch (err) {\n          console.warn('Colab Python bridge fallback ke JS:', err);\n        }\n        return window.ConnectFourAI.chooseMove(board, options);\n      },\n      async compareAlgorithms(board, depth, aiPiece, playerCount) {\n        try {\n          const res = await google.colab.kernel.invokeFunction(\n            'connect4.compare',\n            [JSON.stringify(board), depth, aiPiece, playerCount !== undefined ? playerCount : 2],\n            {}\n          );\n          const data = res && res.data && res.data['application/json'];\n          if (data) return data;\n        } catch (err) {\n          console.warn('Colab Python compare fallback ke JS:', err);\n        }\n        return window.ConnectFourAI.compareAlgorithms(board, depth, aiPiece, playerCount !== undefined ? playerCount : 2);\n      },\n    };\n  }\n})();\n\n(function () {\n  'use strict';\n\n  const AI_LIB = window.ConnectFourAI;\n  const { ROWS, COLS, BLOCKED, EMPTY, HUMAN, AI, WIN_SCORE } = AI_LIB;\n\n  const PLAYER_META = {\n    1: { num: 1, color: 'Coral', css: 'p1' },\n    2: { num: 2, color: 'Kuning', css: 'p2' },\n    3: { num: 3, color: 'Hijau', css: 'p3' },\n    4: { num: 4, color: 'Ungu', css: 'p4' },\n  };\n\n  const FOUR_PRESETS = {\n    '4human': { 1: 'human', 2: 'human', 3: 'human', 4: 'human' },\n    '1h3b': { 1: 'human', 2: 'bot', 3: 'bot', 4: 'bot' },\n    '2h2b': { 1: 'human', 2: 'human', 3: 'bot', 4: 'bot' },\n    '4bot': { 1: 'bot', 2: 'bot', 3: 'bot', 4: 'bot' },\n  };\n\n  const state = {\n    view: 'home', // 'home' | 'setup' | 'game'\n    gameMode: 'hvb', // 'hvb' | 'pvp' | '4p' | 'bvb'\n    fourSeats: { 1: 'human', 2: 'bot', 3: 'bot', 4: 'bot' },\n    boardMode: 'classic', // 'classic' | 'obstacle'\n    boardSize: '6x7', // '6x7' | '7x8' | '7x9'\n    userPickedSize: false,\n    difficulty: 'Medium', // 'Easy' | 'Medium' | 'Hard' | 'Custom'\n    firstChoice: '1', // '1' | '2' | '3' | '4' | 'random'\n    resolvedFirstPlayer: HUMAN,\n    randomStartNote: '',\n    board: AI_LIB.createBoard(),\n    currentPlayer: HUMAN,\n    gameOver: false,\n    winner: null,\n    winCells: [],\n    lastMove: null,\n    lastAiResult: null,\n    moveCount: 0,\n    moveLog: [],\n    history: [],\n    compareTarget: null,\n    isThinking: false,\n    isAnimatingDrop: false,\n    autoPlayPaused: false,\n    matchGeneration: 0,\n    autoPlayTimer: null,\n    pendingBotTimer: null,\n    postGameTimer: null,\n    dropTimer: null,\n    hoverCol: -1,\n  };\n\n  const modalCloseTimers = new WeakMap();\n\n  function getPlayerCount() {\n    return state.gameMode === '4p' ? 4 : 2;\n  }\n\n  function getSeatRoles() {\n    if (state.gameMode === 'hvb') return { 1: 'human', 2: 'bot' };\n    if (state.gameMode === 'pvp') return { 1: 'human', 2: 'human' };\n    if (state.gameMode === 'bvb') return { 1: 'bot', 2: 'bot' };\n    return { ...state.fourSeats };\n  }\n\n  function isBotPlayer(piece) {\n    return getSeatRoles()[piece] === 'bot';\n  }\n\n  function hasAnyBot() {\n    return Object.values(getSeatRoles()).includes('bot');\n  }\n\n  function isAllBots() {\n    return Object.values(getSeatRoles()).every((role) => role === 'bot');\n  }\n\n  function getPlayerLabel(piece, short = false) {\n    const meta = PLAYER_META[piece] || PLAYER_META[1];\n    if (state.gameMode === 'hvb') {\n      return piece === HUMAN ? (short ? 'Manusia' : 'Manusia (Coral)') : (short ? 'Bot AI' : 'Bot AI (Kuning)');\n    }\n    if (state.gameMode === 'pvp') {\n      return short ? `Pemain ${piece}` : `Pemain ${piece} (${meta.color})`;\n    }\n    if (state.gameMode === 'bvb') {\n      return short ? `Bot ${meta.color}` : `Bot ${piece} (${meta.color})`;\n    }\n    const roleTxt = isBotPlayer(piece) ? 'Bot' : 'P';\n    return short ? `${roleTxt}${piece} (${meta.color})` : `${isBotPlayer(piece) ? 'Bot' : 'Pemain'} ${piece} (${meta.color})`;\n  }\n\n  async function requestBotMove(board, options) {\n    const bridge = window.ConnectFourBridge;\n    if (bridge && typeof bridge.chooseMove === 'function') {\n      return await bridge.chooseMove(board, options);\n    }\n    return AI_LIB.chooseMove(board, options);\n  }\n\n  async function requestCompare(board, depth, aiPiece, playerCount) {\n    const bridge = window.ConnectFourBridge;\n    if (bridge && typeof bridge.compareAlgorithms === 'function') {\n      return await bridge.compareAlgorithms(board, depth, aiPiece, playerCount);\n    }\n    return AI_LIB.compareAlgorithms(board, depth, aiPiece, playerCount);\n  }\n\n  const els = {\n    // Views\n    homeView: document.getElementById('homeView'),\n    setupView: document.getElementById('setupView'),\n    gameView: document.getElementById('gameView'),\n    arenaShell: document.getElementById('arenaShell'),\n\n    // Home Menu\n    goSetupBtn: document.getElementById('goSetupBtn'),\n    openRulesBtn: document.getElementById('openRulesBtn'),\n    openInfoBtn: document.getElementById('openInfoBtn'),\n\n    // Setup Menu\n    backHomeBtn: document.getElementById('backHomeBtn'),\n    setupPreviewBadge: document.getElementById('setupPreviewBadge'),\n    modeChoiceGrid: document.getElementById('modeChoiceGrid'),\n    fourPlayerBox: document.getElementById('fourPlayerBox'),\n    fourPresetGrid: document.getElementById('fourPresetGrid'),\n    fourSeatsGrid: document.getElementById('fourSeatsGrid'),\n    boardModeGrid: document.getElementById('boardModeGrid'),\n    boardSizeGrid: document.getElementById('boardSizeGrid'),\n    diffSection: document.getElementById('diffSection'),\n    diffChoiceGrid: document.getElementById('diffChoiceGrid'),\n    firstSectionLabel: document.getElementById('firstSectionLabel'),\n    firstChoiceGrid: document.getElementById('firstChoiceGrid'),\n    firstP1Title: document.getElementById('firstP1Title'),\n    firstP1Sub: document.getElementById('firstP1Sub'),\n    firstP2Title: document.getElementById('firstP2Title'),\n    firstP2Sub: document.getElementById('firstP2Sub'),\n    firstP3Title: document.getElementById('firstP3Title'),\n    firstP4Title: document.getElementById('firstP4Title'),\n    firstRandomSub: document.getElementById('firstRandomSub'),\n    customConfigBox: document.getElementById('customConfigBox'),\n    algorithmSelect: document.getElementById('algorithmSelect'),\n    depthSelect: document.getElementById('depthSelect'),\n    startGameBtn: document.getElementById('startGameBtn'),\n\n    // Modals\n    rulesModal: document.getElementById('rulesModal'),\n    closeRulesBtn: document.getElementById('closeRulesBtn'),\n    infoModal: document.getElementById('infoModal'),\n    closeInfoBtn: document.getElementById('closeInfoBtn'),\n    modalLegendList: document.getElementById('modalLegendList'),\n\n    // Arena\n    arenaPill: document.getElementById('arenaPill'),\n    backMenuBtn: document.getElementById('backMenuBtn'),\n    backSetupBtn: document.getElementById('backSetupBtn'),\n    activeConfigBadge: document.getElementById('activeConfigBadge'),\n    turnCounterBadge: document.getElementById('turnCounterBadge'),\n    undoBtn: document.getElementById('undoBtn'),\n    resetBtn: document.getElementById('resetBtn'),\n    bvbBar: document.getElementById('bvbBar'),\n    botStepBtn: document.getElementById('botStepBtn'),\n    autoPlayBtn: document.getElementById('autoPlayBtn'),\n    statusBanner: document.getElementById('statusBanner'),\n    statusDot: document.getElementById('statusDot'),\n    statusText: document.getElementById('statusText'),\n    thinkingDots: document.getElementById('thinkingDots'),\n    lastMovePill: document.getElementById('lastMovePill'),\n    hoverPreviewRow: document.getElementById('hoverPreviewRow'),\n    colSelectors: document.getElementById('colSelectors'),\n    boardGrid: document.getElementById('boardGrid'),\n    boardLegend: document.getElementById('boardLegend'),\n\n    // Post-Game Analysis\n    postGamePanel: document.getElementById('postGamePanel'),\n    postGameBanner: document.getElementById('postGameBanner'),\n    postGameTitle: document.getElementById('postGameTitle'),\n    postGameSub: document.getElementById('postGameSub'),\n    playAgainBtn: document.getElementById('playAgainBtn'),\n    postMenuBtn: document.getElementById('postMenuBtn'),\n    statTotalMoves: document.getElementById('statTotalMoves'),\n    statBotMoves: document.getElementById('statBotMoves'),\n    statAvgNodes: document.getElementById('statAvgNodes'),\n    statTotalTime: document.getElementById('statTotalTime'),\n    postGameTableBody: document.getElementById('postGameTableBody'),\n    legendTotalBadge: document.getElementById('legendTotalBadge'),\n    heuristicGrid: document.getElementById('heuristicGrid'),\n    compareDepthSelect: document.getElementById('compareDepthSelect'),\n    compareBtn: document.getElementById('compareBtn'),\n    compareContextText: document.getElementById('compareContextText'),\n    compareResult: document.getElementById('compareResult'),\n\n    // UI Presentation Helpers (Lobby Preview, Drawer, Effects, Tabs, Overflow)\n    lobbyPreviewBoard: document.getElementById('lobbyPreviewBoard'),\n    lobbyPreviewLabel: document.getElementById('lobbyPreviewLabel'),\n    lobbyPreviewChips: document.getElementById('lobbyPreviewChips'),\n    openAdvancedBtn: document.getElementById('openAdvancedBtn'),\n    closeAdvancedBtn: document.getElementById('closeAdvancedBtn'),\n    applyAdvancedBtn: document.getElementById('applyAdvancedBtn'),\n    advancedSummaryBadge: document.getElementById('advancedSummaryBadge'),\n    advancedSetupDetails: document.getElementById('advancedSetupDetails'),\n    arenaOverflowMenu: document.getElementById('arenaOverflowMenu'),\n    botSpeedSelect: document.getElementById('botSpeedSelect'),\n    analysisTabSteps: document.getElementById('analysisTabSteps'),\n    analysisTabEfficiency: document.getElementById('analysisTabEfficiency'),\n    helpPanelRules: document.getElementById('helpPanelRules'),\n    helpPanelAi: document.getElementById('helpPanelAi'),\n    winLineSvg: document.getElementById('winLineSvg'),\n    confettiCanvas: document.getElementById('confettiCanvas'),\n    soundToggleBtn: document.getElementById('soundToggleBtn'),\n    sfxVolumeSlider: document.getElementById('sfxVolumeSlider'),\n    pwaInstallBtn: document.getElementById('pwaInstallBtn'),\n    pwaUpdateToast: document.getElementById('pwaUpdateToast'),\n    pwaReloadBtn: document.getElementById('pwaReloadBtn'),\n    boardDustLayer: document.getElementById('boardDustLayer'),\n    pwaBanner: document.getElementById('pwaBanner'),\n    pwaBannerInstallBtn: document.getElementById('pwaBannerInstallBtn'),\n    pwaBannerCloseBtn: document.getElementById('pwaBannerCloseBtn'),\n    pwaGuideModal: document.getElementById('pwaGuideModal'),\n    closePwaGuideBtn: document.getElementById('closePwaGuideBtn'),\n    menuInstallPwaBtn: document.getElementById('menuInstallPwaBtn'),\n    menuTestPwaToastBtn: document.getElementById('menuTestPwaToastBtn'),\n  };\n\n  // =========================================================================\n  // GAME-FEEL PRESENTATION HELPERS (SYNTHESIZED SFX, HAPTICS, DUST, SHAKE)\n  // =========================================================================\n\n  const SFX_STORAGE_KEY = 'connect4_sfx_enabled';\n  const SFX_VOLUME_KEY = 'connect4_sfx_volume';\n  let sfxEnabled = true;\n  let sfxVolume = 0.6;\n  let audioCtx = null;\n  let masterGain = null;\n  let masterCompressor = null;\n  let activeVoices = 0;\n  const MAX_VOICES = 4;\n  let lastTurnSignature = '';\n  let viewLeaveTimer = null;\n  let activeConfettiRaf = null;\n\n  try {\n    if (typeof window !== 'undefined' && window.localStorage) {\n      const storedSfx = window.localStorage.getItem(SFX_STORAGE_KEY);\n      if (storedSfx !== null) {\n        sfxEnabled = storedSfx === 'true';\n      }\n      const storedVol = window.localStorage.getItem(SFX_VOLUME_KEY);\n      if (storedVol !== null) {\n        const v = parseFloat(storedVol);\n        if (!isNaN(v) && v >= 0 && v <= 1) sfxVolume = v;\n      }\n    }\n  } catch (_) {}\n\n  function prefersReducedMotion() {\n    try {\n      return Boolean(\n        typeof window !== 'undefined'\n        && window.matchMedia\n        && window.matchMedia('(prefers-reduced-motion: reduce)').matches\n      );\n    } catch (_) {\n      return false;\n    }\n  }\n\n  function syncSoundToggleUI() {\n    if (!els.soundToggleBtn) return;\n    if (els.soundToggleBtn.classList) {\n      els.soundToggleBtn.classList.toggle('is-active', sfxEnabled);\n    }\n    if (typeof els.soundToggleBtn.setAttribute === 'function') {\n      els.soundToggleBtn.setAttribute('aria-pressed', String(sfxEnabled));\n      els.soundToggleBtn.setAttribute(\n        'aria-label',\n        sfxEnabled ? 'Matikan suara efek game' : 'Aktifkan suara efek game'\n      );\n    }\n    els.soundToggleBtn.title = sfxEnabled ? 'Suara efek: Nyala' : 'Suara efek: Mati';\n    if (els.sfxVolumeSlider) {\n      els.sfxVolumeSlider.value = String(Math.round(sfxVolume * 100));\n    }\n  }\n\n  function setSfxVolume(vol) {\n    sfxVolume = Math.max(0, Math.min(1, vol));\n    try {\n      if (typeof window !== 'undefined' && window.localStorage) {\n        window.localStorage.setItem(SFX_VOLUME_KEY, String(sfxVolume));\n      }\n    } catch (_) {}\n    if (masterGain && audioCtx) {\n      masterGain.gain.setValueAtTime(sfxVolume, audioCtx.currentTime);\n    }\n  }\n\n  function toggleSoundEnabled() {\n    sfxEnabled = !sfxEnabled;\n    try {\n      if (typeof window !== 'undefined' && window.localStorage) {\n        window.localStorage.setItem(SFX_STORAGE_KEY, String(sfxEnabled));\n      }\n    } catch (_) {}\n    syncSoundToggleUI();\n    if (sfxEnabled) playSfx('click');\n  }\n\n  function ensureAudioContext() {\n    if (audioCtx) return audioCtx;\n    if (typeof window === 'undefined') return null;\n    const Ctor = window.AudioContext || window.webkitAudioContext;\n    if (!Ctor) return null;\n    try {\n      audioCtx = new Ctor();\n      masterCompressor = audioCtx.createDynamicsCompressor();\n      masterCompressor.threshold.setValueAtTime(-18, audioCtx.currentTime);\n      masterCompressor.knee.setValueAtTime(12, audioCtx.currentTime);\n      masterCompressor.ratio.setValueAtTime(8, audioCtx.currentTime);\n      masterCompressor.attack.setValueAtTime(0.003, audioCtx.currentTime);\n      masterCompressor.release.setValueAtTime(0.2, audioCtx.currentTime);\n\n      masterGain = audioCtx.createGain();\n      masterGain.gain.setValueAtTime(sfxVolume, audioCtx.currentTime);\n\n      masterCompressor.connect(masterGain);\n      masterGain.connect(audioCtx.destination);\n    } catch (_) {\n      return null;\n    }\n    return audioCtx;\n  }\n\n  function resumeAudioOnGesture() {\n    const ctx = ensureAudioContext();\n    if (ctx && ctx.state === 'suspended' && typeof ctx.resume === 'function') {\n      ctx.resume();\n    }\n  }\n\n  function playSfx(kind, params = {}) {\n    if (!sfxEnabled) return;\n    const ctx = ensureAudioContext();\n    if (!ctx || !masterCompressor) return;\n    try {\n      if (ctx.state === 'suspended' && typeof ctx.resume === 'function') {\n        ctx.resume();\n      }\n      if (activeVoices >= MAX_VOICES && kind === 'hover') return;\n\n      const now = ctx.currentTime;\n      activeVoices++;\n      const releaseVoice = (delaySec) => {\n        setTimeout(() => { activeVoices = Math.max(0, activeVoices - 1); }, Math.round(delaySec * 1000));\n      };\n\n      if (kind === 'click') {\n        const osc = ctx.createOscillator();\n        const gain = ctx.createGain();\n        osc.type = 'triangle';\n        osc.frequency.setValueAtTime(680, now);\n        osc.frequency.exponentialRampToValueAtTime(320, now + 0.035);\n        gain.gain.setValueAtTime(0.12, now);\n        gain.gain.exponentialRampToValueAtTime(0.001, now + 0.04);\n        osc.connect(gain);\n        gain.connect(masterCompressor);\n        osc.start(now);\n        osc.stop(now + 0.045);\n        releaseVoice(0.05);\n      } else if (kind === 'hover') {\n        const osc = ctx.createOscillator();\n        const gain = ctx.createGain();\n        osc.type = 'sine';\n        osc.frequency.setValueAtTime(520, now);\n        gain.gain.setValueAtTime(0.025, now);\n        gain.gain.exponentialRampToValueAtTime(0.001, now + 0.025);\n        osc.connect(gain);\n        gain.connect(masterCompressor);\n        osc.start(now);\n        osc.stop(now + 0.03);\n        releaseVoice(0.035);\n      } else if (kind === 'whoosh') {\n        const osc = ctx.createOscillator();\n        const gain = ctx.createGain();\n        osc.type = 'sine';\n        osc.frequency.setValueAtTime(340, now);\n        osc.frequency.exponentialRampToValueAtTime(95, now + 0.12);\n        gain.gain.setValueAtTime(0.06, now);\n        gain.gain.exponentialRampToValueAtTime(0.001, now + 0.13);\n        osc.connect(gain);\n        gain.connect(masterCompressor);\n        osc.start(now);\n        osc.stop(now + 0.14);\n        releaseVoice(0.15);\n      } else if (kind === 'thud') {\n        const row = typeof params.row === 'number' ? params.row : 3;\n        const maxRows = typeof params.maxRows === 'number' ? params.maxRows : 6;\n        const depthRatio = Math.min(1, Math.max(0, row / (maxRows - 1 || 1)));\n        const baseFreq = Math.round(115 - depthRatio * 45);\n        const vol = 0.16 + depthRatio * 0.10;\n\n        const osc = ctx.createOscillator();\n        const gain = ctx.createGain();\n        osc.type = 'triangle';\n        osc.frequency.setValueAtTime(baseFreq, now);\n        osc.frequency.exponentialRampToValueAtTime(38, now + 0.08);\n        gain.gain.setValueAtTime(vol, now);\n        gain.gain.exponentialRampToValueAtTime(0.001, now + 0.09);\n        osc.connect(gain);\n        gain.connect(masterCompressor);\n        osc.start(now);\n        osc.stop(now + 0.095);\n\n        const bounceOsc = ctx.createOscillator();\n        const bounceGain = ctx.createGain();\n        const bNow = now + 0.068;\n        bounceOsc.type = 'sine';\n        bounceOsc.frequency.setValueAtTime(baseFreq * 1.25, bNow);\n        bounceOsc.frequency.exponentialRampToValueAtTime(50, bNow + 0.05);\n        bounceGain.gain.setValueAtTime(vol * 0.38, bNow);\n        bounceGain.gain.exponentialRampToValueAtTime(0.001, bNow + 0.055);\n        bounceOsc.connect(bounceGain);\n        bounceGain.connect(masterCompressor);\n        bounceOsc.start(bNow);\n        bounceOsc.stop(bNow + 0.06);\n\n        releaseVoice(0.14);\n      } else if (kind === 'botTurn') {\n        const osc = ctx.createOscillator();\n        const gain = ctx.createGain();\n        osc.type = 'sine';\n        osc.frequency.setValueAtTime(460, now);\n        osc.frequency.setValueAtTime(620, now + 0.04);\n        gain.gain.setValueAtTime(0.07, now);\n        gain.gain.exponentialRampToValueAtTime(0.001, now + 0.085);\n        osc.connect(gain);\n        gain.connect(masterCompressor);\n        osc.start(now);\n        osc.stop(now + 0.09);\n        releaseVoice(0.1);\n      } else if (kind === 'win') {\n        const notes = [523.25, 659.25, 783.99, 1046.5];\n        notes.forEach((freq, idx) => {\n          const t0 = now + idx * 0.075;\n          const osc = ctx.createOscillator();\n          const gain = ctx.createGain();\n          osc.type = 'triangle';\n          osc.frequency.setValueAtTime(freq, t0);\n          gain.gain.setValueAtTime(0.12, t0);\n          gain.gain.exponentialRampToValueAtTime(0.001, t0 + 0.22);\n          osc.connect(gain);\n          gain.connect(masterCompressor);\n          osc.start(t0);\n          osc.stop(t0 + 0.23);\n        });\n        releaseVoice(0.5);\n      } else if (kind === 'lose') {\n        const notes = [440, 392, 349.23, 293.66];\n        notes.forEach((freq, idx) => {\n          const t0 = now + idx * 0.085;\n          const osc = ctx.createOscillator();\n          const gain = ctx.createGain();\n          osc.type = 'sine';\n          osc.frequency.setValueAtTime(freq, t0);\n          gain.gain.setValueAtTime(0.10, t0);\n          gain.gain.exponentialRampToValueAtTime(0.001, t0 + 0.20);\n          osc.connect(gain);\n          gain.connect(masterCompressor);\n          osc.start(t0);\n          osc.stop(t0 + 0.21);\n        });\n        releaseVoice(0.45);\n      } else if (kind === 'draw') {\n        [349.23, 440].forEach((freq) => {\n          const osc = ctx.createOscillator();\n          const gain = ctx.createGain();\n          osc.type = 'triangle';\n          osc.frequency.setValueAtTime(freq, now);\n          gain.gain.setValueAtTime(0.09, now);\n          gain.gain.exponentialRampToValueAtTime(0.001, now + 0.26);\n          osc.connect(gain);\n          gain.connect(masterCompressor);\n          osc.start(now);\n          osc.stop(now + 0.28);\n        });\n        releaseVoice(0.3);\n      } else if (kind === 'undo') {\n        const osc = ctx.createOscillator();\n        const gain = ctx.createGain();\n        osc.type = 'triangle';\n        osc.frequency.setValueAtTime(260, now);\n        osc.frequency.exponentialRampToValueAtTime(560, now + 0.08);\n        gain.gain.setValueAtTime(0.09, now);\n        gain.gain.exponentialRampToValueAtTime(0.001, now + 0.09);\n        osc.connect(gain);\n        gain.connect(masterCompressor);\n        osc.start(now);\n        osc.stop(now + 0.095);\n        releaseVoice(0.1);\n      }\n    } catch (_) {}\n  }\n\n  function triggerHaptic(type, arg) {\n    if (!sfxEnabled) return;\n    try {\n      if (typeof window !== 'undefined' && window.navigator && typeof window.navigator.vibrate === 'function') {\n        if (type === 'drop') {\n          const row = typeof arg === 'number' ? arg : 3;\n          window.navigator.vibrate(Math.min(35, 12 + row * 3));\n        } else if (type === 'win') {\n          window.navigator.vibrate([40, 50, 70]);\n        }\n      }\n    } catch (_) {}\n  }\n\n  function triggerBoardImpact(row, col) {\n    if (prefersReducedMotion() || !els.boardGrid) return;\n    const shakePx = Math.min(2.5, 0.8 + row * 0.22);\n    els.boardGrid.style.setProperty('--shake-px', `${shakePx}px`);\n    els.boardGrid.classList.remove('board-impact');\n    if (typeof els.boardGrid.offsetWidth === 'number') void els.boardGrid.offsetWidth;\n    els.boardGrid.classList.add('board-impact');\n    setTimeout(() => {\n      if (els.boardGrid && els.boardGrid.classList) {\n        els.boardGrid.classList.remove('board-impact');\n      }\n    }, 180);\n\n    if (els.colSelectors && els.colSelectors.children && els.colSelectors.children[col]) {\n      const colBtn = els.colSelectors.children[col];\n      colBtn.classList.remove('col-landing-flash');\n      if (typeof colBtn.offsetWidth === 'number') void colBtn.offsetWidth;\n      colBtn.classList.add('col-landing-flash');\n      setTimeout(() => {\n        if (colBtn && colBtn.classList) colBtn.classList.remove('col-landing-flash');\n      }, 220);\n    }\n  }\n\n  function createLandingDust(row, col) {\n    if (prefersReducedMotion() || !els.boardDustLayer) return;\n    const cols = state.board && state.board[0] ? state.board[0].length : 7;\n    const rows = state.board ? state.board.length : 6;\n    const burst = document.createElement('div');\n    burst.className = 'dust-burst';\n    const leftPct = ((col + 0.5) / cols) * 100;\n    const topPct = ((row + 0.95) / rows) * 100;\n    burst.style.left = `${leftPct}%`;\n    burst.style.top = `${topPct}%`;\n\n    for (let i = 0; i < 5; i++) {\n      const p = document.createElement('span');\n      p.className = 'dust-particle';\n      const angle = (Math.PI / 6) + (Math.random() * (2 * Math.PI / 3));\n      const dist = 12 + Math.random() * 16;\n      p.style.setProperty('--dx', `${(Math.cos(angle) * dist).toFixed(1)}px`);\n      p.style.setProperty('--dy', `${(-Math.sin(angle) * dist).toFixed(1)}px`);\n      burst.appendChild(p);\n    }\n    els.boardDustLayer.appendChild(burst);\n    setTimeout(() => {\n      if (burst.parentNode) burst.parentNode.removeChild(burst);\n    }, 280);\n  }\n\n  function getBotSpeedFactor() {\n    const mode = els.botSpeedSelect ? els.botSpeedSelect.value : 'normal';\n    if (mode === 'fast') return 0.35;\n    if (mode === 'slow') return 2.2;\n    return 1;\n  }\n\n  function closeOverflowMenu() {\n    if (els.arenaOverflowMenu) {\n      els.arenaOverflowMenu.open = false;\n    }\n  }\n\n  function switchHelpTab(tabName = 'rules') {\n    const isAi = tabName === 'ai';\n    if (els.helpPanelRules) els.helpPanelRules.style.display = isAi ? 'none' : 'flex';\n    if (els.helpPanelAi) els.helpPanelAi.style.display = isAi ? 'flex' : 'none';\n    if (els.rulesModal && els.rulesModal.querySelectorAll) {\n      els.rulesModal.querySelectorAll('[data-help-tab]').forEach((btn) => {\n        const active = btn.dataset.helpTab === tabName;\n        btn.classList.toggle('active', active);\n        if (typeof btn.setAttribute === 'function') btn.setAttribute('aria-selected', String(active));\n      });\n    }\n    if (isAi) {\n      renderScoringLegend(els.modalLegendList, null, null, AI);\n    }\n  }\n\n  function switchAnalysisTab(tabName = 'steps') {\n    const isEff = tabName === 'efficiency';\n    if (els.analysisTabSteps) els.analysisTabSteps.style.display = isEff ? 'none' : 'flex';\n    if (els.analysisTabEfficiency) els.analysisTabEfficiency.style.display = isEff ? 'flex' : 'none';\n    if (els.postGamePanel && els.postGamePanel.querySelectorAll) {\n      els.postGamePanel.querySelectorAll('[data-analysis-tab]').forEach((btn) => {\n        const active = btn.dataset.analysisTab === tabName;\n        btn.classList.toggle('active', active);\n        if (typeof btn.setAttribute === 'function') btn.setAttribute('aria-selected', String(active));\n      });\n    }\n  }\n\n  function getAiConfig() {\n    const is4P = getPlayerCount() === 4;\n    if (state.difficulty === 'Easy') {\n      return { difficulty: 'Easy', usePruning: false, maxDepth: 0, label: 'Easy • Random' };\n    }\n    if (state.difficulty === 'Medium') {\n      return { difficulty: 'Medium', usePruning: false, maxDepth: 3, label: 'Medium • d=3' };\n    }\n    if (state.difficulty === 'Hard') {\n      const hardDepth = is4P ? 4 : 5;\n      return { difficulty: 'Hard', usePruning: true, maxDepth: hardDepth, label: `Hard • α-β d=${hardDepth}` };\n    }\n    const usePruning = els.algorithmSelect.value === 'alphabeta';\n    const maxDepth = Number(els.depthSelect.value);\n    const algoTxt = usePruning ? 'α-β' : 'Minimax';\n    return {\n      difficulty: 'Custom',\n      usePruning,\n      maxDepth,\n      label: `Custom • ${algoTxt} d=${maxDepth}`,\n    };\n  }\n\n  function formatScore(score) {\n    if (score === null || score === undefined) return '-';\n    if (score >= WIN_SCORE - 100) return `+${score} (Win)`;\n    if (score <= -(WIN_SCORE - 100)) return `${score} (Lose)`;\n    return score > 0 ? `+${score}` : `${score}`;\n  }\n\n  function pulsePickedCard(cardEl) {\n    if (!cardEl) return;\n    cardEl.classList.remove('just-picked');\n    if (typeof cardEl.offsetWidth === 'number') {\n      void cardEl.offsetWidth;\n    }\n    cardEl.classList.add('just-picked');\n  }\n\n  function openModal(modalEl) {\n    if (!modalEl) return;\n    const pending = modalCloseTimers.get(modalEl);\n    if (pending) {\n      clearTimeout(pending);\n      modalCloseTimers.delete(modalEl);\n    }\n    modalEl.classList.remove('is-closing');\n    modalEl.style.display = 'flex';\n  }\n\n  function closeModal(modalEl) {\n    if (!modalEl || modalEl.style.display === 'none' || modalEl.classList.contains('is-closing')) {\n      return;\n    }\n    modalEl.classList.add('is-closing');\n    const timer = setTimeout(() => {\n      modalCloseTimers.delete(modalEl);\n      modalEl.style.display = 'none';\n      modalEl.classList.remove('is-closing');\n    }, 180);\n    modalCloseTimers.set(modalEl, timer);\n  }\n\n  function syncFourPlayerSeatsUI() {\n    if (!els.fourPlayerBox) return;\n    const is4P = state.gameMode === '4p';\n    els.fourPlayerBox.style.display = is4P ? 'block' : 'none';\n    if (!is4P) return;\n\n    for (let seat = 1; seat <= 4; seat++) {\n      const tag = els.fourSeatsGrid.querySelector(`[data-role-for=\"${seat}\"]`);\n      if (!tag) continue;\n      const isBot = state.fourSeats[seat] === 'bot';\n      tag.textContent = isBot ? 'Bot' : 'Manusia';\n      tag.classList.toggle('is-bot', isBot);\n    }\n\n    if (els.fourPresetGrid) {\n      els.fourPresetGrid.querySelectorAll('.preset-chip').forEach((chip) => {\n        const preset = FOUR_PRESETS[chip.dataset.preset];\n        const match = preset && [1, 2, 3, 4].every((s) => preset[s] === state.fourSeats[s]);\n        chip.classList.toggle('active', Boolean(match));\n      });\n    }\n  }\n\n  function syncMenuUI() {\n    els.modeChoiceGrid.querySelectorAll('.choice-card').forEach((btn) => {\n      btn.classList.toggle('active', btn.dataset.mode === state.gameMode);\n    });\n    els.boardModeGrid.querySelectorAll('.choice-card').forEach((btn) => {\n      btn.classList.toggle('active', btn.dataset.bmode === state.boardMode);\n    });\n    if (els.boardSizeGrid) {\n      els.boardSizeGrid.querySelectorAll('.size-btn').forEach((btn) => {\n        btn.classList.toggle('active', btn.dataset.size === state.boardSize);\n      });\n    }\n\n    syncFourPlayerSeatsUI();\n\n    const botsPresent = hasAnyBot();\n    if (els.diffSection) {\n      els.diffSection.style.display = botsPresent ? 'block' : 'none';\n    }\n    els.diffChoiceGrid.querySelectorAll('.choice-card').forEach((btn) => {\n      btn.classList.toggle('active', btn.dataset.diff === state.difficulty);\n    });\n    els.customConfigBox.style.display = botsPresent && state.difficulty === 'Custom' ? 'block' : 'none';\n\n    const is4P = state.gameMode === '4p';\n    if (!is4P && (state.firstChoice === '3' || state.firstChoice === '4')) {\n      state.firstChoice = '1';\n    }\n\n    els.firstChoiceGrid.classList.toggle('cols-5-first', is4P);\n    els.firstChoiceGrid.classList.toggle('cols-3-always', !is4P);\n    els.firstChoiceGrid.querySelectorAll('.choice-card').forEach((btn) => {\n      const f = btn.dataset.first;\n      if (f === '3' || f === '4') {\n        btn.style.display = is4P ? 'flex' : 'none';\n      }\n      btn.classList.toggle('active', f === state.firstChoice);\n    });\n\n    if (els.firstSectionLabel) {\n      els.firstSectionLabel.textContent = 'Jalan pertama';\n    }\n\n    if (state.gameMode === 'hvb') {\n      els.firstP1Title.textContent = 'Manusia';\n      els.firstP2Title.textContent = 'Bot AI';\n    } else if (state.gameMode === 'pvp') {\n      els.firstP1Title.textContent = 'Pemain 1';\n      els.firstP2Title.textContent = 'Pemain 2';\n    } else if (state.gameMode === 'bvb') {\n      els.firstP1Title.textContent = 'Bot Coral';\n      els.firstP2Title.textContent = 'Bot Kuning';\n    } else {\n      els.firstP1Title.textContent = state.fourSeats[1] === 'bot' ? 'Bot 1' : 'Manusia 1';\n      els.firstP2Title.textContent = state.fourSeats[2] === 'bot' ? 'Bot 2' : 'Manusia 2';\n      if (els.firstP3Title) els.firstP3Title.textContent = state.fourSeats[3] === 'bot' ? 'Bot 3' : 'Manusia 3';\n      if (els.firstP4Title) els.firstP4Title.textContent = state.fourSeats[4] === 'bot' ? 'Bot 4' : 'Manusia 4';\n    }\n    if (els.firstRandomSub) {\n      els.firstRandomSub.textContent = is4P ? 'Undi 1–4' : 'Undi 50:50';\n    }\n\n    const modeTagMap = {\n      hvb: '2P AI',\n      pvp: 'PvP 2P',\n      '4p': '4P',\n      bvb: 'Bot Duel',\n    };\n    const sizeInfo = (AI_LIB.BOARD_SIZES && AI_LIB.BOARD_SIZES[state.boardSize]) || { label: state.boardSize };\n    const obsTag = state.boardMode === 'obstacle' ? ' • Batu' : '';\n    const cfg = getAiConfig();\n    const diffShort =\n      cfg.difficulty === 'Custom'\n        ? `Custom d=${cfg.maxDepth}`\n        : cfg.difficulty;\n    els.setupPreviewBadge.textContent = `${modeTagMap[state.gameMode]} • ${sizeInfo.label}${obsTag}`;\n    els.activeConfigBadge.textContent = `${modeTagMap[state.gameMode]}${botsPresent ? ` • ${diffShort}` : ''} • ${sizeInfo.label}${obsTag}`;\n\n    syncAdvancedSummary(botsPresent, is4P, cfg);\n    renderLobbyPreview();\n  }\n\n  function syncAdvancedSummary(botsPresent, is4P, cfg) {\n    if (!els.advancedSummaryBadge) return;\n    const firstMap = {\n      '1': is4P ? 'P1' : (state.gameMode === 'hvb' ? 'Manusia' : 'P1'),\n      '2': is4P ? 'P2' : (state.gameMode === 'hvb' ? 'Bot AI' : 'P2'),\n      '3': 'P3',\n      '4': 'P4',\n      random: 'Acak',\n    };\n    const parts = [];\n    if (is4P) {\n      let humanCount = 0;\n      for (let s = 1; s <= 4; s++) {\n        if (state.fourSeats[s] === 'human') humanCount += 1;\n      }\n      parts.push(`${humanCount}P+${4 - humanCount}B`);\n    }\n    if (botsPresent && cfg && cfg.difficulty === 'Custom') {\n      parts.push(`${cfg.algorithm === 'minimax' ? 'MM' : 'AB'} d=${cfg.maxDepth}`);\n    }\n    parts.push(`Jalan: ${firstMap[state.firstChoice] || 'P1'}`);\n    els.advancedSummaryBadge.textContent = parts.join(' • ');\n  }\n\n  function renderLobbyPreview() {\n    if (!els.lobbyPreviewBoard) return;\n    const sizeInfo = (AI_LIB.BOARD_SIZES && AI_LIB.BOARD_SIZES[state.boardSize]) || { rows: 6, cols: 7, label: '6×7' };\n    const rows = sizeInfo.rows;\n    const cols = sizeInfo.cols;\n    const isObs = state.boardMode === 'obstacle';\n    const pCount = getPlayerCount();\n\n    if (els.lobbyPreviewLabel) {\n      els.lobbyPreviewLabel.textContent = `${isObs ? 'Obstacle' : 'Klasik'} • ${sizeInfo.label}`;\n    }\n\n    if (els.lobbyPreviewChips) {\n      els.lobbyPreviewChips.innerHTML = '';\n      for (let p = 1; p <= pCount; p++) {\n        const meta = PLAYER_META[p];\n        const chip = document.createElement('span');\n        chip.className = 'pv-chip';\n        chip.innerHTML = `<span class=\"pv-dot ${meta.css}\"></span><span>P${p}</span>`;\n        els.lobbyPreviewChips.appendChild(chip);\n      }\n      if (isObs) {\n        const obsChip = document.createElement('span');\n        obsChip.className = 'pv-chip';\n        obsChip.innerHTML = `<span class=\"pv-dot obs\"></span><span>Batu</span>`;\n        els.lobbyPreviewChips.appendChild(obsChip);\n      }\n    }\n\n    if (els.lobbyPreviewBoard.style) {\n      els.lobbyPreviewBoard.style.setProperty('--p-rows', String(rows));\n      els.lobbyPreviewBoard.style.setProperty('--p-cols', String(cols));\n    }\n\n    const previewMap = new Map();\n    if (isObs) {\n      const stones = [\n        [rows - 2, 1],\n        [rows - 2, cols - 2],\n        [Math.floor(rows / 2), Math.floor(cols / 2)],\n        [Math.floor(rows / 2) - 1, Math.floor(cols / 2) - 1],\n      ];\n      stones.forEach(([sr, sc]) => {\n        if (sr >= 0 && sr < rows && sc >= 0 && sc < cols) {\n          previewMap.set(`${sr},${sc}`, 'is-stone');\n        }\n      });\n    }\n\n    const sampleDiscs = pCount === 4\n      ? [\n          [rows - 1, 0, 'p1'],\n          [rows - 1, 2, 'p2'],\n          [rows - 1, 4, 'p3'],\n          [rows - 1, cols - 1, 'p4'],\n          [rows - 2, 2, 'p1'],\n          [rows - 2, 4, 'p2'],\n        ]\n      : [\n          [rows - 1, Math.floor(cols / 2) - 1, 'p1'],\n          [rows - 1, Math.floor(cols / 2), 'p2'],\n          [rows - 1, Math.floor(cols / 2) + 1, 'p1'],\n          [rows - 2, Math.floor(cols / 2), 'p1'],\n          [rows - 2, Math.floor(cols / 2) - 1, 'p2'],\n        ];\n    sampleDiscs.forEach(([dr, dc, cls]) => {\n      const key = `${dr},${dc}`;\n      if (dr >= 0 && dr < rows && dc >= 0 && dc < cols && !previewMap.has(key)) {\n        previewMap.set(key, cls);\n      }\n    });\n\n    els.lobbyPreviewBoard.innerHTML = '';\n    for (let r = 0; r < rows; r++) {\n      for (let c = 0; c < cols; c++) {\n        const cell = document.createElement('div');\n        const extra = previewMap.get(`${r},${c}`) || '';\n        cell.className = extra ? `p-cell ${extra}` : 'p-cell';\n        els.lobbyPreviewBoard.appendChild(cell);\n      }\n    }\n  }\n\n  function openAdvancedDrawer() {\n    if (!els.advancedSetupDetails) return;\n    openModal(els.advancedSetupDetails);\n    els.advancedSetupDetails.open = true;\n  }\n\n  function closeAdvancedDrawer() {\n    if (!els.advancedSetupDetails) return;\n    els.advancedSetupDetails.open = false;\n    closeModal(els.advancedSetupDetails);\n  }\n\n  function clearTimers() {\n    if (state.pendingBotTimer) {\n      clearTimeout(state.pendingBotTimer);\n      state.pendingBotTimer = null;\n    }\n    if (state.autoPlayTimer) {\n      clearTimeout(state.autoPlayTimer);\n      state.autoPlayTimer = null;\n    }\n    if (state.postGameTimer) {\n      clearTimeout(state.postGameTimer);\n      state.postGameTimer = null;\n    }\n    if (state.dropTimer) {\n      clearTimeout(state.dropTimer);\n      state.dropTimer = null;\n    }\n    state.isThinking = false;\n    state.isAnimatingDrop = false;\n  }\n\n  function retriggerStagePop(viewEl) {\n    const stage = viewEl && viewEl.querySelector ? viewEl.querySelector('.animate-pop-in') : null;\n    if (!stage) return;\n    stage.classList.remove('animate-pop-in');\n    if (typeof stage.offsetWidth === 'number') {\n      void stage.offsetWidth;\n    }\n    stage.classList.add('animate-pop-in');\n  }\n\n  function switchView(viewName) {\n    const prevName = state.view;\n    const viewMap = {\n      home: els.homeView,\n      setup: els.setupView,\n      game: els.gameView,\n    };\n    const nextEl = viewMap[viewName];\n    const prevEl = viewMap[prevName];\n\n    if (viewLeaveTimer) {\n      clearTimeout(viewLeaveTimer);\n      viewLeaveTimer = null;\n    }\n\n    state.view = viewName;\n\n    Object.entries(viewMap).forEach(([name, el]) => {\n      if (!el) return;\n      if (el.classList) el.classList.remove('view-entering', 'view-leaving');\n      if (name !== viewName && name !== prevName) {\n        el.style.display = 'none';\n      }\n    });\n\n    if (nextEl) {\n      nextEl.style.display = viewName === 'game' ? 'block' : 'flex';\n      if (prevName !== viewName && !prefersReducedMotion() && nextEl.classList) {\n        nextEl.classList.add('view-entering');\n      }\n    }\n\n    if (prevEl && prevEl !== nextEl) {\n      if (!prefersReducedMotion() && prevEl.classList && typeof window !== 'undefined' && window.document && window.document.body) {\n        prevEl.classList.add('view-leaving');\n        viewLeaveTimer = setTimeout(() => {\n          viewLeaveTimer = null;\n          if (state.view !== prevName) {\n            prevEl.style.display = 'none';\n            prevEl.classList.remove('view-leaving');\n          }\n        }, 180);\n      } else {\n        prevEl.style.display = 'none';\n      }\n    }\n\n    if (viewName === 'home') retriggerStagePop(els.homeView);\n    else if (viewName === 'setup') retriggerStagePop(els.setupView);\n  }\n\n  function renderScoringLegend(targetEl, info, winner, evalPiece = AI) {\n    function renderMiniTiles(dots) {\n      return (\n        '<span class=\"mini-tiles\">' +\n        dots.map((d) => `<span class=\"mini-tile\"><span class=\"mini-dot ${d}\"></span></span>`).join('') +\n        '</span>'\n      );\n    }\n\n    const rows = [\n      {\n        dots: ['ai', 'ai', 'ai', 'ai'],\n        label: '4-in-a-Row Menang',\n        count: winner && winner !== 'DRAW' ? (winner === evalPiece ? 'Menang' : 'Lawan') : info ? '0x' : 'Terminal',\n        pts: '+100000',\n      },\n      {\n        dots: ['ai', 'ai', 'ai', 'empty'],\n        label: '3 Sewarna + 1 Kosong',\n        count: info ? `${info.aiThree}x` : 'per pola',\n        pts: '+12',\n      },\n      {\n        dots: ['ai', 'ai', 'empty', 'empty'],\n        label: '2 Sewarna + 2 Kosong',\n        count: info ? `${info.aiTwo}x` : 'per pola',\n        pts: '+3',\n      },\n      {\n        dots: ['opp', 'opp', 'opp', 'empty'],\n        label: 'Ancaman 3 Lawan',\n        count: info ? `${info.oppThree}x` : 'per pola',\n        pts: '-80',\n      },\n      {\n        dots: ['opp', 'opp', 'empty', 'empty'],\n        label: '2 Lawan + 2 Kosong',\n        count: info ? `${info.oppTwo}x` : 'per pola',\n        pts: '-2',\n      },\n      {\n        dots: ['center'],\n        label: 'Kontrol Kolom Tengah',\n        count: info ? `P${evalPiece} ${info.centerAi}:${info.centerOpp}` : 'per selisih',\n        pts: info ? `${info.centerScore >= 0 ? '+' : ''}${info.centerScore}` : '+4 / +2',\n      },\n    ];\n\n    targetEl.innerHTML = '';\n    rows.forEach((row) => {\n      const el = document.createElement('div');\n      el.className = 'legend-score-row';\n      el.innerHTML = `\n        <div class=\"legend-pattern\">\n          ${renderMiniTiles(row.dots)}\n          <span>${row.label}</span>\n        </div>\n        <div class=\"legend-right\">\n          <span class=\"legend-live-count\">${row.count}</span>\n          <span class=\"legend-points\">${row.pts}</span>\n        </div>\n      `;\n      targetEl.appendChild(el);\n    });\n  }\n\n  function getWinningIndex(r, c) {\n    for (let i = 0; i < state.winCells.length; i++) {\n      if (state.winCells[i][0] === r && state.winCells[i][1] === c) return i;\n    }\n    return -1;\n  }\n\n  function saveSnapshot() {\n    state.history.push({\n      board: AI_LIB.cloneBoard(state.board),\n      currentPlayer: state.currentPlayer,\n      gameOver: state.gameOver,\n      winner: state.winner,\n      winCells: state.winCells.slice(),\n      lastMove: state.lastMove ? { ...state.lastMove } : null,\n      lastAiResult: state.lastAiResult ? { ...state.lastAiResult, columnScores: { ...state.lastAiResult.columnScores } } : null,\n      moveCount: state.moveCount,\n      moveLog: state.moveLog.slice(),\n      compareTarget: state.compareTarget\n        ? {\n            board: AI_LIB.cloneBoard(state.compareTarget.board),\n            aiPiece: state.compareTarget.aiPiece,\n            playerCount: state.compareTarget.playerCount,\n            step: state.compareTarget.step,\n          }\n        : null,\n    });\n  }\n\n  function restoreSnapshot(snap) {\n    state.board = AI_LIB.cloneBoard(snap.board);\n    state.currentPlayer = snap.currentPlayer;\n    state.gameOver = snap.gameOver;\n    state.winner = snap.winner;\n    state.winCells = snap.winCells.slice();\n    state.lastMove = snap.lastMove ? { ...snap.lastMove } : null;\n    state.lastAiResult = snap.lastAiResult;\n    state.moveCount = snap.moveCount;\n    state.moveLog = snap.moveLog.slice();\n    state.compareTarget = snap.compareTarget\n      ? {\n          board: AI_LIB.cloneBoard(snap.compareTarget.board),\n          aiPiece: snap.compareTarget.aiPiece,\n          playerCount: snap.compareTarget.playerCount,\n          step: snap.compareTarget.step,\n        }\n      : null;\n  }\n\n  function renderBoard() {\n    const humanCanClick = !state.gameOver && !state.isThinking && !isBotPlayer(state.currentPlayer);\n    const rows = state.board.length;\n    const cols = state.board[0].length;\n    const activeCss = (PLAYER_META[state.currentPlayer] || PLAYER_META[1]).css;\n\n    if (els.arenaShell && els.arenaShell.classList) {\n      els.arenaShell.classList.toggle('is-wide-arena', getPlayerCount() >= 4 || cols >= 8);\n    }\n    if (els.hoverPreviewRow && els.hoverPreviewRow.classList) {\n      els.hoverPreviewRow.classList.toggle('is-idle', !humanCanClick);\n    }\n\n    [els.arenaShell, els.hoverPreviewRow, els.colSelectors, els.boardGrid].forEach((el) => {\n      if (!el || !el.style) return;\n      el.style.setProperty('--board-cols', String(cols));\n      el.style.setProperty('--board-rows', String(rows));\n    });\n\n    // Ghost Hover Preview Row (colored to match current player)\n    els.hoverPreviewRow.innerHTML = '';\n    for (let c = 0; c < cols; c++) {\n      const slot = document.createElement('div');\n      slot.className = 'ghost-slot';\n      const ghost = document.createElement('div');\n      ghost.className = `ghost-disc ${activeCss}`;\n      ghost.dataset.col = String(c);\n      slot.appendChild(ghost);\n      els.hoverPreviewRow.appendChild(slot);\n    }\n\n    // Column selector buttons\n    els.colSelectors.innerHTML = '';\n    for (let c = 0; c < cols; c++) {\n      const btn = document.createElement('button');\n      btn.type = 'button';\n      btn.className = 'col-btn';\n      const valid = AI_LIB.isValidLocation(state.board, c);\n      if (state.lastAiResult && state.lastAiResult.bestCol === c) {\n        btn.classList.add('col-best');\n      }\n      btn.disabled = !humanCanClick || !valid;\n      btn.textContent = String(c);\n      btn.title = `Kolom ${c} (tekan ${c})`;\n      btn.addEventListener('mouseenter', () => setHoverCol(c));\n      btn.addEventListener('mouseleave', () => setHoverCol(-1));\n      btn.addEventListener('click', () => handleHumanMove(c));\n      els.colSelectors.appendChild(btn);\n    }\n\n    // Dynamic Grid (6x7 / 7x8 / 7x9)\n    els.boardGrid.innerHTML = '';\n    for (let r = 0; r < rows; r++) {\n      for (let c = 0; c < cols; c++) {\n        const cell = document.createElement('div');\n        cell.className = 'cell';\n        cell.dataset.row = String(r);\n        cell.dataset.col = String(c);\n\n        const val = state.board[r][c];\n        const blocked = val === BLOCKED;\n        const isPiece = val >= 1 && val <= 4;\n\n        if (blocked) {\n          cell.classList.add('is-blocked', 'disabled');\n        } else if (!humanCanClick || !AI_LIB.isValidLocation(state.board, c)) {\n          cell.classList.add('disabled');\n        }\n\n        const winIdx = getWinningIndex(r, c);\n        if (winIdx !== -1) {\n          cell.classList.add('win-cell');\n          cell.style.setProperty('--win-idx', String(winIdx));\n        }\n\n        if (isPiece) {\n          const meta = PLAYER_META[val] || PLAYER_META[1];\n          const disc = document.createElement('div');\n          disc.className = `disc ${meta.css}`;\n          if (state.lastMove && state.lastMove.row === r && state.lastMove.col === c) {\n            disc.classList.add('is-last-disc');\n            if (state.lastMove.animate) {\n              disc.classList.add('drop-anim');\n              disc.style.setProperty('--drop-rows', String(r + 1));\n              if (state.lastMove.duration) {\n                disc.style.setProperty('--drop-duration', `${state.lastMove.duration}ms`);\n              }\n            }\n          }\n          cell.appendChild(disc);\n        }\n\n        if (!blocked) {\n          cell.addEventListener('mouseenter', () => setHoverCol(c));\n          cell.addEventListener('mouseleave', () => setHoverCol(-1));\n          cell.addEventListener('click', () => {\n            if (humanCanClick) handleHumanMove(c);\n          });\n        }\n        els.boardGrid.appendChild(cell);\n      }\n    }\n\n    if (state.lastMove) {\n      state.lastMove.animate = false;\n    }\n    applyHoverVisuals();\n    renderWinLineOverlay();\n  }\n\n  function renderWinLineOverlay() {\n    if (!els.winLineSvg) return;\n    els.winLineSvg.innerHTML = '';\n    if (!state.gameOver || !state.winCells || state.winCells.length < 2) return;\n    if (typeof document === 'undefined' || typeof document.createElementNS !== 'function') return;\n\n    const cols = state.board[0].length;\n    const rows = state.board.length;\n    const first = state.winCells[0];\n    const last = state.winCells[state.winCells.length - 1];\n    const firstCell = els.boardGrid.children[first[0] * cols + first[1]];\n    const lastCell = els.boardGrid.children[last[0] * cols + last[1]];\n\n    const svgNS = 'http://www.w3.org/2000/svg';\n    const w = els.boardGrid.clientWidth || 0;\n    const h = els.boardGrid.clientHeight || 0;\n    let x1;\n    let y1;\n    let x2;\n    let y2;\n\n    if (w > 0 && h > 0 && firstCell && lastCell && typeof firstCell.offsetLeft === 'number') {\n      els.winLineSvg.setAttribute('viewBox', `0 0 ${w} ${h}`);\n      x1 = firstCell.offsetLeft + firstCell.offsetWidth / 2;\n      y1 = firstCell.offsetTop + firstCell.offsetHeight / 2;\n      x2 = lastCell.offsetLeft + lastCell.offsetWidth / 2;\n      y2 = lastCell.offsetTop + lastCell.offsetHeight / 2;\n    } else {\n      els.winLineSvg.setAttribute('viewBox', '0 0 100 100');\n      x1 = ((first[1] + 0.5) / cols) * 100;\n      y1 = ((first[0] + 0.5) / rows) * 100;\n      x2 = ((last[1] + 0.5) / cols) * 100;\n      y2 = ((last[0] + 0.5) / rows) * 100;\n    }\n\n    const glow = document.createElementNS(svgNS, 'line');\n    glow.setAttribute('x1', String(x1));\n    glow.setAttribute('y1', String(y1));\n    glow.setAttribute('x2', String(x2));\n    glow.setAttribute('y2', String(y2));\n    glow.setAttribute('class', 'win-beam-glow');\n\n    const core = document.createElementNS(svgNS, 'line');\n    core.setAttribute('x1', String(x1));\n    core.setAttribute('y1', String(y1));\n    core.setAttribute('x2', String(x2));\n    core.setAttribute('y2', String(y2));\n    core.setAttribute('class', 'win-beam-core');\n\n    els.winLineSvg.appendChild(glow);\n    els.winLineSvg.appendChild(core);\n  }\n\n  function applyHoverVisuals() {\n    const humanCanClick = !state.gameOver && !state.isThinking && !isBotPlayer(state.currentPlayer);\n    const col = state.hoverCol;\n    const landingRow =\n      humanCanClick && col >= 0 && AI_LIB.isValidLocation(state.board, col)\n        ? AI_LIB.getNextOpenRow(state.board, col)\n        : -1;\n\n    els.hoverPreviewRow.querySelectorAll('.ghost-disc').forEach((g, idx) => {\n      g.classList.toggle('visible', idx === col && landingRow !== -1);\n    });\n\n    els.colSelectors.querySelectorAll('.col-btn').forEach((btn, idx) => {\n      btn.classList.toggle('col-hover', idx === col && landingRow !== -1);\n    });\n\n    els.boardGrid.querySelectorAll('.cell').forEach((cell) => {\n      const r = Number(cell.dataset.row);\n      const c = Number(cell.dataset.col);\n      const isColHover = c === col && landingRow !== -1 && state.board[r][c] === EMPTY;\n      cell.classList.toggle('col-active-hover', isColHover && r !== landingRow);\n      cell.classList.toggle('target-landing', isColHover && r === landingRow);\n    });\n  }\n\n  function setHoverCol(col) {\n    if (state.hoverCol === col) return;\n    state.hoverCol = col;\n    applyHoverVisuals();\n    if (col >= 0 && !state.gameOver && !state.isThinking && !state.isAnimatingDrop && !isBotPlayer(state.currentPlayer)) {\n      playSfx('hover');\n    }\n  }\n\n  function renderBoardLegend() {\n    if (!els.boardLegend) return;\n    const count = getPlayerCount();\n    els.boardLegend.innerHTML = '';\n\n    for (let p = 1; p <= count; p++) {\n      const meta = PLAYER_META[p];\n      const item = document.createElement('span');\n      item.className = 'legend-item';\n      if (!state.gameOver && state.currentPlayer === p) {\n        item.classList.add('is-turn');\n      }\n      const sw = document.createElement('span');\n      sw.className = `legend-swatch ${meta.css}`;\n      const lbl = document.createElement('span');\n      lbl.textContent = getPlayerLabel(p, true);\n      item.appendChild(sw);\n      item.appendChild(lbl);\n      els.boardLegend.appendChild(item);\n    }\n\n    if (state.boardMode === 'obstacle') {\n      const obs = document.createElement('span');\n      obs.className = 'legend-item';\n      obs.textContent = '✕ Batu Rintangan';\n      els.boardLegend.appendChild(obs);\n    }\n  }\n\n  function renderArenaHeaderAndStatus() {\n    const modeTitleMap = {\n      hvb: 'ARENA 2P AI',\n      pvp: 'PEMAIN VS PEMAIN',\n      '4p': 'ARENA 4-PLAYER',\n      bvb: 'BOT VS BOT',\n    };\n    const obsLabel = state.boardMode === 'obstacle' ? ' • OBSTACLE' : '';\n    els.arenaPill.textContent = `${modeTitleMap[state.gameMode]}${obsLabel}`;\n\n    els.bvbBar.style.display = isAllBots() && !state.gameOver ? 'flex' : 'none';\n    els.autoPlayBtn.textContent = state.autoPlayPaused ? 'Lanjut auto-play' : 'Jeda auto-play';\n\n    els.turnCounterBadge.textContent = `#${state.moveCount}`;\n    els.undoBtn.disabled = state.history.length === 0 || state.isThinking;\n\n    if (state.lastAiResult) {\n      els.lastMovePill.style.display = 'inline-block';\n      els.lastMovePill.textContent = `Kol ${state.lastAiResult.bestCol} (${formatScore(state.lastAiResult.bestScore)})`;\n    } else {\n      els.lastMovePill.style.display = 'none';\n    }\n\n    renderBoardLegend();\n\n    const activeMeta = PLAYER_META[state.currentPlayer] || PLAYER_META[1];\n    els.statusBanner.className = 'turn-pill';\n    els.statusDot.className = 'status-dot';\n    els.thinkingDots.classList.remove('active');\n\n    if (state.gameOver) {\n      if (state.winner === 'DRAW') {\n        els.statusBanner.classList.add('game-draw');\n        els.statusDot.classList.add('neutral');\n        els.statusText.textContent = 'Permainan seri';\n      } else {\n        const winMeta = PLAYER_META[state.winner] || PLAYER_META[1];\n        els.statusBanner.classList.add('game-win');\n        els.statusDot.classList.add(winMeta.css);\n        els.statusText.textContent = `${getPlayerLabel(state.winner, true)} menang`;\n      }\n      return;\n    }\n\n    els.statusBanner.classList.add(`turn-${activeMeta.css}`);\n    els.statusDot.classList.add(activeMeta.css);\n\n    if (state.isThinking) {\n      els.statusText.textContent = `${getPlayerLabel(state.currentPlayer, true)} berpikir`;\n      els.thinkingDots.classList.add('active');\n      return;\n    }\n\n    if (state.gameMode === 'hvb' && state.currentPlayer === HUMAN) {\n      els.statusText.textContent = 'Giliran Anda';\n    } else {\n      els.statusText.textContent = `Giliran ${getPlayerLabel(state.currentPlayer, true)}`;\n    }\n\n    pulseTurnBanner(`${state.gameOver ? state.winner : state.currentPlayer}-${state.isThinking ? 'wait' : 'ready'}`);\n  }\n\n  function pulseTurnBanner(sig) {\n    if (sig === lastTurnSignature) return;\n    lastTurnSignature = sig;\n    if (!els.statusBanner || !els.statusBanner.classList || prefersReducedMotion()) return;\n    els.statusBanner.classList.remove('turn-chip-pop');\n    if (typeof els.statusBanner.offsetWidth === 'number') {\n      void els.statusBanner.offsetWidth;\n    }\n    els.statusBanner.classList.add('turn-chip-pop');\n  }\n\n  function renderPostGameAnalysis(showPanel) {\n    if (!showPanel || !state.gameOver) {\n      els.arenaShell.classList.remove('has-analysis');\n      els.postGamePanel.style.display = 'none';\n      return;\n    }\n\n    els.arenaShell.classList.add('has-analysis');\n    els.postGamePanel.style.display = 'flex';\n\n    els.postGameBanner.className = 'postgame-banner';\n    if (state.winner === 'DRAW') {\n      els.postGameBanner.classList.add('is-draw');\n      els.postGameTitle.textContent = 'Hasil Akhir: SERI!';\n      els.postGameSub.textContent = `Papan penuh setelah ${state.moveCount} langkah tanpa 4-in-a-row.`;\n    } else {\n      const winnerIsBot = isBotPlayer(state.winner);\n      if (winnerIsBot && state.gameMode === 'hvb') {\n        els.postGameBanner.classList.add('is-bot-win');\n      }\n      els.postGameTitle.textContent = `${getPlayerLabel(state.winner, false)} Menang!`;\n      els.postGameSub.textContent = `4 bidak terhubung dalam total ${state.moveCount} langkah (${getPlayerCount()}-Player).`;\n    }\n\n    const botEntries = state.moveLog.filter((m) => m.isBot);\n    const totalNodes = botEntries.reduce((acc, m) => acc + m.nodes, 0);\n    const totalTime = botEntries.reduce((acc, m) => acc + m.elapsedMs, 0);\n    const avgNodes = botEntries.length > 0 ? Math.round(totalNodes / botEntries.length) : 0;\n\n    animateCountUp(els.statTotalMoves, state.moveCount, (n) => String(Math.round(n)));\n    animateCountUp(els.statBotMoves, botEntries.length, (n) => String(Math.round(n)));\n    animateCountUp(els.statAvgNodes, avgNodes, (n) => Math.round(n).toLocaleString('id-ID'));\n    els.statTotalTime.textContent = `${totalTime.toFixed(1)} ms`;\n\n    if (state.winner !== 'DRAW') {\n      launchConfettiBurst();\n    }\n\n    els.postGameTableBody.innerHTML = '';\n    state.moveLog.forEach((entry, idx) => {\n      const tr = document.createElement('tr');\n      tr.style.setProperty('--row-idx', String(idx));\n      if (entry.isBot) tr.classList.add('row-bot');\n      if (idx === state.moveLog.length - 1 && state.winner !== 'DRAW') tr.classList.add('row-win');\n      tr.innerHTML = `\n        <td>#${entry.step}</td>\n        <td>${entry.actor}</td>\n        <td>Kol ${entry.col} (R${entry.row})</td>\n        <td>${entry.scoreText}</td>\n        <td>${entry.isBot ? entry.nodes.toLocaleString('id-ID') : '-'}</td>\n        <td>${entry.isBot ? `${entry.elapsedMs} ms` : '-'}</td>\n      `;\n      els.postGameTableBody.appendChild(tr);\n    });\n\n    const pCount = getPlayerCount();\n    const evalPiece = state.winner && state.winner !== 'DRAW' ? state.winner : (state.compareTarget ? state.compareTarget.aiPiece : AI);\n    const finalInfo = AI_LIB.analyzeBoard(state.board, evalPiece, pCount);\n    els.legendTotalBadge.textContent = `Total (P${evalPiece}): ${formatScore(finalInfo.totalScore)}`;\n    renderScoringLegend(els.heuristicGrid, finalInfo, state.winner, evalPiece);\n\n    if (state.compareTarget) {\n      els.compareContextText.textContent =\n        `Bandingkan jumlah node pada langkah #${state.compareTarget.step} sebelum penutup (mode ${pCount}P).`;\n    }\n  }\n\n  function renderAll(showPostGameNow = false) {\n    renderArenaHeaderAndStatus();\n    renderBoard();\n    renderPostGameAnalysis(state.gameOver && showPostGameNow);\n  }\n\n  function executeDrop(col, player, aiResult) {\n    const row = AI_LIB.getNextOpenRow(state.board, col);\n    if (row === -1) return false;\n\n    const pCount = getPlayerCount();\n    const preMoveBoard = AI_LIB.cloneBoard(state.board);\n    saveSnapshot();\n    AI_LIB.dropPiece(state.board, row, col, player);\n    state.moveCount += 1;\n\n    // Realistic physics: duration scales with distance (row + 1)\n    const dropDistance = row + 1;\n    const dropDuration = prefersReducedMotion() ? 60 : Math.round(140 + Math.sqrt(dropDistance) * 75);\n    state.isAnimatingDrop = true;\n    state.lastMove = { row, col, animate: true, duration: dropDuration };\n\n    if (aiResult) {\n      state.lastAiResult = aiResult;\n    }\n\n    state.compareTarget = {\n      board: preMoveBoard,\n      aiPiece: player,\n      playerCount: pCount,\n      step: state.moveCount,\n    };\n\n    const isBot = isBotPlayer(player);\n    const actor = getPlayerLabel(player, true);\n    const scoreText = aiResult\n      ? formatScore(aiResult.bestScore)\n      : formatScore(AI_LIB.evaluateBoard(state.board, player, pCount));\n\n    state.moveLog.push({\n      step: state.moveCount,\n      actor,\n      col,\n      row,\n      isBot,\n      scoreText,\n      nodes: aiResult ? aiResult.nodes : 0,\n      elapsedMs: aiResult ? aiResult.elapsedMs : 0,\n    });\n\n    renderAll(false);\n    playSfx('whoosh');\n\n    const totalRows = state.board ? state.board.length : 6;\n    const impactTime = prefersReducedMotion() ? 0 : Math.round(dropDuration * 0.68);\n\n    if (impactTime > 0) {\n      setTimeout(() => {\n        playSfx('thud', { row, maxRows: totalRows });\n        triggerBoardImpact(row, col);\n        createLandingDust(row, col);\n        triggerHaptic('drop', row);\n      }, impactTime);\n    } else {\n      playSfx('thud', { row, maxRows: totalRows });\n    }\n\n    if (state.dropTimer) clearTimeout(state.dropTimer);\n    state.dropTimer = setTimeout(() => {\n      state.dropTimer = null;\n      state.isAnimatingDrop = false;\n\n      const winInfo = AI_LIB.getWinnerInfo(state.board);\n      if (winInfo.winner !== null) {\n        state.gameOver = true;\n        state.winner = winInfo.winner;\n        state.winCells = winInfo.cells;\n        clearTimers();\n        renderAll(false);\n\n        if (winInfo.winner === 'DRAW') {\n          playSfx('draw');\n          if (els.boardGrid && !prefersReducedMotion()) {\n            els.boardGrid.classList.add('board-draw-shake');\n            setTimeout(() => {\n              if (els.boardGrid && els.boardGrid.classList) els.boardGrid.classList.remove('board-draw-shake');\n            }, 300);\n          }\n        } else {\n          const humanWon = state.gameMode === 'hvb' ? winInfo.winner === HUMAN : true;\n          playSfx(humanWon ? 'win' : 'lose');\n          triggerHaptic('win');\n        }\n\n        state.postGameTimer = setTimeout(() => {\n          state.postGameTimer = null;\n          renderPostGameAnalysis(true);\n        }, 360);\n        return;\n      }\n\n      state.currentPlayer = AI_LIB.getNextPlayer(player, pCount);\n      renderAll(false);\n\n      if (isBotPlayer(state.currentPlayer) && !state.gameOver) {\n        scheduleBotTurn(isAllBots() ? 120 : 180);\n      }\n    }, dropDuration);\n\n    return true;\n  }\n\n  function handleHumanMove(col) {\n    if (state.view !== 'game' || state.gameOver || state.isThinking || state.isAnimatingDrop) return;\n    if (isBotPlayer(state.currentPlayer)) return;\n    if (!AI_LIB.isValidLocation(state.board, col)) return;\n\n    executeDrop(col, state.currentPlayer, null);\n  }\n\n  function scheduleBotTurn(delayMs = 80) {\n    const gen = state.matchGeneration;\n    state.isThinking = true;\n    renderArenaHeaderAndStatus();\n    applyHoverVisuals();\n    playSfx('botTurn');\n\n    state.pendingBotTimer = setTimeout(async () => {\n      state.pendingBotTimer = null;\n      if (state.gameOver || gen !== state.matchGeneration) {\n        state.isThinking = false;\n        renderArenaHeaderAndStatus();\n        return;\n      }\n      await runSingleBotStep(gen);\n    }, Math.round(delayMs * getBotSpeedFactor()));\n  }\n\n  async function runSingleBotStep(expectedGen = state.matchGeneration) {\n    if (state.gameOver || expectedGen !== state.matchGeneration) return;\n    const piece = state.currentPlayer;\n    if (!isBotPlayer(piece)) return;\n\n    const pCount = getPlayerCount();\n    state.isThinking = true;\n    renderArenaHeaderAndStatus();\n\n    const boardCopy = AI_LIB.cloneBoard(state.board);\n    let res;\n    if (state.gameMode === 'bvb' && piece === HUMAN) {\n      res = await requestBotMove(boardCopy, {\n        difficulty: 'Medium',\n        aiPiece: HUMAN,\n        usePruning: false,\n        customDepth: 3,\n        playerCount: pCount,\n      });\n    } else {\n      const cfg = getAiConfig();\n      res = await requestBotMove(boardCopy, {\n        difficulty: cfg.difficulty,\n        aiPiece: piece,\n        usePruning: cfg.usePruning,\n        customDepth: cfg.difficulty === 'Custom' ? cfg.maxDepth : undefined,\n        playerCount: pCount,\n      });\n    }\n\n    if (expectedGen !== state.matchGeneration || state.gameOver) {\n      state.isThinking = false;\n      return;\n    }\n\n    state.isThinking = false;\n    if (res && res.bestCol !== -1) {\n      executeDrop(res.bestCol, piece, res);\n    } else {\n      renderArenaHeaderAndStatus();\n    }\n  }\n\n  async function startAutoPlayLoop() {\n    if (!isAllBots() || state.gameOver) return;\n    if (state.autoPlayTimer) {\n      clearTimeout(state.autoPlayTimer);\n      state.autoPlayTimer = null;\n    }\n    const gen = state.matchGeneration;\n    state.autoPlayPaused = false;\n    renderArenaHeaderAndStatus();\n\n    while (!state.gameOver && !state.autoPlayPaused && gen === state.matchGeneration && state.view === 'game') {\n      await runSingleBotStep(gen);\n      if (state.gameOver || state.autoPlayPaused || gen !== state.matchGeneration || state.view !== 'game') {\n        break;\n      }\n      await new Promise((resolve) => {\n        state.autoPlayTimer = setTimeout(() => {\n          state.autoPlayTimer = null;\n          resolve();\n        }, Math.round(420 * getBotSpeedFactor()));\n      });\n    }\n  }\n\n  function resetGame() {\n    state.matchGeneration += 1;\n    clearTimers();\n    playSfx('undo');\n\n    const dims = (AI_LIB.BOARD_SIZES && AI_LIB.BOARD_SIZES[state.boardSize]) || { rows: ROWS, cols: COLS };\n    const setup = AI_LIB.createSpecialSetup(state.boardMode, dims.rows, dims.cols);\n    state.board = setup.board;\n\n    const pCount = getPlayerCount();\n    if (state.firstChoice === 'random') {\n      state.resolvedFirstPlayer = 1 + Math.floor(Math.random() * pCount);\n      state.randomStartNote = `[Acak: ${getPlayerLabel(state.resolvedFirstPlayer, true)}]`;\n    } else {\n      const parsed = Number(state.firstChoice);\n      state.resolvedFirstPlayer = parsed >= 1 && parsed <= pCount ? parsed : 1;\n      state.randomStartNote = '';\n    }\n\n    state.currentPlayer = state.resolvedFirstPlayer;\n    state.gameOver = false;\n    state.winner = null;\n    state.winCells = [];\n    state.lastMove = null;\n    state.lastAiResult = null;\n    state.moveCount = 0;\n    state.moveLog = [];\n    state.history = [];\n    state.compareTarget = null;\n    state.hoverCol = -1;\n    state.autoPlayPaused = false;\n\n    if (els.compareDepthSelect) {\n      els.compareDepthSelect.value = dims.cols >= 10 ? '3' : '4';\n    }\n    switchAnalysisTab('steps');\n    closeOverflowMenu();\n    els.compareResult.innerHTML =\n      'Tekan tombol di atas untuk membandingkan efisiensi node pencarian.';\n\n    renderAll(false);\n\n    if (isAllBots()) {\n      startAutoPlayLoop();\n    } else if (isBotPlayer(state.currentPlayer)) {\n      scheduleBotTurn(120);\n    }\n  }\n\n  function startNewMatchFromMenu() {\n    syncMenuUI();\n    switchView('game');\n    resetGame();\n  }\n\n  function goToHome() {\n    state.matchGeneration += 1;\n    clearTimers();\n    closeOverflowMenu();\n    switchView('home');\n  }\n\n  function goToSetup() {\n    state.matchGeneration += 1;\n    clearTimers();\n    closeOverflowMenu();\n    syncMenuUI();\n    switchView('setup');\n  }\n\n  function undoMove() {\n    if (state.history.length === 0 || state.isThinking || state.isAnimatingDrop) return;\n    state.matchGeneration += 1;\n    clearTimers();\n    playSfx('undo');\n\n    if (!isAllBots() && Object.values(getSeatRoles()).includes('human')) {\n      let targetIdx = state.history.length - 1;\n      while (targetIdx > 0 && isBotPlayer(state.history[targetIdx].currentPlayer)) {\n        targetIdx--;\n      }\n      const snap = state.history[targetIdx];\n      state.history = state.history.slice(0, targetIdx);\n      restoreSnapshot(snap);\n      renderAll(false);\n      if (isBotPlayer(state.currentPlayer) && !state.gameOver) {\n        scheduleBotTurn(120);\n      }\n    } else {\n      state.autoPlayPaused = true;\n      const snap = state.history.pop();\n      restoreSnapshot(snap);\n      renderAll(false);\n    }\n  }\n\n  function toggleAutoPlay() {\n    if (!isAllBots() || state.gameOver) return;\n    if (state.autoPlayPaused) {\n      startAutoPlayLoop();\n    } else {\n      state.autoPlayPaused = true;\n      if (state.autoPlayTimer) {\n        clearTimeout(state.autoPlayTimer);\n        state.autoPlayTimer = null;\n      }\n      renderArenaHeaderAndStatus();\n    }\n  }\n\n  async function runPostGameComparison() {\n    const target = state.compareTarget;\n    if (!target) return;\n    const depth = Number(els.compareDepthSelect.value);\n    els.compareBtn.disabled = true;\n    els.compareResult.innerHTML = 'Menghitung perbandingan Minimax vs Alpha-Beta...';\n    const comp = await requestCompare(target.board, depth, target.aiPiece, target.playerCount || getPlayerCount());\n    els.compareBtn.disabled = false;\n\n    els.compareResult.innerHTML = `\n      <table class=\"compare-table\">\n        <thead>\n          <tr>\n            <th>Algoritma (Depth ${comp.maxDepth})</th>\n            <th>Kolom</th>\n            <th>Skor Eval</th>\n            <th>Node</th>\n            <th>Waktu</th>\n          </tr>\n        </thead>\n        <tbody>\n          <tr>\n            <td>Depth-Limited Minimax</td>\n            <td>Kol ${comp.pureMinimax.bestCol}</td>\n            <td>${formatScore(comp.pureMinimax.bestScore)}</td>\n            <td>${comp.pureMinimax.nodes.toLocaleString('id-ID')}</td>\n            <td>${comp.pureMinimax.elapsedMs} ms</td>\n          </tr>\n          <tr>\n            <td>Alpha-Beta Pruning</td>\n            <td>Kol ${comp.alphaBeta.bestCol}</td>\n            <td>${formatScore(comp.alphaBeta.bestScore)}</td>\n            <td>${comp.alphaBeta.nodes.toLocaleString('id-ID')}</td>\n            <td>${comp.alphaBeta.elapsedMs} ms</td>\n          </tr>\n        </tbody>\n      </table>\n      <div class=\"compare-summary\">\n        Alpha-Beta memangkas <strong>${comp.reductionPct}%</strong> node pencarian (${(comp.pureMinimax.nodes - comp.alphaBeta.nodes).toLocaleString('id-ID')} node dihemat) dengan keputusan ${comp.sameDecision ? '100% IDENTIK' : 'berbeda'}.\n      </div>\n    `;\n  }\n\n  function animateCountUp(el, targetVal, formatter) {\n    if (!el) return;\n    if (\n      targetVal <= 0\n      || prefersReducedMotion()\n      || typeof window === 'undefined'\n      || typeof window.requestAnimationFrame !== 'function'\n    ) {\n      el.textContent = formatter(targetVal);\n      return;\n    }\n    const start = (typeof performance !== 'undefined' && performance.now) ? performance.now() : Date.now();\n    const duration = 280;\n    function step(now) {\n      const elapsed = Math.min(1, Math.max(0, (now - start) / duration));\n      const eased = 1 - Math.pow(1 - elapsed, 3);\n      el.textContent = formatter(targetVal * eased);\n      if (elapsed < 1) {\n        window.requestAnimationFrame(step);\n      } else {\n        el.textContent = formatter(targetVal);\n      }\n    }\n    window.requestAnimationFrame(step);\n  }\n\n  function launchConfettiBurst() {\n    if (\n      !els.confettiCanvas\n      || prefersReducedMotion()\n      || typeof window === 'undefined'\n      || typeof window.requestAnimationFrame !== 'function'\n      || typeof els.confettiCanvas.getContext !== 'function'\n    ) {\n      return;\n    }\n    const canvas = els.confettiCanvas;\n    const ctx = canvas.getContext('2d');\n    if (!ctx) return;\n    if (activeConfettiRaf) {\n      window.cancelAnimationFrame(activeConfettiRaf);\n      activeConfettiRaf = null;\n    }\n    canvas.width = window.innerWidth || 1280;\n    canvas.height = window.innerHeight || 720;\n\n    const palette = ['#fd5a46', '#ffc567', '#00995e', '#552cb7', '#058cd7'];\n    const particles = [];\n    for (let i = 0; i < 46; i++) {\n      particles.push({\n        x: (canvas.width * 0.5) + (Math.random() - 0.5) * 260,\n        y: canvas.height * 0.25 + (Math.random() - 0.5) * 60,\n        vx: (Math.random() - 0.5) * 9,\n        vy: -Math.random() * 7 - 2,\n        w: 8 + Math.random() * 6,\n        h: 6 + Math.random() * 5,\n        rot: Math.random() * Math.PI * 2,\n        vr: (Math.random() - 0.5) * 0.22,\n        color: palette[i % palette.length],\n      });\n    }\n\n    const startTime = (typeof performance !== 'undefined' && performance.now) ? performance.now() : Date.now();\n    function tick(now) {\n      const elapsed = now - startTime;\n      ctx.clearRect(0, 0, canvas.width, canvas.height);\n      if (elapsed > 1750) {\n        activeConfettiRaf = null;\n        return;\n      }\n      const alpha = elapsed > 1250 ? Math.max(0, 1 - (elapsed - 1250) / 500) : 1;\n      ctx.globalAlpha = alpha;\n      particles.forEach((p) => {\n        p.x += p.vx;\n        p.y += p.vy;\n        p.vy += 0.34;\n        p.rot += p.vr;\n        ctx.save();\n        ctx.translate(p.x, p.y);\n        ctx.rotate(p.rot);\n        ctx.fillStyle = p.color;\n        ctx.fillRect(-p.w / 2, -p.h / 2, p.w, p.h);\n        ctx.strokeStyle = '#111111';\n        ctx.lineWidth = 1.5;\n        ctx.strokeRect(-p.w / 2, -p.h / 2, p.w, p.h);\n        ctx.restore();\n      });\n      ctx.globalAlpha = 1;\n      activeConfettiRaf = window.requestAnimationFrame(tick);\n    }\n    activeConfettiRaf = window.requestAnimationFrame(tick);\n  }\n\n  function bindGridKeyboardNav(containerEl) {\n    if (!containerEl || typeof containerEl.addEventListener !== 'function') return;\n    containerEl.addEventListener('keydown', (e) => {\n      if (!['ArrowLeft', 'ArrowRight', 'ArrowUp', 'ArrowDown'].includes(e.key)) return;\n      if (!containerEl.querySelectorAll) return;\n      const items = Array.from(containerEl.querySelectorAll('button')).filter(\n        (b) => !b.disabled && (!b.style || b.style.display !== 'none')\n      );\n      if (items.length <= 1) return;\n      const idx = items.indexOf(e.target);\n      if (idx === -1) return;\n      e.preventDefault();\n      const delta = (e.key === 'ArrowRight' || e.key === 'ArrowDown') ? 1 : -1;\n      const next = items[(idx + delta + items.length) % items.length];\n      if (next && typeof next.focus === 'function') next.focus();\n    });\n  }\n\n  function bindEvents() {\n    if (els.soundToggleBtn) {\n      els.soundToggleBtn.addEventListener('click', toggleSoundEnabled);\n    }\n\n    if (els.openAdvancedBtn) {\n      els.openAdvancedBtn.addEventListener('click', () => {\n        playSfx('click');\n        openAdvancedDrawer();\n      });\n    }\n    if (els.closeAdvancedBtn) {\n      els.closeAdvancedBtn.addEventListener('click', closeAdvancedDrawer);\n    }\n    if (els.applyAdvancedBtn) {\n      els.applyAdvancedBtn.addEventListener('click', () => {\n        playSfx('click');\n        closeAdvancedDrawer();\n      });\n    }\n    if (els.advancedSetupDetails) {\n      els.advancedSetupDetails.addEventListener('click', (e) => {\n        if (e.target === els.advancedSetupDetails) closeAdvancedDrawer();\n      });\n    }\n\n    [\n      els.modeChoiceGrid,\n      els.boardModeGrid,\n      els.boardSizeGrid,\n      els.diffChoiceGrid,\n      els.firstChoiceGrid,\n      els.fourPresetGrid,\n      els.fourSeatsGrid,\n    ].forEach(bindGridKeyboardNav);\n\n    els.modeChoiceGrid.addEventListener('click', (e) => {\n      const btn = e.target.closest('.choice-card');\n      if (!btn) return;\n      const nextMode = btn.dataset.mode;\n      if (!state.userPickedSize) {\n        state.boardSize = nextMode === '4p' ? '8x10' : '6x7';\n      }\n      state.gameMode = nextMode;\n      syncMenuUI();\n      pulsePickedCard(btn);\n      playSfx('click');\n      if (nextMode === '4p' && els.openAdvancedBtn && els.openAdvancedBtn.classList) {\n        els.openAdvancedBtn.classList.remove('has-highlight');\n        if (typeof els.openAdvancedBtn.offsetWidth === 'number') void els.openAdvancedBtn.offsetWidth;\n        els.openAdvancedBtn.classList.add('has-highlight');\n      }\n    });\n\n    if (els.fourPresetGrid) {\n      els.fourPresetGrid.addEventListener('click', (e) => {\n        const chip = e.target.closest('.preset-chip');\n        if (!chip) return;\n        const preset = FOUR_PRESETS[chip.dataset.preset];\n        if (!preset) return;\n        state.fourSeats = { ...preset };\n        syncMenuUI();\n        playSfx('click');\n      });\n    }\n\n    if (els.fourSeatsGrid) {\n      els.fourSeatsGrid.addEventListener('click', (e) => {\n        const btn = e.target.closest('.seat-btn');\n        if (!btn) return;\n        const seat = Number(btn.dataset.seat);\n        state.fourSeats[seat] = state.fourSeats[seat] === 'bot' ? 'human' : 'bot';\n        syncMenuUI();\n        pulsePickedCard(btn);\n        playSfx('click');\n      });\n    }\n\n    els.boardModeGrid.addEventListener('click', (e) => {\n      const btn = e.target.closest('.choice-card');\n      if (!btn) return;\n      state.boardMode = btn.dataset.bmode;\n      syncMenuUI();\n      pulsePickedCard(btn);\n      playSfx('click');\n    });\n\n    if (els.boardSizeGrid) {\n      els.boardSizeGrid.addEventListener('click', (e) => {\n        const btn = e.target.closest('.size-btn');\n        if (!btn) return;\n        state.boardSize = btn.dataset.size;\n        state.userPickedSize = true;\n        syncMenuUI();\n        pulsePickedCard(btn);\n        playSfx('click');\n      });\n    }\n\n    els.diffChoiceGrid.addEventListener('click', (e) => {\n      const btn = e.target.closest('.choice-card');\n      if (!btn) return;\n      state.difficulty = btn.dataset.diff;\n      syncMenuUI();\n      pulsePickedCard(btn);\n      playSfx('click');\n      if (state.difficulty === 'Custom') {\n        openAdvancedDrawer();\n      }\n    });\n\n    els.firstChoiceGrid.addEventListener('click', (e) => {\n      const btn = e.target.closest('.choice-card');\n      if (!btn) return;\n      state.firstChoice = btn.dataset.first;\n      syncMenuUI();\n      pulsePickedCard(btn);\n      playSfx('click');\n    });\n\n    els.algorithmSelect.addEventListener('change', syncMenuUI);\n    els.depthSelect.addEventListener('change', syncMenuUI);\n\n    els.goSetupBtn.addEventListener('click', goToSetup);\n    els.backHomeBtn.addEventListener('click', goToHome);\n    els.startGameBtn.addEventListener('click', startNewMatchFromMenu);\n    els.backMenuBtn.addEventListener('click', goToHome);\n    els.backSetupBtn.addEventListener('click', goToSetup);\n    els.postMenuBtn.addEventListener('click', goToHome);\n    els.playAgainBtn.addEventListener('click', resetGame);\n    els.resetBtn.addEventListener('click', resetGame);\n    els.undoBtn.addEventListener('click', undoMove);\n\n    els.botStepBtn.addEventListener('click', () => {\n      if (isAllBots() && !state.gameOver && !state.isThinking) {\n        state.autoPlayPaused = true;\n        if (state.autoPlayTimer) {\n          clearTimeout(state.autoPlayTimer);\n          state.autoPlayTimer = null;\n        }\n        runSingleBotStep(state.matchGeneration);\n      }\n    });\n    els.autoPlayBtn.addEventListener('click', toggleAutoPlay);\n    els.compareBtn.addEventListener('click', runPostGameComparison);\n\n    els.openRulesBtn.addEventListener('click', () => {\n      switchHelpTab('rules');\n      openModal(els.rulesModal);\n    });\n    els.closeRulesBtn.addEventListener('click', () => closeModal(els.rulesModal));\n    els.rulesModal.addEventListener('click', (e) => {\n      if (e.target === els.rulesModal) closeModal(els.rulesModal);\n      const helpTabBtn = e.target.closest ? e.target.closest('[data-help-tab]') : null;\n      if (helpTabBtn) {\n        switchHelpTab(helpTabBtn.dataset.helpTab);\n      }\n    });\n\n    els.openInfoBtn.addEventListener('click', () => {\n      switchHelpTab('ai');\n      openModal(els.rulesModal);\n    });\n    els.closeInfoBtn.addEventListener('click', () => closeModal(els.infoModal));\n    els.infoModal.addEventListener('click', (e) => {\n      if (e.target === els.infoModal) closeModal(els.infoModal);\n    });\n\n    if (els.postGamePanel) {\n      els.postGamePanel.addEventListener('click', (e) => {\n        const tabBtn = e.target.closest ? e.target.closest('[data-analysis-tab]') : null;\n        if (tabBtn) {\n          switchAnalysisTab(tabBtn.dataset.analysisTab);\n        }\n      });\n    }\n\n    document.addEventListener('click', (e) => {\n      if (els.arenaOverflowMenu && els.arenaOverflowMenu.open) {\n        if (!els.arenaOverflowMenu.contains || !els.arenaOverflowMenu.contains(e.target)) {\n          closeOverflowMenu();\n        }\n      }\n    });\n\n    document.addEventListener('keydown', (e) => {\n      if (e.key === 'Escape') {\n        if (els.advancedSetupDetails && els.advancedSetupDetails.style.display === 'flex') {\n          closeAdvancedDrawer();\n          return;\n        }\n        if (els.arenaOverflowMenu && els.arenaOverflowMenu.open) {\n          closeOverflowMenu();\n          return;\n        }\n        if (els.rulesModal.style.display === 'flex') {\n          closeModal(els.rulesModal);\n          return;\n        }\n        if (els.infoModal.style.display === 'flex') {\n          closeModal(els.infoModal);\n          return;\n        }\n        if (els.pwaGuideModal && els.pwaGuideModal.style.display === 'flex') {\n          closeModal(els.pwaGuideModal);\n          return;\n        }\n      }\n      if (state.view !== 'game' || e.target.tagName === 'SELECT' || e.target.tagName === 'INPUT') return;\n      const maxColKey = String(state.board[0].length - 1);\n      if (e.key >= '0' && e.key <= maxColKey) {\n        handleHumanMove(Number(e.key));\n      }\n    });\n\n    // Web Audio first gesture unlock\n    if (typeof window !== 'undefined' && typeof window.addEventListener === 'function') {\n      ['click', 'touchstart', 'keydown'].forEach((evt) => {\n        window.addEventListener(evt, resumeAudioOnGesture, { once: true, passive: true });\n      });\n    }\n\n    // Volume Slider in Overflow Menu\n    if (els.sfxVolumeSlider) {\n      els.sfxVolumeSlider.value = String(Math.round(sfxVolume * 100));\n      els.sfxVolumeSlider.addEventListener('input', (e) => {\n        const v = Number(e.target.value) / 100;\n        setSfxVolume(v);\n      });\n    }\n\n    // PWA Install & Notifications\n    const PWA_DISMISSED_KEY = 'connect4_pwa_banner_dismissed';\n    let deferredInstallPrompt = null;\n\n    function openPwaGuideModal() {\n      openModal(els.pwaGuideModal);\n    }\n\n    async function triggerPwaInstallFlow() {\n      playSfx('click');\n      if (deferredInstallPrompt) {\n        deferredInstallPrompt.prompt();\n        try {\n          const choice = await deferredInstallPrompt.userChoice;\n          if (choice && choice.outcome === 'accepted') {\n            if (els.pwaInstallBtn) els.pwaInstallBtn.style.display = 'none';\n            if (els.pwaBanner) els.pwaBanner.style.display = 'none';\n          }\n        } catch (_) {}\n        deferredInstallPrompt = null;\n      } else {\n        openPwaGuideModal();\n      }\n    }\n\n    if (typeof window !== 'undefined' && typeof window.addEventListener === 'function') {\n      window.addEventListener('beforeinstallprompt', (e) => {\n        e.preventDefault();\n        deferredInstallPrompt = e;\n        if (els.pwaInstallBtn) {\n          els.pwaInstallBtn.style.display = 'inline-flex';\n        }\n        let dismissed = false;\n        try {\n          dismissed = window.localStorage && window.localStorage.getItem(PWA_DISMISSED_KEY) === 'true';\n        } catch (_) {}\n        if (!dismissed && els.pwaBanner) {\n          els.pwaBanner.style.display = 'flex';\n        }\n      });\n    }\n\n    if (els.pwaInstallBtn) {\n      els.pwaInstallBtn.addEventListener('click', triggerPwaInstallFlow);\n    }\n\n    if (els.pwaBannerInstallBtn) {\n      els.pwaBannerInstallBtn.addEventListener('click', triggerPwaInstallFlow);\n    }\n\n    if (els.pwaBannerCloseBtn) {\n      els.pwaBannerCloseBtn.addEventListener('click', () => {\n        playSfx('click');\n        try {\n          if (typeof window !== 'undefined' && window.localStorage) {\n            window.localStorage.setItem(PWA_DISMISSED_KEY, 'true');\n          }\n        } catch (_) {}\n        if (els.pwaBanner) els.pwaBanner.style.display = 'none';\n      });\n    }\n\n    if (els.closePwaGuideBtn) {\n      els.closePwaGuideBtn.addEventListener('click', () => closeModal(els.pwaGuideModal));\n    }\n    if (els.pwaGuideModal) {\n      els.pwaGuideModal.addEventListener('click', (e) => {\n        if (e.target === els.pwaGuideModal) closeModal(els.pwaGuideModal);\n      });\n    }\n\n    if (els.menuInstallPwaBtn) {\n      els.menuInstallPwaBtn.addEventListener('click', () => {\n        closeOverflowMenu();\n        triggerPwaInstallFlow();\n      });\n    }\n\n    if (els.menuTestPwaToastBtn) {\n      els.menuTestPwaToastBtn.addEventListener('click', () => {\n        closeOverflowMenu();\n        playSfx('click');\n        showPwaUpdateToast();\n      });\n    }\n\n    // Auto-show PWA banner after 1.2s if not standalone and not dismissed\n    if (typeof window !== 'undefined' && typeof window.setTimeout === 'function') {\n      setTimeout(() => {\n        const isStandalone = window.matchMedia && window.matchMedia('(display-mode: standalone)').matches;\n        let dismissed = false;\n        try {\n          dismissed = window.localStorage && window.localStorage.getItem(PWA_DISMISSED_KEY) === 'true';\n        } catch (_) {}\n        if (!isStandalone && !dismissed && els.pwaBanner) {\n          els.pwaBanner.style.display = 'flex';\n        }\n      }, 1200);\n    }\n\n    // PWA Service Worker Registration & Update Notification\n    let swRegistration = null;\n    function showPwaUpdateToast() {\n      if (els.pwaUpdateToast) {\n        els.pwaUpdateToast.style.display = 'flex';\n      }\n    }\n\n    if (\n      typeof window !== 'undefined'\n      && typeof window.addEventListener === 'function'\n      && typeof navigator !== 'undefined'\n      && 'serviceWorker' in navigator\n      && window.location\n      && window.location.protocol\n      && window.location.protocol.startsWith('http')\n    ) {\n      window.addEventListener('load', () => {\n        navigator.serviceWorker.register('./sw.js').then((reg) => {\n          swRegistration = reg;\n          reg.addEventListener('updatefound', () => {\n            const newWorker = reg.installing;\n            if (newWorker) {\n              newWorker.addEventListener('statechange', () => {\n                if (newWorker.state === 'installed' && navigator.serviceWorker.controller) {\n                  showPwaUpdateToast();\n                }\n              });\n            }\n          });\n        }).catch((err) => {\n          console.warn('SW registration skipped or failed:', err);\n        });\n      });\n    }\n\n    if (els.pwaReloadBtn) {\n      els.pwaReloadBtn.addEventListener('click', () => {\n        playSfx('click');\n        if (swRegistration && swRegistration.waiting) {\n          swRegistration.waiting.postMessage({ type: 'SKIP_WAITING' });\n        }\n        window.location.reload();\n      });\n    }\n\n    if (typeof window !== 'undefined' && typeof window.addEventListener === 'function') {\n      window.addEventListener('resize', renderWinLineOverlay);\n    }\n  }\n\n  syncSoundToggleUI();\n  syncMenuUI();\n  bindEvents();\n  switchView('home');\n})();\n\n</script>"


def _colab_bot_move(board_json, difficulty="Hard", ai_piece=2, use_pruning=None, custom_depth=None, player_count=2):
    from IPython.display import JSON
    board = json.loads(board_json)
    depth_arg = int(custom_depth) if custom_depth is not None else None
    p_count = int(player_count) if player_count is not None else 2
    res = choose_move(
        board,
        difficulty=difficulty,
        ai_piece=int(ai_piece),
        use_pruning=use_pruning,
        custom_depth=depth_arg,
        player_count=p_count,
    )
    return JSON({
        "bestCol": int(res["best_col"]),
        "bestScore": int(res["best_score"]),
        "columnScores": {str(k): (int(v) if v is not None else None) for k, v in res["column_scores"].items()},
        "nodes": int(res["nodes"]),
        "elapsedMs": float(res["elapsed_ms"]),
        "maxDepth": int(res["max_depth"]),
        "usePruning": bool(res["use_pruning"]),
        "isRandom": bool(res.get("is_random", False)),
    })


def _colab_compare(board_json, max_depth=4, ai_piece=2, player_count=2):
    from IPython.display import JSON
    board = json.loads(board_json)
    p_count = int(player_count) if player_count is not None else 2
    comp = compare_algorithms(board, max_depth=int(max_depth), ai_piece=int(ai_piece), player_count=p_count)
    return JSON({
        "maxDepth": int(comp["max_depth"]),
        "reductionPct": float(comp["reduction_pct"]),
        "sameDecision": bool(comp["same_decision"]),
        "pureMinimax": {
            "bestCol": int(comp["pure"]["best_col"]),
            "bestScore": int(comp["pure"]["best_score"]),
            "nodes": int(comp["pure"]["nodes"]),
            "elapsedMs": float(comp["pure"]["elapsed_ms"]),
        },
        "alphaBeta": {
            "bestCol": int(comp["alphabeta"]["best_col"]),
            "bestScore": int(comp["alphabeta"]["best_score"]),
            "nodes": int(comp["alphabeta"]["nodes"]),
            "elapsedMs": float(comp["alphabeta"]["elapsed_ms"]),
        },
    })


def launch_retro_colab_ui():
    """Render UI Retro 3-Layar terhubung langsung ke fungsi Python di Google Colab."""
    if "choose_move" not in globals():
        print("Jalankan semua sel di atas terlebih dahulu (Runtime -> Run all / Jalankan semua)!")
        return None

    try:
        from IPython.display import HTML, display
    except ImportError:
        print("Info: IPython tidak tersedia di environment ini. Gunakan play_game() pada Bagian 6A untuk mode konsol.")
        return None

    try:
        from google.colab import output
        output.register_callback("connect4.bot_move", _colab_bot_move)
        output.register_callback("connect4.compare", _colab_compare)
        print("[Colab Python Kernel Terhubung] Bot AI & Benchmark di UI mengeksekusi Python runtime Colab.")
    except ImportError:
        print("[Mode Notebook Lokal] google.colab tidak ditemukan; UI berjalan dengan fallback JS 1:1.")

    display(HTML(RETRO_UI_HTML))
    return True


launch_retro_colab_ui()